# Notebook 37 · WeatherNext 2 JPCZ ensemble verification

Compare **all 64 members** with ERA5 for **925-hPa horizontal divergence** in the offshore
JPCZ polygon and coastal wedge, at **1, 5, and 10 days**. Negative values indicate
convergence. The two polygons are evaluated independently, including their overlap.

**Catalog** mode processes all selected events. **Pilot** mode processes one event and
reports six ERA5 ranks. Each event contributes one rank per polygon and lead. Synced Drive
checkpoints allow each completed ERA5 target and forecast lead to be reused after a
runtime disconnect.

The default batch is divergence only. Section 3a's `VERIFY_FIELDS` also accepts native
**u/v components**, wind speed derived from them, geopotential, and geopotential height
at a chosen pressure level. Each selection keeps its own units, ranks and diagrams.
Set `LOAD_SAVED_RESULTS_ONLY = True` after retrieval to rebuild reports from Drive
tables. Section 5 collects the saved values into one Excel sheet without weather-field
retrieval or recalculation.

Run sections 1–5 in this notebook. Its archive reader and rank calculation are included
below. Each helper cell prints a confirmation when it finishes.

**CRPS** scores the full 64-member distribution of each polygon's continuous value
against ERA5 at the same target. Lower is better; its units match the selected field.
Section 5 exports case scores, lead-time summaries and plots from the saved values.
**CRPSS** needs the independent climatological reference specified in the study plan;
this notebook does not substitute the test events as their own climatology.

The catalog is the same CSV Notebook 35 reads. Notebook 36 supplies the archive reader and
onset timing rule. You can run this notebook directly using that CSV and your approved
WeatherNext account. The plotting batches in Notebooks 35 and 36 are optional.

**New winter events:** start with [Notebook 38 · update and search the catalog](https://colab.research.google.com/github/angelicasophyaramirez-blip/JPCZcatalogcolab/blob/codex/notebook36-weathernext2/notebooks/38_update_winter_catalog.ipynb).
It extends the established Notebook 04/06 detector using frozen thresholds, then writes
the same Drive CSV used here. Reload the catalog and rerun this notebook to update
the ranks and reliability plots. Counts always come from that file; no 275-event limit.

[WeatherNext 2 data and attribution](https://developers.google.com/earth-engine/datasets/catalog/projects_gcp-public-data-weathernext_assets_weathernext_2_0_0)
· [Rank histogram method, Gneiting et al. (2007)](https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jrssb.pdf)
· [Interpretation and case selection, Hamill (2001)](https://doi.org/10.1175/1520-0493%282001%29129%3C0550%3AIORHFV%3E2.0.CO%3B2)


In [ ]:
# @title 1 · Install dependencies in a fresh CPU runtime
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy>=1.26,<2.4', 'pandas>=2.2,<3', 'xarray>=2026.4,<2027',
                'zarr>=3.1,<4', 'gcsfs>=2025.3,<2027', 'dask[array]>=2025.1,<2027',
                'matplotlib>=3.9,<3.11', 'xlsxwriter>=3.2,<4', 'pypdf>=5,<7'], check=True)


In [ ]:
# @title 2 · Connect Google Drive and authenticate for WeatherNext
from google.colab import auth, drive

drive.mount('/content/drive')
auth.authenticate_user()
import google.auth
credentials, detected_project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/cloud-platform'])
print('Drive mounted and credentials acquired. Use the approved account for WN2.')


## 3 · Choose the cases

The study dates match Notebook 36’s provisional 2023-onward window by default. The
operational model’s training cutoff has not been verified. Pilot event **0** selects the
first event in the study window. You can enter an event number that worked in Notebook 36
instead. The inventory displayed below includes the numbers and timing offsets.

The target is the nearest 00, 06, 12, or 18 UTC on the original onset’s UTC date, with
ties going earlier. ERA5 is evaluated at that exact time. The catalog’s peak D12 is kept
as context. If required by your Cloud account, enter your existing project ID.


In [ ]:
# @title 3a · Settings
from pathlib import Path

RUN_MODE = 'Catalog'  # @param ["Pilot", "Catalog"]
PILOT_EVENT_NUMBER = 0  # @param {type:"integer"}
PROJECT_ID = ''  # @param {type:"string"}
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
CATALOG_PATH = DRIVE_ROOT / 'jpcz_catalog_ndjf_merged_12h.csv'
NOTEBOOK35_DIR = DRIVE_ROOT / 'imerg_manual_case_review'
OUTPUT_DIR = DRIVE_ROOT / 'weathernext2_ensemble_verification'
STUDY_START_UTC = '2023-01-01'  # @param {type:"date"}
STUDY_END_UTC = ''  # @param {type:"string"} # Optional exclusive UTC date.
ALLOW_ERA5T = True  # Use and label preliminary references; refresh them on every rerun.
LOAD_SAVED_RESULTS_ONLY = False  # True: read saved CSVs only; no weather downloads or field calculations.
VERIFY_FIELDS = [
    {'field':'divergence', 'level':925, 'threshold':-2.0, 'direction':'less'},
]
# Other supported fields: 'u', 'v', 'winds' (speed derived from u/v), 'geopotential', 'height'.
# Each gets separate ranks, units, checkpoints, and plots. Add a threshold in that field's
# displayed units before using reliability diagrams; None exports ranks/values only.
RUN_OVERRIDES = {}


In [ ]:
# @title 3b · Load WeatherNext archive helpers (in this notebook)
"""WeatherNext 2 single-member atlas, also embedded in standalone Notebook 36.

No credentials, remote listings, or forecast fields are loaded at import time.
Build the notebook with tools/build_weathernext_notebook.py after editing this file.
"""
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
import hashlib
import html
import json
import re
import gc

import numpy as np
import pandas as pd
import xarray as xr


LEAD_HOURS = (24, 120, 240)
DATE_COLUMNS = ("event_start", "event_end", "event_peak")
ARCHIVE_ROOT = "weathernext/weathernext_2_0_0/zarr"
SCHEMA_VERSION = 2
PLOT_STYLE_VERSION = 4
VORTEX_LABEL_SCHEME = "notebook35_vortex_screen_v3"
WN2_PRESSURE_LEVELS = (50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000)
EARTH_RADIUS_M = 6371000.0
EARTH_OMEGA_S1 = 7.292115e-5
STANDARD_GRAVITY = 9.80665
KAPPA = 2.0 / 7.0
DERIVED_FIELDS = {"divergence", "ertel_pv"}
# The historical merged stores omit variable-level units. Use the published WN2
# schema only for known variables with ABSENT units; never override declared units.
WN2_SCHEMA_UNITS = {
    "geopotential": "m2 s-2", "u_component_of_wind": "m s-1",
    "v_component_of_wind": "m s-1", "temperature": "K",
    "specific_humidity": "kg kg-1", "vertical_velocity": "Pa s-1",
    "mean_sea_level_pressure": "Pa", "2m_temperature": "K",
    "sea_surface_temperature": "K", "total_precipitation_6hr": "m",
    "10m_u_component_of_wind": "m s-1", "10m_v_component_of_wind": "m s-1",
    "100m_u_component_of_wind": "m s-1", "100m_v_component_of_wind": "m s-1",
}
POLYGONS = {
    "JPCZ polygon": [(129.5, 41.0), (136.0, 37.4), (134.5, 35.8), (128.8, 38.0)],
    "Coastal wedge": [(133.05, 35.55), (136.05, 35.55), (139.55, 39.00), (139.55, 42.55)],
}
DOMAINS = {
    "Circulation (110–160°E, 25–60°N)": (110., 160., 25., 60.),
    "Notebook 35 region (125–145°E, 33–47°N)": (125., 145., 33., 47.),
}


@dataclass(frozen=True)
class FieldSpec:
    key: str
    label: str
    variables: tuple[str, ...]
    units: str
    palette: str
    limits: tuple[float, float]
    level: int | None = None
    conversion: str = "identity"

    @property
    def title(self):
        return f"{self.level}-hPa {self.label}" if self.level is not None else self.label


FIELD_PRESETS = {
    "u": FieldSpec("u", "eastward wind component", ("u_component_of_wind",), "m s⁻¹", "RdBu_r", (-30, 30), 925, "wind_component"),
    "v": FieldSpec("v", "northward wind component", ("v_component_of_wind",), "m s⁻¹", "RdBu_r", (-30, 30), 925, "wind_component"),
    "geopotential": FieldSpec("geopotential", "geopotential", ("geopotential",), "m² s⁻²", "viridis", (45000, 60000), 500, "geopotential"),
    "height": FieldSpec("height", "geopotential height", ("geopotential",), "m", "viridis", (4800, 6000), 500, "height"),
    "winds": FieldSpec("winds", "winds", ("u_component_of_wind", "v_component_of_wind"), "m s⁻¹", "YlGnBu", (0, 30), 925, "wind"),
    "divergence": FieldSpec("divergence", "horizontal divergence (derived)", ("u_component_of_wind", "v_component_of_wind"), "10⁻⁵ s⁻¹", "RdBu_r", (-10, 10), 925, "divergence"),
    "ertel_pv": FieldSpec("ertel_pv", "Ertel PV (derived)", ("temperature", "u_component_of_wind", "v_component_of_wind"), "PVU", "RdBu_r", (-2, 2), 1000, "ertel_pv"),
    "temperature": FieldSpec("temperature", "temperature", ("temperature",), "°C", "coolwarm", (-40, 20), 850, "temperature"),
    "humidity": FieldSpec("humidity", "specific humidity", ("specific_humidity",), "g kg⁻¹", "YlGnBu", (0, 10), 925, "humidity"),
    "vertical_velocity": FieldSpec("vertical_velocity", "vertical velocity", ("vertical_velocity",), "Pa s⁻¹", "RdBu_r", (-1, 1), 700),
    "mslp": FieldSpec("mslp", "mean sea-level pressure", ("mean_sea_level_pressure",), "hPa", "viridis", (980, 1040), None, "pressure"),
    "t2m": FieldSpec("t2m", "2-m temperature", ("2m_temperature",), "°C", "coolwarm", (-20, 20), None, "temperature"),
    "winds10m": FieldSpec("winds10m", "10-m winds", ("10m_u_component_of_wind", "10m_v_component_of_wind"), "m s⁻¹", "YlGnBu", (0, 25), None, "wind"),
    "precipitation": FieldSpec("precipitation", "preceding 6-h precipitation", ("total_precipitation_6hr",), "mm / 6 h", "YlGnBu", (0, 30), None, "precipitation"),
}


def utc_naive(value):
    """Use timezone-free UTC for xarray; never interpret a timestamp as local time."""
    stamp = pd.Timestamp(value)
    if pd.isna(stamp):
        raise ValueError("Missing UTC timestamp.")
    return stamp.tz_localize("UTC").tz_localize(None) if stamp.tzinfo is None else stamp.tz_convert("UTC").tz_localize(None)


def iso_utc(value):
    return utc_naive(value).isoformat() + "Z"


def vortex_classification(record):
    """Use completed event-screen results; legacy manual case labels are unrelated."""
    status = str(record.get("vortex_scan_status", record.get("status", "not_scanned"))).strip().lower()
    if status == "ambiguous":
        return "Vortex status ambiguous"
    if status in {"unavailable", "failed", "error"}:
        return "Vortex status unavailable"
    if status != "complete":
        return "Vortex not checked"
    value = record.get("vortex_detected")
    if isinstance(value, (bool, np.bool_)):
        detected = bool(value)
    elif isinstance(value, (int, float, np.integer, np.floating)) and pd.notna(value) and value in (0, 1):
        detected = bool(value)
    elif isinstance(value, str) and value.strip().lower() in {"true", "false", "0", "1"}:
        detected = value.strip().lower() in {"true", "1"}
    else:
        return "Vortex status unknown"
    return "Vortex" if detected else "Non-vortex"


def vortex_plot_label(record):
    """Use the positive association flag for the two labels shown on figures."""
    if vortex_classification(record) == "Vortex":
        return "Vortex-associated"
    return "Non-vortex-associated"


def vortex_label_metadata(record):
    label = vortex_classification(record)
    detected = True if label == "Vortex" else False if label == "Non-vortex" else None
    return {"vortex_classification": label, "vortex_plot_label": vortex_plot_label(record),
            "vortex_scan_status": str(record.get("vortex_scan_status", "not_scanned")),
            "vortex_detected": detected, "vortex_label_scheme": VORTEX_LABEL_SCHEME,
            "vortex_label_source": str(record.get("vortex_label_source", "saved Notebook 35 event metadata")),
            "vortex_screen_signature": str(record.get("vortex_screen_signature", ""))}


def refresh_vortex_labels(frame, review_dir):
    """Mirror Notebook 35's per-event JSON cache under its active screen signature.

    The inventory CSV can predate a scan. The JSON event records are what Notebook
    35 uses in its event dropdown, so they take precedence, without rerunning scans.
    """
    frame, root = frame.copy(), Path(review_dir) / "vortex_screen"
    active = root / "active_screen.json"
    folder, issue = None, None
    if active.exists():
        try:
            signature = json.loads(active.read_text())["signature"]
            if not isinstance(signature, str) or not re.fullmatch(r"[0-9a-f]{16}", signature):
                raise ValueError("Invalid active vortex-screen signature.")
            folder = root / signature
        except (ValueError, KeyError, OSError, TypeError):
            issue = "unavailable"
    else:
        folders = [p for p in root.glob("*") if p.is_dir() and re.fullmatch(r"[0-9a-f]{16}", p.name)]
        if len(folders) == 1:
            folder = folders[0]
        elif len(folders) > 1:
            issue = "ambiguous"
    records = []
    for _, row in frame.iterrows():
        record = row.to_dict()
        if issue:
            record.update(vortex_scan_status=issue, vortex_detected=None,
                          vortex_label_source="Refresh catalog in updated Notebook 35 to identify its active screen.")
        elif folder is not None:
            path = folder / ("event_" + hashlib.sha256(str(row.review_key).encode()).hexdigest()[:20] + ".json")
            record.update(vortex_scan_status="not_scanned", vortex_detected=None,
                          vortex_label_source=str(path), vortex_screen_signature=folder.name)
            if path.exists():
                try:
                    saved = json.loads(path.read_text())
                    if (not isinstance(saved, dict) or saved.get("status") != "complete" or saved.get("review_key") != row.review_key
                            or saved.get("signature") != folder.name):
                        raise ValueError("Vortex record identity, signature, or completion status does not match.")
                    record.update(vortex_scan_status="complete", vortex_detected=saved.get("vortex_detected"))
                except (ValueError, OSError):
                    record.update(vortex_scan_status="unavailable", vortex_detected=None)
        else:
            record["vortex_label_source"] = "saved Notebook 35 inventory; no event cache folder found"
        records.append(vortex_label_metadata(record))
    updates = pd.DataFrame(records, index=frame.index)
    for column in updates:
        frame[column] = updates[column]
    return frame


def nearest_start_date_time(start):
    """Nearest 00/06/12/18 on the SAME UTC date; equal distances choose earlier."""
    start = utc_naive(start)
    choices = [start.normalize() + pd.Timedelta(hours=h) for h in (0, 6, 12, 18)]
    return min(choices, key=lambda value: (abs(value - start), value))


def normalise_catalog(frame):
    frame = frame.copy()
    for column in DATE_COLUMNS:
        if column not in frame:
            raise ValueError(f"Catalog is missing {column}; use Notebook 35's current merged catalog.")
        frame[column] = pd.to_datetime(frame[column], utc=True, errors="raise").dt.tz_localize(None)
        if frame[column].isna().any():
            raise ValueError(f"Missing timestamps in {column}.")
    if ((frame.event_end < frame.event_start) | (frame.event_peak < frame.event_start)
            | (frame.event_peak > frame.event_end)).any():
        raise ValueError("Each event must satisfy start ≤ peak ≤ end.")
    frame["review_key"] = frame[list(DATE_COLUMNS)].apply(lambda row: "|".join(t.isoformat() for t in row), axis=1)
    if frame.review_key.duplicated().any():
        raise ValueError("Duplicate event start/end/peak identities in catalog.")
    frame = frame.sort_values("event_peak").reset_index(drop=True)
    numbers = pd.to_numeric(frame.get("event_number", pd.Series(index=frame.index, dtype=float)), errors="coerce")
    valid = numbers.notna().all() and numbers.gt(0).all() and numbers.eq(numbers.round()).all() and numbers.is_unique
    frame["event_number"] = numbers.astype(int) if valid else np.arange(1, len(frame) + 1)
    span = (frame.event_end - frame.event_start) / pd.Timedelta(hours=1) + 1
    duration = pd.to_numeric(frame.get("episode_span_hours", frame.get("duration_hours", span)), errors="coerce")
    frame["event_duration_hours"] = duration.where(duration.gt(0)).fillna(span)
    frame["target_valid_time"] = frame.event_start.map(nearest_start_date_time)
    frame["onset_offset_hours"] = (frame.target_valid_time - frame.event_start) / pd.Timedelta(hours=1)
    if "event_peak_D_1e5_s-1" not in frame:
        frame["event_peak_D_1e5_s-1"] = pd.to_numeric(frame.get("event_peak_D_s-1", np.nan), errors="coerce") * 1e5
    return frame


def read_catalog(catalog_path, review_dir, study_start="2023-01-01", study_end=None):
    """Read original inventory; merge reviews only on exact event windows."""
    catalog_path, review_dir = Path(catalog_path), Path(review_dir)
    if not catalog_path.exists():
        raise FileNotFoundError(f"Current catalog not found: {catalog_path}. Check the Drive path used in Notebook 35.")
    catalog = normalise_catalog(pd.read_csv(catalog_path))
    raw_path = catalog_path.parent / "jpcz_catalog_ndjf.csv"
    if raw_path.exists():
        raw = normalise_catalog(pd.read_csv(raw_path))
        missing = [r.event_start for r in raw.itertuples() if not
                   ((catalog.event_start <= r.event_start) & (catalog.event_end >= r.event_end)).any()]
        if missing:
            raise ValueError(f"Merged catalog omits {len(missing)} saved raw fragments. Refresh it with Notebook 06, as in Notebook 35.")
    for filename in ("current_catalog_review_inventory.csv", "case_review_labels.csv"):
        path = review_dir / filename
        if not path.exists():
            continue
        saved = pd.read_csv(path)
        if all(c in saved for c in DATE_COLUMNS):
            for column in DATE_COLUMNS:
                saved[column] = pd.to_datetime(saved[column], utc=True, errors="raise").dt.tz_localize(None)
            saved["review_key"] = saved[list(DATE_COLUMNS)].apply(lambda row: "|".join(t.isoformat() for t in row), axis=1)
        if "review_key" not in saved:
            continue
        columns = [c for c in saved if c.startswith("historical_") or c in
                   ("case_label", "case_notes", "vortex_detected", "vortex_scan_status")]
        if saved.review_key.duplicated().any():
            raise ValueError(f"Duplicate event identities in {path}; resolve before joining reviews.")
        saved = saved.set_index("review_key")
        for column in columns:
            mapped = catalog.review_key.map(saved[column])
            catalog[column] = mapped.combine_first(catalog[column]) if column in catalog else mapped
    catalog = refresh_vortex_labels(catalog, review_dir)
    start = utc_naive(study_start)
    included = catalog.event_start.ge(start)
    reasons = pd.Series("included", index=catalog.index)
    reasons.loc[~included] = "before configured study start (training cutoff unverified)"
    if study_end:
        after = catalog.event_start.ge(utc_naive(study_end))
        included &= ~after
        reasons.loc[after] = "at/after configured study end (exclusive)"
    report = catalog.copy()
    report["selection_status"] = reasons
    report["training_cutoff_status"] = "operational archive training cutoff not verified"
    report["forecast_availability"] = "not checked; use WN2 access check for each event"
    return catalog.loc[included].copy(), report


def forecast_requests(row):
    target = nearest_start_date_time(row["event_start"])
    return pd.DataFrame([{
        "lead_hours": lead, "lead_days": lead // 24,
        "init_time": target - pd.Timedelta(hours=lead), "valid_time": target,
        "event_start": utc_naive(row["event_start"]),
        "onset_offset_hours": (target - utc_naive(row["event_start"])) / pd.Timedelta(hours=1),
    } for lead in LEAD_HOURS])


class GoogleCloudArchive:
    """Metadata-first discovery of WN2 runs, without recursive archive downloads."""
    def __init__(self, credentials, project_id="", root=ARCHIVE_ROOT, overrides=None, *, lazy_indexing=False, input_cache_dir=None):
        import gcsfs
        self.root = root.removeprefix("gs://").rstrip("/")
        if self.root != ARCHIVE_ROOT:
            raise ValueError("Use the WN2 ensemble archive root; do not use /success or the mean archive.")
        self.storage_options = {"token": credentials}
        if project_id.strip():
            self.storage_options["project"] = project_id.strip()
        self.fs = gcsfs.GCSFileSystem(**self.storage_options)
        self.lazy_indexing = lazy_indexing
        self.input_cache_dir = Path(input_cache_dir) if input_cache_dir else None
        self.overrides = {iso_utc(k): str(v).removeprefix("gs://").rstrip("/") for k, v in (overrides or {}).items()}
        self.listings = {}
        self.datasets = {}
        self.resolved = {}

    def _list(self, path):
        if path not in self.listings:
            self.listings[path] = self.fs.ls(path, detail=False)
        return self.listings[path]

    def _is_store(self, path):
        return self.fs.exists(path + "/zarr.json") or self.fs.exists(path + "/.zgroup")

    def resolve(self, init_time):
        init_time = utc_naive(init_time)
        identity = iso_utc(init_time)
        if identity in self.resolved:
            return self.resolved[identity]
        if identity in self.overrides:
            candidates = [self.overrides[identity]]
        else:
            partitions = []
            for path in self._list(self.root):
                match = re.fullmatch(r"(\d{4})_to_(\d{4}|present)", path.rstrip("/").rsplit("/", 1)[-1])
                if match and int(match[1]) <= init_time.year <= (9999 if match[2] == "present" else int(match[2])):
                    partitions.append(path.rstrip("/"))
            prefix = init_time.strftime("%Y%m%d_%Hhr")
            candidates = []
            for partition in partitions:
                historical = partition + "/predictions.zarr"
                if self._is_store(historical):
                    # Labels such as 2022_to_2023 use an exclusive upper boundary.
                    # Inspect the actual initialization coordinate, not the label.
                    if contains_initialization(self.open(historical), init_time):
                        candidates.append(historical)
                    continue
                for path in self._list(partition):
                    name = path.rstrip("/").rsplit("/", 1)[-1]
                    if name.startswith(prefix + "_") or name == prefix:
                        candidates.append(path.rstrip("/") + "/predictions.zarr")
        stores = sorted({path for path in candidates if self._is_store(path)})
        if not stores:
            raise FileNotFoundError(f"No WN2 store contains exact initialization {identity}. "
                                    "Checked historical period stores and dated run folders; no other run was substituted.")
        if len(stores) != 1:
            raise ValueError(f"Multiple WN2 stores for {identity}: {stores}. Set RUN_OVERRIDES to an explicit store.")
        if not stores[0].startswith(self.root + "/"):
            raise ValueError("Run overrides must remain within the WN2 ensemble archive.")
        self.resolved[identity] = stores[0]
        return stores[0]

    def open(self, store):
        if store not in self.datasets:
            # xarray/zarr construct an async-capable fsspec store from this URL.
            ds = xr.open_zarr("gs://" + store, storage_options=self.storage_options,
                              chunks=None if self.lazy_indexing else {}, consolidated=None)
            if "sample" not in ds.coords or "sample" not in ds.dims:
                ds.close()
                raise ValueError("Expected the member-resolved WN2 sample coordinate, not ensemble-mean data.")
            # Keep only a few metadata handles; fields remain lazy.
            if len(self.datasets) >= 4:
                self.datasets.pop(next(iter(self.datasets))).close()
            self.datasets[store] = ds
        return self.datasets[store]

    def run(self, init_time):
        store = self.resolve(init_time)
        return store, canonical_run(self.open(store), init_time)

    def clear(self):
        for ds in self.datasets.values():
            ds.close()
        self.datasets.clear()
        self.listings.clear()
        self.resolved.clear()
        self.fs.invalidate_cache()


def contains_initialization(ds, init_time):
    init_time = utc_naive(init_time)
    if "prediction_timedelta" in ds.dims and "time" in ds.coords:
        if not np.issubdtype(ds.time.dtype, np.datetime64):
            raise ValueError("Historical WN2 time must decode to initialization datetimes.")
        return np.datetime64(init_time, "ns") in ds.time.values
    if "init_time" in ds.coords and ds.init_time.size == 1:
        return utc_naive(ds.init_time.values.reshape(-1)[0]) == init_time
    raise ValueError("Unrecognized WN2 initialization-coordinate layout.")


def canonical_run(ds, init_time):
    """Normalize historical and per-run layouts without loading forecast fields."""
    init_time = utc_naive(init_time)
    if not contains_initialization(ds, init_time):
        raise ValueError(f"Store does not contain exact initialization {iso_utc(init_time)}.")
    if "prediction_timedelta" in ds.dims:
        run = ds.sel(time=np.datetime64(init_time, "ns"), drop=True)
        if not np.issubdtype(run.prediction_timedelta.dtype, np.timedelta64):
            raise ValueError("Historical prediction_timedelta must decode to timedeltas.")
        run = run.rename({"prediction_timedelta": "time"})
        if "lead_time_secs" in run.coords:
            expected = run.time.values / np.timedelta64(1, "s")
            if not np.array_equal(run.lead_time_secs.values, expected):
                raise ValueError("Historical lead_time_secs disagrees with prediction_timedelta.")
        run = run.assign_coords(init_time=np.datetime64(init_time, "ns"),
                                datetime=("time", np.datetime64(init_time, "ns") + run.time.values))
        layout = "historical_period_store"
    else:
        run, layout = ds, "dated_run_store"
    run = run.copy(deep=False)
    run.attrs = {**run.attrs, "source_layout": layout}
    for name in run.data_vars:
        attrs = dict(run[name].attrs)
        if not attrs.get("units") and name in WN2_SCHEMA_UNITS:
            attrs.update(units=WN2_SCHEMA_UNITS[name],
                         units_source="official WN2 schema; absent from source variable attributes")
        else:
            attrs["units_source"] = "source variable attributes"
        run[name].attrs = attrs
    return run


def select_forecast(ds, init_time, lead_hours, member, variables, level, domain, *, level_stencil=None):
    """Select an exact lead, level, and region, keeping all samples when member is None."""
    required = {"sample", "time", "lat", "lon", "init_time"}
    if not required.issubset(ds.coords):
        raise ValueError(f"WN2 schema missing coordinates: {sorted(required - set(ds.coords))}.")
    init_time = utc_naive(init_time)
    if ds.init_time.size != 1 or utc_naive(ds.init_time.values.reshape(-1)[0]) != init_time:
        raise ValueError("Store init_time does not match the requested initialization.")
    if not np.issubdtype(ds.time.dtype, np.timedelta64):
        raise ValueError("WN2 time must decode to forecast-lead timedeltas; refusing ambiguous time units.")
    if not (0 < lead_hours <= 360 and lead_hours % 6 == 0):
        raise ValueError("Expected a six-hour lead within the 15-day forecast horizon.")
    lead = pd.Timedelta(hours=lead_hours)
    if member is not None and member not in ds.sample.values:
        raise ValueError(f"Member label {member!r} is not available in this run.")
    if np.timedelta64(lead.value, "ns") not in ds.time.values:
        raise ValueError(f"Exact {lead_hours}-hour lead is unavailable; no nearest forecast was substituted.")
    missing = set(variables) - set(ds.data_vars)
    if missing:
        raise ValueError(f"Requested fields unavailable: {sorted(missing)}.")
    snap = ds[list(variables)].sel(time=lead)
    if member is not None:
        snap = snap.sel(sample=member)
    target = init_time + lead
    if "datetime" in snap.coords and (snap.datetime.size != 1 or utc_naive(snap.datetime.values.reshape(-1)[0]) != target):
        raise ValueError("WN2 datetime is inconsistent with initialization + lead.")
    if level is not None:
        if "level" not in snap.dims or level not in snap.level.values:
            raise ValueError(f"Pressure level {level} hPa is unavailable.")
        if level_stencil is not None:
            if len(level_stencil) != 3 or len(set(level_stencil)) != 3 or level not in level_stencil:
                raise ValueError("PV needs three distinct pressure levels including the target.")
            if not set(level_stencil).issubset(set(snap.level.values)):
                raise ValueError("Requested PV pressure stencil is unavailable.")
            snap = snap.sel(level=list(level_stencil)).sortby("level")
        else:
            snap = snap.sel(level=level)
    elif "level" in snap.dims:
        raise ValueError("Surface preset unexpectedly retains a pressure-level dimension.")
    west, east, south, north = domain
    if not (-180 <= west < east <= 360 and -90 <= south < north <= 90):
        raise ValueError("Invalid geographic domain.")
    # The atlas domains are in East Asia, valid in both common longitude conventions.
    for coordinate, lo, hi in (("lat", south, north), ("lon", west, east)):
        values = np.asarray(snap[coordinate].values)
        if values.ndim != 1 or len(values) < 2:
            raise ValueError(f"Expected a one-dimensional {coordinate} grid.")
        snap = snap.sel({coordinate: slice(lo, hi) if values[0] < values[-1] else slice(hi, lo)})
        if snap.sizes[coordinate] < 2:
            raise ValueError(f"Requested region has too few {coordinate} cells.")
        actual = snap[coordinate].values
        if float(np.min(actual)) > lo + .26 or float(np.max(actual)) < hi - .26:
            raise ValueError(f"Forecast grid does not cover requested {coordinate} bounds.")
    sample_dims = ("sample",) if member is None else ()
    pressure_dims = ("level",) if level_stencil is not None else ()
    order = sample_dims + pressure_dims + ("lat", "lon")
    for name in variables:
        if set(snap[name].dims) != set(order):
            raise ValueError(f"Unexpected remaining dimensions for {name}: {snap[name].dims}.")
    return snap.transpose(*order).sortby("lat").sortby("lon")


def pv_pressure_stencil(available_levels, target):
    levels = sorted({int(p) for p in available_levels})
    if target not in levels or len(levels) < 3:
        raise ValueError("Ertel PV requires the target and at least three available pressure levels.")
    index = levels.index(target)
    first = max(0, min(index - 1, len(levels) - 3))
    stencil = levels[first:first + 3]
    method = "centered" if stencil[0] < target < stencil[-1] else "one-sided at pressure-range boundary"
    return stencil, method


def diagnostic_settings(ds, spec, domain):
    if spec.key not in DERIVED_FIELDS:
        return None
    settings = {"algorithm_version": 2, "numerical_precision": "float64", "earth_radius_m": EARTH_RADIUS_M,
                "horizontal_derivative": "second-order differences on spherical latitude/longitude grid",
                "halo_cells": 2, "below_ground_mask": "not applied; source surface pressure/terrain unavailable",
                "temporal_operation": "instantaneous forecast snapshot"}
    if spec.key == "ertel_pv":
        stencil, method = pv_pressure_stencil(ds.level.values, spec.level)
        settings.update(pressure_stencil_hpa=stencil, vertical_derivative=method,
                        pressure_derivative_units="Pa", theta_reference_hpa=1000,
                        theta_kappa=KAPPA, earth_omega_s1=EARTH_OMEGA_S1,
                        gravity_ms2=STANDARD_GRAVITY, pv_definition="dry hydrostatic pressure-coordinate Ertel PV",
                        formula="-g * (du_dp*dtheta_dy - dv_dp*dtheta_dx + (zeta+f)*dtheta_dp)",
                        display_multiplier=1e6)
    else:
        settings.update(formula="[du/dlambda + d(v*cos(phi))/dphi] / [R*cos(phi)]", display_multiplier=1e5)
    return settings


def diagnostic_domain(ds, domain):
    """Include two source cells outside map bounds when available, then crop after differentiation."""
    lat, lon = ds.lat.values, ds.lon.values
    dlat, dlon = abs(float(lat[1] - lat[0])), abs(float(lon[1] - lon[0]))
    west, east, south, north = domain
    return (max(west - 2 * dlon, float(np.min(lon))), min(east + 2 * dlon, float(np.max(lon))),
            max(south - 2 * dlat, float(np.min(lat))), min(north + 2 * dlat, float(np.max(lat))))


def compact_source_load(dataset):
    """Decode one selected variable, copy its small arrays, and release its source buffer."""
    import zarr
    from numcodecs import blosc
    blosc.set_nthreads(1)
    selected = dataset.copy(deep=False)
    with zarr.config.set({'async.concurrency': 1, 'threading.max_workers': 1}):
        for name in list(selected.data_vars):
            loaded = selected[name].load(scheduler='synchronous')
            selected[name] = xr.DataArray(np.array(loaded.values, copy=True, order='C'),
                dims=loaded.dims, coords={d:np.array(loaded[d].values, copy=True) for d in loaded.dims},
                attrs=dict(loaded.attrs))
            del loaded
            gc.collect()
    return selected


def derive_field(snap, spec, domain):
    """Dry hydrostatic PV and signed horizontal divergence with spherical metric terms.

    u/v are eastward/northward winds. Pressure is in Pa; theta references 1000 hPa.
    PV includes both vertical-shear/horizontal-theta terms, not just (zeta+f)*dtheta/dp.
    """
    for name in spec.variables:
        expected = {"kelvin"} if name == "temperature" else {"wind"}
        if unit_kind(snap[name].attrs.get("units", "")) not in expected:
            raise ValueError(f"Unexpected units for derived-field input {name}: {snap[name].attrs.get('units')!r}.")
    # Only regional inputs are retained; original packed source chunks are decoded serially.
    snap = compact_source_load(snap)
    phi = np.deg2rad(np.asarray(snap.lat.values, dtype=float))
    lam = np.deg2rad(np.asarray(snap.lon.values, dtype=float))
    if len(phi) < 3 or len(lam) < 3 or np.any(np.abs(np.cos(phi)) < 1e-6):
        raise ValueError("Diagnostics need at least three latitudes/longitudes away from the poles.")
    if np.any(np.diff(phi) <= 0) or np.any(np.diff(lam) <= 0):
        raise ValueError("Diagnostic latitude/longitude must increase; select_forecast normalizes ordering.")
    cosphi = np.cos(phi)[:, None]
    u = np.asarray(snap.u_component_of_wind.values, dtype=float)
    v = np.asarray(snap.v_component_of_wind.values, dtype=float)
    if spec.key == "divergence":
        values = (np.gradient(u, lam, axis=-1, edge_order=2)
                  + np.gradient(v * cosphi, phi, axis=-2, edge_order=2)) / (EARTH_RADIUS_M * cosphi) * 1e5
        dims = snap.u_component_of_wind.dims
    elif spec.key == "ertel_pv":
        if "sample" in snap.dims:
            raise ValueError("Select one member for Ertel PV.")
        pressure = snap.level.values.astype(float) * 100.0
        if len(pressure) != 3 or np.any(np.diff(pressure) <= 0):
            raise ValueError("PV needs three distinct, increasing pressure levels.")
        theta = np.asarray(snap.temperature.values, dtype=float) * (100000.0 / pressure[:, None, None]) ** KAPPA
        theta_p = np.gradient(theta, pressure, axis=0, edge_order=2)
        u_p = np.gradient(u, pressure, axis=0, edge_order=2)
        v_p = np.gradient(v, pressure, axis=0, edge_order=2)
        theta_x = np.gradient(theta, lam, axis=-1, edge_order=2) / (EARTH_RADIUS_M * cosphi)
        theta_y = np.gradient(theta, phi, axis=-2, edge_order=2) / EARTH_RADIUS_M
        zeta = (np.gradient(v, lam, axis=-1, edge_order=2)
                - np.gradient(u * cosphi, phi, axis=-2, edge_order=2)) / (EARTH_RADIUS_M * cosphi)
        coriolis = 2 * EARTH_OMEGA_S1 * np.sin(phi)[:, None]
        pv = -STANDARD_GRAVITY * (u_p * theta_y - v_p * theta_x + (zeta + coriolis) * theta_p)
        index = list(snap.level.values).index(spec.level)
        values = pv[index] * 1e6
        dims = ("lat", "lon")
    else:
        raise ValueError(f"Unknown derived diagnostic {spec.key}.")
    result = xr.Dataset({"value": (dims, values)}, coords={dim: snap[dim] for dim in dims})
    west, east, south, north = domain
    result = result.sel(lat=slice(south, north), lon=slice(west, east)).reset_coords(drop=True)
    result.value.attrs = {"units": spec.units, "long_name": spec.title}
    return result


def unit_kind(units):
    value = str(units).lower().strip().replace(" ", "").replace("**", "^")
    value = value.replace("²", "2").replace("⁻¹", "-1").replace("⁻²", "-2").replace("^", "")
    value = value.replace("(", "").replace(")", "").replace("·", "")
    groups = {
        "geopotential": {"m2/s2", "m2s-2"}, "height": {"m", "gpm", "geopotentialmetres", "geopotentialmeters"},
        "wind": {"m/s", "ms-1"}, "kelvin": {"k", "kelvin"}, "celsius": {"degc", "celsius", "°c"},
        "humidity": {"kg/kg", "kgkg-1", "1"}, "pa": {"pa"}, "hpa": {"hpa", "mbar", "mb"},
        "vertical_velocity": {"pa/s", "pas-1"}, "millimeters": {"mm"},
    }
    return next((kind for kind, choices in groups.items() if value in choices), None)


def convert_field(snap, spec):
    def field(name, accepted):
        result = snap[name]
        kind = unit_kind(result.attrs.get("units", ""))
        if kind not in accepted:
            raise ValueError(f"Unexpected/missing units for {name}: {result.attrs.get('units')!r}. Check schema before converting.")
        return result, kind

    if spec.conversion == "wind":
        u, _ = field(spec.variables[0], {"wind"})
        v, _ = field(spec.variables[1], {"wind"})
        result = xr.Dataset({"value": np.hypot(u, v), "u": u, "v": v})
    else:
        accepted = {"height": {"geopotential", "height"}, "temperature": {"kelvin", "celsius"},
                    "humidity": {"humidity"}, "pressure": {"pa", "hpa"},
                    "precipitation": {"height", "millimeters"}, "identity": {"vertical_velocity"},
                    "wind_component": {"wind"}, "geopotential": {"geopotential"}}
        value, kind = field(spec.variables[0], accepted[spec.conversion])
        if spec.conversion == "height" and kind == "geopotential":
            value = value / 9.80665
        elif spec.conversion == "temperature" and kind == "kelvin":
            value = value - 273.15
        elif spec.conversion == "humidity":
            value = value * 1000
        elif spec.conversion == "pressure" and kind == "pa":
            value = value / 100
        elif spec.conversion == "precipitation" and kind == "height":
            value = value * 1000
        result = xr.Dataset({"value": value})
    # Cache only spatial fields, not scalar time/level coordinates that differ per row.
    result = result.reset_coords(drop=True)
    result["value"].attrs = {"units": spec.units, "long_name": spec.title}
    if "u" in result:
        result.u.attrs = result.v.attrs = {"units": "m s-1"}
    return result


def cache_identity(store, init_time, lead, member, spec, domain, diagnostic=None):
    identity = {"schema": SCHEMA_VERSION, "store": store, "init_time": iso_utc(init_time),
            "lead_hours": int(lead), "member": str(member), "field": spec.key,
            "variables": list(spec.variables), "level_hpa": spec.level,
            "conversion": spec.conversion, "domain": list(domain)}
    if diagnostic is not None:
        identity["diagnostic"] = diagnostic
    return identity


def forecast_input_group(archive, store, ds, request, spec, domain, start, end, *, level_stencil=None):
    """Shared regional source-variable cache; compatible 36/37 reads reuse u/v without decoding again."""
    selected = select_forecast(ds,request['init_time'],int(request['lead_hours']),None,
        spec.variables,spec.level,domain,level_stencil=level_stencil).isel(sample=slice(start,end))
    root=getattr(archive,'input_cache_dir',None)
    if root is None:return compact_source_load(selected)
    pieces=[]
    for name in spec.variables:
        identity=dict(version=1,store=store,init_time=iso_utc(request['init_time']),lead_hours=int(request['lead_hours']),
            members=[str(x) for x in selected.sample.values.tolist()],variable=name,
            member_coordinate_dtype=str(selected.sample.dtype),
            level_hpa=spec.level,level_stencil=level_stencil,domain=list(domain))
        encoded=json.dumps(identity,sort_keys=True)
        path=Path(root)/('input_'+hashlib.sha256(encoded.encode()).hexdigest()[:24]+'.nc')
        if path.is_file():
            with xr.open_dataset(path) as saved:
                if saved.attrs.get('identity')!=encoded:raise ValueError('Regional input cache identity mismatch.')
                piece=saved.load()
        else:
            piece=compact_source_load(selected[[name]]).reset_coords(drop=True)
            # Do not freeze an unfinished archive chunk into a persistent cache.
            if np.isfinite(piece[name].values).all():
                path.parent.mkdir(parents=True,exist_ok=True);piece.attrs={'identity':encoded}
                temporary=path.with_suffix('.tmp.nc');piece.to_netcdf(temporary);temporary.replace(path)
        pieces.append(piece)
    # combine_attrs='drop' also drops DATA-VARIABLE attrs, including the wind units
    # required by the derivative/unit checks. Drop only dataset-level cache identity;
    # the u/v pieces intentionally have different identities.
    merged=xr.merge(pieces,combine_attrs='drop')
    for piece in pieces:
        for name in piece.data_vars:
            merged[name].attrs=dict(piece[name].attrs)
    return merged


def load_panel(archive, cache_dir, request, member, spec, domain):
    store, ds = archive.run(request["init_time"])
    diagnostic = diagnostic_settings(ds, spec, domain)
    identity = cache_identity(store, request["init_time"], request["lead_hours"], member, spec, domain, diagnostic)
    encoded = json.dumps(identity, sort_keys=True)
    digest = hashlib.sha256(encoded.encode()).hexdigest()[:24]
    path = Path(cache_dir) / f"wn2_{digest}.nc"
    if path.exists():
        with xr.open_dataset(path) as cached:
            if cached.attrs.get("identity") != encoded:
                raise ValueError(f"Cache identity mismatch: {path}.")
            result = cached.load()
        hit = True
    else:
        selection_domain = diagnostic_domain(ds, domain) if diagnostic else domain
        stencil = diagnostic.get("pressure_stencil_hpa") if diagnostic else None
        if getattr(archive,'input_cache_dir',None) is not None:
            labels=ds.sample.values.tolist();position=labels.index(member)
            start=(position//4)*4;end=min(start+4,len(labels))
            selection_domain=diagnostic_domain(ds,domain)
            snap=forecast_input_group(archive,store,ds,request,spec,selection_domain,start,end,level_stencil=stencil).sel(sample=member)
        else:
            snap = select_forecast(ds, request["init_time"], int(request["lead_hours"]), member,
                                   spec.variables, spec.level, selection_domain, level_stencil=stencil)
        # Historical chunks include 4 members and all 13 levels over the globe.
        # Decode chunks serially to limit RAM; only the selected regional slice is retained.
        result = derive_field(snap, spec, domain) if diagnostic else convert_field(compact_source_load(snap), spec)
        if not diagnostic:
            result=result.sel(lat=slice(domain[2],domain[3]),lon=slice(domain[0],domain[1]))
        if not np.isfinite(result.value.values).any():
            raise ValueError("Selected forecast field has no finite data in this region.")
        result.attrs["identity"] = encoded
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".tmp.nc")
        result.to_netcdf(temporary)
        temporary.replace(path)
        hit = False
    record = {**identity, "event_number": int(request["event_number"]), "review_key": request["review_key"],
              "event_start": iso_utc(request["event_start"]), "event_end": iso_utc(request["event_end"]),
              "event_peak": iso_utc(request["event_peak"]), "valid_time": iso_utc(request["valid_time"]),
              "onset_offset_hours": float(request["onset_offset_hours"]), "units": spec.units,
              "field_title": spec.title, "display_min": spec.limits[0], "display_max": spec.limits[1],
              "palette": spec.palette, "cache_hit": hit, "cache_file": str(path), "status": "complete",
              "source_layout": ds.attrs.get("source_layout", "documented_single_run_schema"),
              "source_variable_units": json.dumps({name: {"units": ds[name].attrs.get("units"),
                  "units_source": ds[name].attrs.get("units_source", "source variable attributes")} for name in spec.variables}, sort_keys=True),
              "plot_style_version": PLOT_STYLE_VERSION,
              "wind_vector_spacing_degrees": 1 if domain[1] - domain[0] <= 25 else 2,
              "wind_quiver_scale": 350, "wind_key_ms": 10,
              "height_contour_interval_m": (60 if spec.level is not None and spec.level <= 500 else 30) if spec.conversion == "height" else None,
              "training_cutoff_status": "operational archive training cutoff not verified"}
    return {"data": result, "metadata": record, "spec": spec, "domain": tuple(domain)}


def event_panels(archive, cache_dir, row, member, specs, domains, progress=print):
    if not specs or len(specs) != len(domains):
        raise ValueError("Choose one or more fields with a domain for each plot row.")
    requests = forecast_requests(row)
    panels = []
    for row_index, (spec, domain) in enumerate(zip(specs, domains)):
        for request in requests.to_dict("records"):
            request.update(event_number=int(row.event_number), review_key=row.review_key,
                           event_end=row.event_end, event_peak=row.event_peak)
            progress(f"Row {row_index + 1} | {spec.title} | {request['lead_days']}-day lead | member {member}")
            panel = load_panel(archive, cache_dir, request, member, spec, domain)
            panel["metadata"].update(vortex_label_metadata(row))
            panel["metadata"]["row_index"] = row_index
            panels.append(panel)
    verify_six_panels(panels)
    return panels


def verify_six_panels(panels):
    if not panels or len(panels) % 3:
        raise ValueError("A complete event needs three leads for each field row.")
    for key in ("member", "valid_time", "review_key"):
        if len({p["metadata"][key] for p in panels}) != 1:
            raise ValueError(f"Six-panel group mixes {key} values.")
    rows = sorted({p['metadata']['row_index'] for p in panels})
    if rows != list(range(len(panels)//3)):
        raise ValueError('Plot row indices are not consecutive.')
    for row in rows:
        group = [p for p in panels if p["metadata"]["row_index"] == row]
        if sorted(p["metadata"]["lead_hours"] for p in group) != list(LEAD_HOURS):
            raise ValueError("Each row must contain exactly the 1-, 5-, and 10-day leads.")
        if len({json.dumps([p["spec"].key, p["spec"].level, p["spec"].limits, p["domain"]]) for p in group}) != 1:
            raise ValueError("Plot settings differ within a row.")
    for panel in panels:
        m = panel["metadata"]
        if utc_naive(m["init_time"]) + pd.Timedelta(hours=m["lead_hours"]) != utc_naive(m["valid_time"]):
            raise ValueError("Initialization + forecast lead does not equal valid time.")


def prepare_map_assets():
    """Resolve Natural Earth files once, before drawing, so network errors are clear."""
    import cartopy.io.shapereader as shpreader
    for category, name in (("physical", "coastline"), ("cultural", "admin_0_boundary_lines_land")):
        shpreader.natural_earth(resolution="110m", category=category, name=name)


def draw_panel(ax, panel, features=True):
    from matplotlib.colors import TwoSlopeNorm
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    spec, ds, metadata = panel["spec"], panel["data"], panel["metadata"]
    crs = ccrs.PlateCarree()
    ax.set_extent(panel["domain"], crs=crs)
    low, high = spec.limits
    if spec.key in DERIVED_FIELDS:
        if not low < 0 < high:
            raise ValueError("Divergence/PV color scales must include negative values, zero, and positive values.")
        levels = np.r_[np.linspace(low, 0, 7), np.linspace(0, high, 7)[1:]]
        norm = TwoSlopeNorm(vmin=low, vcenter=0, vmax=high)
    else:
        levels, norm = np.linspace(low, high, 13), None
    shade = ax.contourf(ds.lon.values, ds.lat.values, ds.value.values, levels=levels,
                        cmap=spec.palette, norm=norm, extend="both", transform=crs)
    if spec.conversion == "height":
        interval = 60 if spec.level is not None and spec.level <= 500 else 30
        lo, hi = float(ds.value.min()), float(ds.value.max())
        contours = np.arange(np.floor(lo / interval) * interval, hi + interval, interval)
        if len(contours) > 1 and hi > lo:
            cs = ax.contour(ds.lon.values, ds.lat.values, ds.value.values,
                            levels=contours, colors="0.2", linewidths=.6, transform=crs)
            ax.clabel(cs, fontsize=7, fmt="%.0f", inline=True)
    if "u" in ds:
        # Regional winds get 1° vectors; the larger circulation map gets 2°.
        spacing = metadata["wind_vector_spacing_degrees"]
        lat_step = max(1, int(round(spacing / abs(float(ds.lat[1] - ds.lat[0])))))
        lon_step = max(1, int(round(spacing / abs(float(ds.lon[1] - ds.lon[0])))))
        sub = ds.isel(lat=slice(None, None, lat_step), lon=slice(None, None, lon_step))
        arrows = ax.quiver(sub.lon.values, sub.lat.values, sub.u.values, sub.v.values,
                           transform=crs, angles="uv", scale=metadata["wind_quiver_scale"], width=.003)
        ax.quiverkey(arrows, .82, -.10, metadata["wind_key_ms"], "10 m s⁻¹", labelpos="E", coordinates="axes", fontproperties={"size": 8})
    if features:
        ax.coastlines(resolution="110m", linewidth=.7)
        ax.add_feature(cfeature.BORDERS.with_scale("110m"), linewidth=.4, edgecolor=".4")
    for name, vertices in POLYGONS.items():
        xy = np.asarray(vertices + [vertices[0]])
        ax.plot(xy[:, 0], xy[:, 1], color="darkmagenta" if name == "JPCZ polygon" else "darkorange",
                linewidth=1.2, transform=crs, label=name)
    grid = ax.gridlines(draw_labels=True, linewidth=.35, color=".4", alpha=.5)
    grid.top_labels = grid.right_labels = False
    grid.xlabel_style = grid.ylabel_style = {"size": 8}
    title = (f"Observed event: {vortex_plot_label(metadata)}\n"
                 f"{spec.title} | member {metadata['member']} | {metadata['lead_hours'] // 24}-day lead\n"
                 f"Init: {utc_naive(metadata['init_time']):%Y-%m-%d %H:%M} UTC\n"
                 f"Valid: {utc_naive(metadata['valid_time']):%Y-%m-%d %H:%M} UTC")
    if spec.key == "ertel_pv":
        method = metadata["diagnostic"]["vertical_derivative"]
        stencil = "/".join(map(str, metadata["diagnostic"]["pressure_stencil_hpa"]))
        title += f"\nPV stencil {stencil} hPa · {'one-sided' if method.startswith('one-sided') else 'centered'}"
    ax.set_title(title, fontsize=9, pad=9)
    return shade


def colorbar_label(spec):
    return "10⁻⁵ s⁻¹\nblue: convergence; red: divergence" if spec.key == "divergence" else spec.units


def render_event(panels, row, features=True):
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    verify_six_panels(panels)
    rows = len(panels)//3
    fig, axes = plt.subplots(rows, 3, figsize=(17, 3.9*rows+3), squeeze=False,
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.subplots_adjust(left=.045, right=.91, top=.80, bottom=.12, hspace=.52, wspace=.22)
    for index, panel in enumerate(panels):
        artist = draw_panel(axes.flat[index], panel, features)
        if index % 3 == 2:
            position = axes.flat[index].get_position()
            cax = fig.add_axes([.935, position.y0, .012, position.height])
            fig.colorbar(artist, cax=cax, label=colorbar_label(panel["spec"]))
    m = panels[0]["metadata"]
    peak = pd.to_numeric(row.get("event_peak_D_1e5_s-1", np.nan), errors="coerce")
    peak = float(peak) if pd.notna(peak) else float('nan')
    fig.suptitle(f"WeatherNext 2 | JPCZ event {int(row.event_number)} | Observed event: {vortex_plot_label(row)} | ensemble member {m['member']}\n"
                 f"Catalog start: {utc_naive(row.event_start):%Y-%m-%d %H:%M} UTC  |  "
                 f"end: {utc_naive(row.event_end):%Y-%m-%d %H:%M} UTC\n"
                 f"Target: {utc_naive(m['valid_time']):%Y-%m-%d %H:%M} UTC  "
                 f"(target − onset: {m['onset_offset_hours']:+g} h)  |  "
                 f"Peak: {utc_naive(row.event_peak):%Y-%m-%d %H:%M} UTC; D12 = {peak:.2f} × 10⁻⁵ s⁻¹",
                 fontsize=12, y=.985, linespacing=1.6)
    axes[0, 0].legend(loc="lower left", fontsize=7, framealpha=.85)
    fig.text(.05, .045, "1 / 5 / 10 days = exact forecast leads to the same target valid time. "
             "Magenta: JPCZ polygon; orange: coastal wedge.\n"
             "Study window is provisional: operational WN2 training cutoff unverified. "
             "Pressure-level fields/diagnostics are unmasked below the surface (especially 1000 hPa).\n"
             "Source: Google DeepMind / Google Research WeatherNext 2 experimental historical data (CC BY 4.0).",
             fontsize=8, linespacing=1.7)
    return fig


def export_event(panels, row, output_dir, figure=None, features=True):
    """Six standalone panels plus the overview, metadata, settings, and attribution."""
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    verify_six_panels(panels)
    metadata = [p["metadata"] for p in panels]
    # Include plotting settings and event identity; cache-hit status is irrelevant.
    stable = [{k: v for k, v in m.items() if k not in ("cache_hit", "cache_file")} for m in metadata]
    signature = hashlib.sha256(json.dumps(stable, sort_keys=True).encode()).hexdigest()[:12]
    m = metadata[0]
    safe_member = re.sub(r"[^A-Za-z0-9_.-]", "_", m["member"])
    destination = Path(output_dir) / (f"event_{int(row.event_number):04d}_start{utc_naive(row.event_start):%Y%m%dT%H%M}"
                                    f"_member{safe_member}_{signature}")
    destination.mkdir(parents=True, exist_ok=True)
    required = [destination / f"overview.{ext}" for ext in ("png", "pdf")]
    for panel in panels:
        item = panel["metadata"]
        stem = f"row{item['row_index'] + 1}_{panel['spec'].key}_lead{item['lead_hours']:03d}h"
        required.extend(destination / f"{stem}.{ext}" for ext in ("png", "pdf"))
    required.extend(destination / name for name in ("forecast_metadata.csv", "settings.json", "attribution.txt"))
    marker = destination / "complete.json"
    if marker.exists() and all(path.exists() and path.stat().st_size > 0 for path in required):
        saved = json.loads(marker.read_text())
        if saved.get("signature") == signature:
            return destination
    marker.unlink(missing_ok=True)
    own_figure = figure is None
    figure = render_event(panels, row, features) if own_figure else figure
    for extension in ("png", "pdf"):
        figure.savefig(destination / f"overview.{extension}", dpi=180, bbox_inches="tight")
    if own_figure:
        plt.close(figure)
    for panel in panels:
        item = panel["metadata"]
        fig, ax = plt.subplots(figsize=(8, 6.8), subplot_kw={"projection": ccrs.PlateCarree()})
        fig.subplots_adjust(bottom=.20, top=.80)
        artist = draw_panel(ax, panel, features)
        fig.colorbar(artist, ax=ax, shrink=.8, pad=.08, label=colorbar_label(panel["spec"]))
        fig.suptitle(f"JPCZ event {int(row.event_number)} | WN2 ensemble member {item['member']}\n"
                     f"Onset {utc_naive(row.event_start):%Y-%m-%d %H:%M} UTC; target − onset {item['onset_offset_hours']:+g} h", fontsize=11)
        fig.text(.08, .04, "Google DeepMind / Google Research WN2 historical data, CC BY 4.0.\n"
                 "Magenta: JPCZ polygon; orange: coastal wedge. Pressure-level fields/diagnostics are unmasked below the surface.\n"
                 "Operational model training cutoff unverified; see metadata and attribution.txt.", fontsize=7)
        stem = f"row{item['row_index'] + 1}_{panel['spec'].key}_lead{item['lead_hours']:03d}h"
        for extension in ("png", "pdf"):
            fig.savefig(destination / f"{stem}.{extension}", dpi=180, bbox_inches="tight")
        plt.close(fig)
    pd.DataFrame(metadata).to_csv(destination / "forecast_metadata.csv", index=False)
    (destination / "settings.json").write_text(json.dumps(stable, indent=2, ensure_ascii=False))
    (destination / "attribution.txt").write_text(
        "© 2025 DeepMind Technologies Limited's machine learning models used to create the experimental data made available at "
        "https://developers.google.com/earth-engine/datasets/catalog/projects_gcp-public-data-weathernext_assets_weathernext_2_0_0 "
        "under CC BY 4.0 licence terms. This data is intended for experimental modelling only and is not intended, validated, "
        "or approved for real world use.\n\nOperational WN2 model training cutoff has not been verified.\n")
    # A complete marker is written only after all seven figure pairs and metadata exist.
    (destination / "complete.json").write_text(json.dumps({"signature": signature, "forecast_panels": len(panels), "member": m["member"]}))
    return destination


def find_notebook35_references(review_dir, row):
    records = []
    for path in sorted(Path(review_dir).glob("*_metadata.csv")):
        try:
            saved = pd.read_csv(path)
            if not all(c in saved for c in DATE_COLUMNS):
                continue
            match = pd.Series(True, index=saved.index)
            for column in DATE_COLUMNS:
                match &= pd.to_datetime(saved[column], utc=True, errors="coerce").dt.tz_localize(None).eq(utc_naive(row[column]))
            image = path.with_name(path.name.removesuffix("_metadata.csv") + ".png")
            for item in saved.loc[match].to_dict("records"):
                current = vortex_plot_label(row)
                matches_label = item.get("label_scheme") == VORTEX_LABEL_SCHEME and item.get("label") == current
                records.append({**item, "current_vortex_classification": current, "reference_label_current": matches_label,
                                "metadata_file": str(path), "image_file": str(image) if image.exists() else ""})
        except (ValueError, OSError, pd.errors.ParserError):
            continue
    result = pd.DataFrame(records)
    return result.sort_values("reference_label_current", ascending=False, kind="stable") if len(result) else result



print('Archive helpers loaded. Continue to section 3c.')


In [ ]:
# @title 3c · Load rank and Excel helpers (in this notebook)
"""Onset-aligned verification of WN2 against ERA5 for the two JPCZ regions."""

from __future__ import annotations

from functools import lru_cache
import hashlib
import json
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd
import xarray as xr
import zarr



ERA5_STORE = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
VERIFICATION_DOMAIN = (125., 145., 33., 47.)
ENSEMBLE_SIZE = 64
DIVERGENCE_SPEC = FIELD_PRESETS["divergence"]
DIVERGENCE_COLUMN = "divergence_1e5_s-1"
RANK_GROUPS = ("1–13", "14–26", "27–39", "40–52", "53–65")
CHECKPOINT_VERSION = 1


def polygon_divergence(field):
    """Area-weighted means on the native grid, independently for each polygon."""
    from matplotlib.path import Path as PolygonPath

    lon, lat = np.meshgrid(field.lon.values, field.lat.values)
    points = np.column_stack((lon.ravel(), lat.ravel()))
    means, counts = {}, {}
    for region, vertices in POLYGONS.items():
        mask = PolygonPath(vertices).contains_points(points).reshape(lon.shape)
        weights = np.cos(np.deg2rad(lat[mask]))
        values = field.values[..., mask]
        if not mask.any() or not np.isfinite(values).all():
            raise ValueError(f"{region} lacks a complete finite set of field cells.")
        means[region] = np.average(values, weights=weights, axis=-1)
        counts[region] = int(mask.sum())
    return means, counts


@lru_cache(maxsize=1)
def open_verification_era5():
    # Creating Dask graphs for all decades and variables can exceed Colab RAM
    # before any field is read. Keep backend indexing lazy through subsetting.
    return xr.open_zarr(ERA5_STORE, chunks=None, storage_options={"token": "anon"})


def era5_polygon_values(valid_time, era5, *, allow_era5t=False, spec=None):
    """Evaluate the selected field at the exact ERA5 time and pressure level."""
    spec = spec or DIVERGENCE_SPEC
    target = utc_naive(valid_time)
    stable = era5.attrs.get('valid_time_stop')
    preliminary = era5.attrs.get('valid_time_stop_era5t')
    product = 'ERA5'
    if stable and target > utc_naive(stable):
        if not allow_era5t or not preliminary or target > utc_naive(preliminary):
            raise ValueError(f'ERA5 pending for {iso_utc(target)}; final through {stable}; '
                             f'ERA5T through {preliminary}. No nearest time is substituted.')
        product = 'ERA5T'
    winds = era5[list(spec.variables)].sel(time=target)
    if spec.level is not None:
        winds = winds.sel(level=spec.level)
    winds = winds.rename({"latitude": "lat", "longitude": "lon"})
    west, east, south, north = (diagnostic_domain(winds, VERIFICATION_DOMAIN)
                                if spec.key == 'divergence' else VERIFICATION_DOMAIN)
    latitude = slice(south, north) if winds.lat[0] < winds.lat[-1] else slice(north, south)
    longitude = slice(west, east) if winds.lon[0] < winds.lon[-1] else slice(east, west)
    winds = winds.sel(lat=latitude, lon=longitude)
    if any(abs(actual - expected) > 1e-5 for actual, expected in zip(
            (float(winds.lon.min()), float(winds.lon.max()), float(winds.lat.min()), float(winds.lat.max())),
            (west, east, south, north))):
        raise ValueError("ERA5 grid does not cover the verification domain and its derivative halo.")
    # Decode one wind variable at a time, only after selecting the region.
    winds = compact_source_load(winds)
    winds = winds.transpose("lat", "lon").sortby("lat").sortby("lon")
    field = (derive_field(winds, spec, VERIFICATION_DOMAIN) if spec.key == 'divergence'
             else convert_field(winds, spec)).value
    means, counts = polygon_divergence(field)
    return {"valid_time": iso_utc(target), "source_store": ERA5_STORE, "polygon_cells": counts,
            "reference_product": product, "archive_last_updated": str(era5.attrs.get('last_updated', '')),
            "values": {region: float(value) for region, value in means.items()}}


def forecast_polygon_values(archive, request, progress=print, *, spec=None, group_cache=None,
                            member_start=0, member_stop=64):
    """Read groups of native WN2 samples and immediately reduce each field to two means."""
    spec = spec or DIVERGENCE_SPEC
    store, ds = archive.run(request["init_time"])
    labels = [str(value) for value in ds.sample.values.tolist()]
    if len(labels) != ENSEMBLE_SIZE or len(set(labels)) != ENSEMBLE_SIZE:
        raise ValueError(f"Expected {ENSEMBLE_SIZE} distinct WN2 member labels, found {len(set(labels))}.")
    snap = select_forecast(
        ds, request["init_time"], request["lead_hours"], None, spec.variables,
        spec.level, diagnostic_domain(ds, VERIFICATION_DOMAIN) if spec.key == 'divergence' else VERIFICATION_DOMAIN,
    )
    # Source groups are at most four members; never decode all 64 together.
    if not (0<=member_start<member_stop<=ENSEMBLE_SIZE and member_start%4==0 and member_stop%4==0):
        raise ValueError('Member ranges must cover complete native four-member groups.')
    sizes = (4,) * ((member_stop-member_start)//4)
    values = {region: [] for region in POLYGONS}
    start = member_start
    for size in sizes:
        end = start + size
        progress(f"  Reading {request['lead_days']}-day members {start + 1}–{end} of {ENSEMBLE_SIZE}")
        identity = json.loads(json.dumps(dict(store=store, init_time=iso_utc(request['init_time']),
            lead_hours=int(request['lead_hours']), members=labels[start:end], field=spec.key, level=spec.level,
            units=spec.units, domain=VERIFICATION_DOMAIN, polygons=POLYGONS,
            diagnostic=diagnostic_settings(ds, spec, VERIFICATION_DOMAIN)),sort_keys=True))
        path = Path(group_cache)/f'group_{start:02d}_{end:02d}.json' if group_cache else None
        def calculate_group():
            block = forecast_input_group(archive,store,ds,request,spec,
                diagnostic_domain(ds,VERIFICATION_DOMAIN),start,end)
            field = (derive_field(block, spec, VERIFICATION_DOMAIN) if spec.key == 'divergence'
                     else convert_field(block, spec)).value
            means, counts = polygon_divergence(field)
            return dict(values={region:np.asarray(value).tolist() for region,value in means.items()},
                        polygon_cells=counts)
        group, _ = checkpoint(path, identity, calculate_group) if path else (calculate_group(), False)
        counts = group['polygon_cells']
        for region in POLYGONS:
            if len(group['values'][region]) != size or not np.isfinite(group['values'][region]).all():
                raise ValueError('Invalid saved member-group values.')
            values[region].extend(group['values'][region])
        start = end
    return {"init_time": iso_utc(request["init_time"]), "valid_time": iso_utc(request["valid_time"]),
            "lead_hours": request["lead_hours"], "source_store": store, "members": labels[member_start:member_stop],
            "polygon_cells": counts, "values": values}


def checkpoint_identity(archive, spec=None):
    """Reuse verified values only with the same fields, polygons and source configuration."""
    spec = spec or DIVERGENCE_SPEC
    result = {
        "version": CHECKPOINT_VERSION, "era5_store": ERA5_STORE, "wn2_root": archive.root,
        "wn2_overrides": archive.overrides, "domain": VERIFICATION_DOMAIN, "polygons": POLYGONS,
        "pressure_hpa": spec.level, "ensemble_size": ENSEMBLE_SIZE,
        "diagnostic": diagnostic_settings(None, spec, VERIFICATION_DOMAIN),
    }
    if spec != DIVERGENCE_SPEC:
        result.update(field=spec.key, units=spec.units)
    return result


def atomic_json_file(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as temporary:
        temporary_path = Path(temporary.name)
        try:
            json.dump(payload, temporary, allow_nan=False, indent=2)
        except Exception:
            temporary.close()
            temporary_path.unlink(missing_ok=True)
            raise
    temporary_path.replace(path)


def checkpoint(path, identity, calculate):
    if path.is_file():
        try:
            saved = json.loads(path.read_text())
            if saved["request"] == identity:
                return saved["data"], True
        except (KeyError, ValueError, TypeError):
            pass
    data = calculate()
    atomic_json_file(path, {"request": identity, "data": data})
    return data, False


def observation_rank(members, reference, case_key, *, tie_seed=37):
    """Return the ERA5 rank, its tied interval, and its position relative to the ensemble."""
    values = np.asarray(members, dtype=np.float64)
    if values.shape != (ENSEMBLE_SIZE,) or not np.isfinite(values).all() or not np.isfinite(reference):
        raise ValueError("Ranking requires 64 finite member values and one finite ERA5 value.")
    less = int(np.count_nonzero(values < reference))
    equal = int(np.count_nonzero(values == reference))
    lower, upper = less + 1, less + equal + 1
    key = f"{tie_seed}|{case_key}".encode()
    rng = np.random.default_rng(int.from_bytes(hashlib.sha256(key).digest()[:8], "big"))
    rank = lower if lower == upper else int(rng.integers(lower, upper + 1))
    return {
        "era5_rank": rank, "rank_min_if_tied": lower, "rank_max_if_tied": upper,
        "bin_number": (rank - 1) // 13 + 1,
        "rank_group": RANK_GROUPS[(rank - 1) // 13],
        "below_member_min": bool(reference < values.min()),
        "above_member_max": bool(reference > values.max()),
    }


def case_identity(row):
    return {
        "event_number": int(row.event_number), "review_key": row.review_key,
        "event_start_utc": iso_utc(row.event_start), "event_end_utc": iso_utc(row.event_end),
        "event_peak_utc": iso_utc(row.event_peak), "valid_time_utc": iso_utc(row.target_valid_time),
        "target_minus_onset_hours": float(row.onset_offset_hours),
        "catalog_peak_D12_1e5_s-1": row["event_peak_D_1e5_s-1"],
        "vortex_label": row.get("vortex_plot_label", ""),
        "vortex_scan_status": row.get("vortex_scan_status", ""),
        "vortex_classification": row.get('vortex_classification', vortex_classification(row)),
    }


def run_ensemble_verification(catalog, archive, output_dir, *, era5_reader=open_verification_era5,
                              allow_era5t=False, progress=print, spec=None, job_runner=None):
    """Checkpoint one ERA5 target and each WN2 run/lead independently."""
    spec = spec or DIVERGENCE_SPEC
    identity = checkpoint_identity(archive, spec)
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    cache = Path(output_dir) / "checkpoints" / signature
    member_rows, reference_rows, rank_rows, status_rows = [], [], [], []

    for _, row in catalog.iterrows():
        event = case_identity(row)
        target = utc_naive(row.target_valid_time)
        stamp = target.strftime("%Y%m%dT%H%MZ")
        progress(f"Event {event['event_number']} | verification {event['valid_time_utc']}")
        reference = None
        try:
            progress(f"  ERA5: reading one {spec.title} snapshot (or reusing its checkpoint)")
            reference_path = cache / f"era5_{stamp}.json"
            # Preliminary data can change. Refresh it on every run, including when it
            # has become final. Forecast checkpoints remain independent and reusable.
            if reference_path.is_file():
                try:
                    saved = json.loads(reference_path.read_text())
                    if saved['data'].get('reference_product') == 'ERA5T':
                        reference_path.unlink()
                except (KeyError, ValueError, TypeError):
                    pass
            reference, cached = checkpoint(
                reference_path, {"valid_time": event["valid_time_utc"]},
                lambda: (job_runner('reference', dict(valid_time=iso_utc(target), allow_era5t=allow_era5t, spec=spec.__dict__))
                         if job_runner else era5_polygon_values(target, era5_reader(), allow_era5t=allow_era5t, spec=spec)),
            )
            reference_rows.extend([
                {**event, "polygon": region, "pressure_hpa": spec.level,
                 'field':spec.key, 'field_title':spec.title, 'units':spec.units,
                 'value':reference['values'][region],
                 **({DIVERGENCE_COLUMN:reference['values'][region]} if spec.key=='divergence' else {}),
                 "polygon_cells": reference["polygon_cells"][region], "source_store": reference["source_store"],
                 "reference_product": reference.get('reference_product', 'ERA5')}
                for region in POLYGONS
            ])
            ref_status = "cached" if cached else "computed"
            ref_error = ""
            progress(f"  ERA5 {ref_status}: polygon means saved")
        except Exception as error:
            reference = None
            ref_status = "failed"
            ref_error = f"{type(error).__name__}: {error}"
            progress(f"  ERA5 unavailable: {ref_error}")

        event_ranks = []
        for request in forecast_requests(row).to_dict("records"):
            lead = request["lead_hours"]
            timing = {"lead_hours": lead, "lead_days": request["lead_days"], "init_time_utc": iso_utc(request["init_time"])}
            status = {**event, **timing, "era5_status": ref_status, "era5_error": ref_error,
                      "reference_product": reference.get('reference_product', 'ERA5') if reference else 'pending'}
            try:
                forecast, cached = checkpoint(
                    cache / f"wn2_{stamp}_lead{lead:03d}h.json",
                    {"init_time": timing["init_time_utc"], "lead_hours": lead, "valid_time": event["valid_time_utc"]},
                    lambda: (job_runner('forecast',dict(request={k:iso_utc(v) if isinstance(v,pd.Timestamp) else v
                        for k,v in request.items()}, spec=spec.__dict__, group_cache=str(cache/f'groups_{stamp}_{lead:03d}h')))
                        if job_runner else forecast_polygon_values(archive, request, progress, spec=spec,
                            group_cache=cache/f'groups_{stamp}_{lead:03d}h')),
                )
                members = forecast["members"]
                if len(members) != ENSEMBLE_SIZE or len(set(members)) != ENSEMBLE_SIZE:
                    raise ValueError("Forecast checkpoint does not contain 64 distinct members.")
                case_values, case_ranks = [], []
                for region in POLYGONS:
                    values = np.asarray(forecast["values"][region], dtype=float)
                    if values.shape != (ENSEMBLE_SIZE,) or not np.isfinite(values).all():
                        raise ValueError(f"{region} does not contain 64 finite member values.")
                    info = {**event, **timing, "polygon": region, "pressure_hpa": spec.level,
                            'field':spec.key, 'field_title':spec.title, 'units':spec.units}
                    case_values.extend({**info, "member": member, 'value':value,
                                        **({DIVERGENCE_COLUMN:value} if spec.key=='divergence' else {}),
                                        "polygon_cells": forecast["polygon_cells"][region],
                                        "source_store": forecast["source_store"]} for member, value in zip(members, values))
                    if reference is not None:
                        observed = reference["values"][region]
                        rank = observation_rank(values, observed, f"{row.review_key}|{lead}|{region}")
                        case_ranks.append({**info, "member_count": len(members),
                                           "reference_product": reference.get('reference_product', 'ERA5'),
                                           'era5_value':observed, 'member_min':values.min(),
                                           'member_mean':values.mean(), 'member_max':values.max(),
                                           **({'era5_divergence_1e5_s-1':observed,
                                               'member_min_1e5_s-1':values.min(), 'member_mean_1e5_s-1':values.mean(),
                                               'member_max_1e5_s-1':values.max()} if spec.key=='divergence' else {}), **rank})
                member_rows.extend(case_values)
                event_ranks.extend(case_ranks)
                status.update(wn2_status="cached" if cached else "computed", wn2_error="")
            except Exception as error:
                message = f"{type(error).__name__}: {error}"
                status.update(wn2_status="failed", wn2_error=message)
                progress(f"  {request['lead_days']}-day forecast unavailable: {message}")
            status_rows.append({**status,'field':spec.key,'field_title':spec.title,'pressure_hpa':spec.level,'units':spec.units})
            # Keep a lightweight progress table even if the kernel stops during the next lead.
            progress_path = Path(output_dir)/'progress'/signature/'batch_status.csv'
            progress_path.parent.mkdir(parents=True,exist_ok=True)
            temporary = progress_path.with_suffix('.tmp.csv')
            pd.DataFrame(status_rows).to_csv(temporary,index=False)
            temporary.replace(progress_path)
        common_sample = len(event_ranks) == len(LEAD_HOURS) * len(POLYGONS)
        for record in event_ranks:
            record["in_common_sample"] = common_sample
        rank_rows.extend(event_ranks)
    return {
        "case_ranks": pd.DataFrame(rank_rows),
        "member_values": pd.DataFrame(member_rows),
        "era5_reference": pd.DataFrame(reference_rows),
        "batch_status": pd.DataFrame(status_rows),
    }, identity


def rank_histogram_counts(ranks, *, grouped=True):
    """Count every rank, including empty bins, using common cases at each lead."""
    if 'field' in ranks and ranks['field'].nunique()>1:
        raise ValueError('Plot each verification field separately; do not pool different variables.')
    if ranks.empty or not ranks.in_common_sample.any():
        raise ValueError("No event has complete ERA5 and all three WN2 leads.")
    complete = ranks.loc[ranks.in_common_sample]
    n_events = complete.review_key.nunique()
    bin_width = 13 if grouped else 1
    records = []
    for region in POLYGONS:
        for lead in LEAD_HOURS:
            matched = complete.loc[(complete.polygon == region) & (complete.lead_hours == lead)]
            counts = np.bincount(matched.era5_rank.to_numpy(dtype=int), minlength=ENSEMBLE_SIZE + 2)[1:]
            counts = counts.reshape(-1, bin_width).sum(axis=1)
            records.extend({
                "polygon": region, "lead_days": lead // 24, "matched_events": n_events,
                "bin_number": index + 1,
                "rank_from": index * bin_width + 1, "rank_through": (index + 1) * bin_width,
                "case_count": int(count), "uniform_expected_count": n_events * bin_width / (ENSEMBLE_SIZE + 1),
            } for index, count in enumerate(counts))
    return pd.DataFrame(records)


def plot_rank_histograms(ranks, *, grouped=True, title_suffix='All events'):
    """Show either all 65 ranks or five groups, with the same cases at each lead."""
    import matplotlib.pyplot as plt

    bins = rank_histogram_counts(ranks, grouped=grouped)
    complete = ranks.loc[ranks.in_common_sample]
    n_events = complete.review_key.nunique()
    largest_bin = bins.case_count.max()
    expectation = bins.uniform_expected_count.iloc[0]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True, layout="constrained")
    for r, region in enumerate(POLYGONS):
        for c, lead in enumerate(LEAD_HOURS):
            ax = axes[r, c]
            matched = complete.loc[(complete.polygon == region) & (complete.lead_hours == lead)]
            panel_bins = bins.loc[(bins.polygon == region) & (bins.lead_days == lead // 24)]
            positions = ([f"Bin {index + 1}\nRanks {label}" for index, label in enumerate(RANK_GROUPS)]
                         if grouped else np.arange(1, ENSEMBLE_SIZE + 2))
            ax.bar(positions, panel_bins.case_count, color="#277486", width=.8)
            ax.axhline(expectation, color=".25", linestyle="--", linewidth=1)
            ax.set_title(f"{region} · {lead // 24}-day lead · N = {len(matched)}")
            if not grouped:
                ax.set_xticks([1, 9, 17, 25, 33, 41, 49, 57, 65])
            ax.tick_params(axis="x", labelsize=8)
            ax.set_xlabel("ERA5 rank position among 64 forecast values")
            ax.text(.03, .96, f"Outside ensemble range: {matched.below_member_min.sum()} below, "
                    f"{matched.above_member_max.sum()} above", transform=ax.transAxes, va="top", fontsize=8)
            ax.set_ylim(0, 1.22 * max(largest_bin, expectation))
            ax.yaxis.get_major_locator().set_params(integer=True)
            if c == 0:
                ax.set_ylabel("Number of events")
    field_title = str(ranks.field_title.iloc[0]) if 'field_title' in ranks else '925-hPa horizontal divergence'
    convergence = 'field' not in ranks or ranks['field'].eq('divergence').all()
    fig.suptitle(f"{title_suffix} · {field_title} polygon mean at the onset target · {n_events} matched "
                 f"{'event' if n_events == 1 else 'events'}\n"
                 f"{'13 ranks' if grouped else '1 rank'} per bar · dashed line = uniform rank reference\n"
                 + ("Lower rank = ERA5 more convergent relative to forecasts · Higher rank = ERA5 less convergent"
                    if convergence else 'Lower rank = ERA5 lower relative to forecasts · Higher rank = ERA5 higher'))
    return fig


def ordered_verification_values(members, ranks):
    """Pool 64 forecast values and ERA5, preserving the exact saved tie-broken rank."""
    keys = ["review_key", "polygon", "lead_hours"]
    columns = [*keys, "event_number", "valid_time_utc", "init_time_utc", "lead_days", "pressure_hpa",
               "in_common_sample", "position", "bin_number", "rank_range", "source", "member", DIVERGENCE_COLUMN,
               'value','field','field_title','units','vortex_label','vortex_scan_status','vortex_classification']
    if ranks.empty:
        return pd.DataFrame(columns=columns)
    if ranks.duplicated(keys).any():
        raise ValueError("Duplicate event/polygon/lead ranks cannot be exported.")
    groups = {key: group for key, group in members.groupby(keys, sort=False)}
    records = []
    for rank in ranks.to_dict("records"):
        key = tuple(rank[name] for name in keys)
        group = groups.get(key)
        if group is None or len(group) != ENSEMBLE_SIZE or group.member.astype(str).nunique() != ENSEMBLE_SIZE:
            raise ValueError(f"Ordered values require 64 distinct members for {key}.")
        value_column = 'value' if 'value' in group else DIVERGENCE_COLUMN
        values = group[value_column].to_numpy(dtype=float)
        observed = rank['era5_value'] if 'era5_value' in rank else rank['era5_divergence_1e5_s-1']
        if not np.isfinite(values).all() or not np.isfinite(observed):
            raise ValueError(f"Ordered values contain missing or nonfinite divergence for {key}.")
        position = int(rank["era5_rank"])
        lower = int((values < observed).sum()) + 1
        upper = lower + int((values == observed).sum())
        if not lower <= position <= upper:
            raise ValueError(f"Saved ERA5 rank disagrees with the member values for {key}.")
        ordered = group.sort_values(value_column, kind="stable")
        entries = [
            {"source": "WN2", "member": str(item["member"]), 'value':float(item[value_column])}
            for item in ordered.to_dict("records")
        ]
        entries.insert(position - 1, {"source": "ERA5", "member": "", 'value':float(observed)})
        info = {name: rank[name] for name in columns[:9]}
        info.update(field=rank.get('field','divergence'),field_title=rank.get('field_title','925-hPa horizontal divergence'),
                    units=rank.get('units','10⁻⁵ s⁻¹'),vortex_label=rank.get('vortex_label',''),
                    vortex_scan_status=rank.get('vortex_scan_status',''),
                    vortex_classification=rank.get('vortex_classification',vortex_classification(rank)))
        for index, entry in enumerate(entries):
            if info['field']=='divergence':entry[DIVERGENCE_COLUMN]=entry['value']
            bin_index = index // 13
            records.append({**info, "position": index + 1, "bin_number": bin_index + 1,
                            "rank_range": RANK_GROUPS[bin_index], **entry})
    return pd.DataFrame(records, columns=columns)


def write_ordered_bins(writer, ordered, ranks):
    """Show thirteen entries in each of five bin columns for every verifiable case."""
    sheet = writer.book.add_worksheet("Ordered bins")
    sheet.hide_gridlines(2)
    sheet.set_zoom(85)
    sheet.freeze_panes(4, 0)
    sheet.set_landscape()
    sheet.fit_to_pages(1, 0)
    book = writer.book
    title_format = book.add_format({"bold": True, "font_size": 13, "font_color": "#17324D"})
    note_format = book.add_format({"text_wrap": True, "valign": "top"})
    bin_format = book.add_format({"bold": True, "bg_color": "#DDECF1", "align": "center"})
    header_format = book.add_format({"bold": True, "text_wrap": True, "bottom": 1})
    value_format = book.add_format({"num_format": "0.000000"})
    era5_value_format = book.add_format({"num_format": "0.000000", "bold": True, "bg_color": "#DDF0DE"})
    era5_label_format = book.add_format({"bold": True, "bg_color": "#DDF0DE"})
    title = str(ranks.field_title.iloc[0]) if len(ranks) and 'field_title' in ranks else '925-hPa polygon-average divergence'
    units = str(ranks.units.iloc[0]) if len(ranks) and 'units' in ranks else '10⁻⁵ s⁻¹'
    convergence_note = (' Negative divergence is convergence.' if 'field' not in ranks
                        or (not ranks.empty and ranks.field.eq('divergence').all()) else '')
    sheet.merge_range(0, 0, 0, 14, 'Ordered '+title, title_format)
    sheet.merge_range(1, 0, 2, 14,
                      'Each block contains 64 WN2 values and one ERA5 value, sorted from lowest to highest. '
                      f'ERA5 is highlighted. Values are in {units}.{convergence_note} '
                      "Histogram counts use only ERA5's saved position.", note_format)
    for bin_index in range(5):
        start = bin_index * 3
        sheet.set_column(start, start, 8)
        sheet.set_column(start + 1, start + 1, 20)
        sheet.set_column(start + 2, start + 2, 16, value_format)
    if ordered.empty:
        sheet.write(4, 0, "No event/polygon/lead has both ERA5 and all 64 members.")
        return
    keys = ["review_key", "polygon", "lead_hours"]
    rank_lookup = {tuple(item[name] for name in keys): item for item in ranks.to_dict("records")}
    row = 4
    for key, group in ordered.groupby(keys, sort=False):
        case = rank_lookup[key]
        sheet.merge_range(row, 0, row, 14,
                          f"Event {case['event_number']} · {case['polygon']} · {case['lead_days']}-day lead · {case['pressure_hpa']} hPa",
                          title_format)
        sheet.merge_range(row + 1, 0, row + 1, 14,
                          f"Valid {case['valid_time_utc']} · Initialized {case['init_time_utc']} · "
                          f"ERA5 rank {case['era5_rank']} of 65 · "
                          f"{'Included in all-lead histograms' if case['in_common_sample'] else 'Incomplete all-lead comparison'}")
        for bin_index, label in enumerate(RANK_GROUPS):
            start = bin_index * 3
            sheet.merge_range(row + 2, start, row + 2, start + 2,
                              f"Bin {bin_index + 1} · ranks {label}", bin_format)
            for offset, text in enumerate(("Position", "Source", f'Value ({units})')):
                sheet.write(row + 3, start + offset, text, header_format)
        for item in group.to_dict("records"):
            cell_row = row + 4 + (item["position"] - 1) % 13
            cell_col = (item["bin_number"] - 1) * 3
            label_format = era5_label_format if item["source"] == "ERA5" else None
            number_format = era5_value_format if item["source"] == "ERA5" else value_format
            sheet.write_number(cell_row, cell_col, item["position"], label_format)
            source = "ERA5" if item["source"] == "ERA5" else f"WN2 sample {item['member']}"
            sheet.write_string(cell_row, cell_col + 1, source, label_format)
            sheet.write_number(cell_row, cell_col + 2, item['value'] if 'value' in item else item[DIVERGENCE_COLUMN], number_format)
        row += 19
    sheet.print_area(0, 0, row - 1, 14)


def export_verification(tables, identity, destination):
    """Write the results and a workbook for inspecting the computed ranks in Drive."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    tables = dict(tables)
    ranks = tables["case_ranks"]
    tables["ordered_values"] = ordered_verification_values(tables["member_values"], ranks)
    tables['crps_cases'] = continuous_ranked_scores(tables)
    tables['crps_summary'] = continuous_score_summary(tables['crps_cases'])
    tables['event_accounting'] = verification_event_accounting(tables)
    # A previously complete run must not leave count tables that describe an old sample.
    for name in ('rank_bin_counts', 'rank_group_counts', 'bin_counts'):
        (destination / f'{name}.csv').unlink(missing_ok=True)
    if not ranks.empty and ranks.in_common_sample.any():
        tables["rank_bin_counts"] = rank_histogram_counts(ranks, grouped=False)
        tables["rank_group_counts"] = rank_histogram_counts(ranks, grouped=True)
        tables["bin_counts"] = tables["rank_group_counts"].pivot(
            index=["polygon", "lead_days", "matched_events"], columns="bin_number", values="case_count"
        ).rename(columns={index + 1: f"Bin {index + 1} · ranks {label}"
                          for index, label in enumerate(RANK_GROUPS)}).reset_index()
    atomic_json_file(destination / "calculation_settings.json", identity)
    for name, frame in tables.items():
        frame.to_csv(destination / f"{name}.csv", index=False)
    with NamedTemporaryFile(suffix=".xlsx", dir=destination, delete=False) as temporary:
        temporary_path = Path(temporary.name)
    try:
        with pd.ExcelWriter(temporary_path, engine="xlsxwriter",
                            engine_kwargs={"options": {"strings_to_formulas": False, "strings_to_urls": False}}) as writer:
            era5_format = writer.book.add_format({"bold": True, "bg_color": "#DDF0DE"})
            for name, frame in tables.items():
                sheet = name.replace("_", " ").capitalize()
                frame.to_excel(writer, sheet_name=sheet, index=False)
                worksheet = writer.sheets[sheet]
                worksheet.freeze_panes(1, 0)
                worksheet.hide_gridlines(2)
                worksheet.set_column(0, max(0, len(frame.columns) - 1), 18)
                if len(frame.columns):
                    worksheet.autofilter(0, 0, len(frame), len(frame.columns) - 1)
                    for index, column in enumerate(frame.columns):
                        width = max(len(column) + 2, 14)
                        if not frame.empty:
                            width = max(width, int(frame[column].head(100).astype(str).str.len().max()) + 2)
                        worksheet.set_column(index, index, min(width, 55))
                if name == "ordered_values" and not frame.empty:
                    from xlsxwriter.utility import xl_col_to_name
                    source_col = xl_col_to_name(frame.columns.get_loc("source"))
                    worksheet.conditional_format(1, 0, len(frame), len(frame.columns) - 1,
                        {"type": "formula", "criteria": "=$" + source_col + '2="ERA5"', "format": era5_format})
                if name == "case_ranks":
                    write_ordered_bins(writer, tables["ordered_values"], ranks)
        temporary_path.replace(destination / "ensemble_verification.xlsx")
    finally:
        temporary_path.unlink(missing_ok=True)
    return destination


def empirical_ensemble_crps(members, reference):
    """CRPS of the equally weighted empirical CDF, in the original variable units.

    mean|X-y| - (1/(2*M**2))*sum_ij|Xi-Xj|. The sorted form uses O(M log M)
    work and O(M) memory. This scores the finite ensemble CDF shown in the proposal;
    it is not the independence-dependent 'fair' estimator of an underlying distribution.
    """
    values=np.asarray(members,dtype=np.float64)
    if values.ndim!=1 or not len(values) or not np.isfinite(values).all() or not np.isfinite(reference):
        raise ValueError('CRPS requires finite members and a finite verifying value.')
    count=len(values)
    first=float(np.mean(np.abs(values-reference)))
    centered=np.sort(values-values[0])
    second=float(np.dot(2*np.arange(1,count+1)-count-1,centered)/(count*count))
    score=first-second
    if score < -1e-12*max(1.,first,second):raise ArithmeticError('Negative empirical CRPS beyond rounding tolerance.')
    return max(0.,score)


def continuous_ranked_scores(tables, *, group='All'):
    """Score saved polygon means only, retaining the common all-lead event sample."""
    ranks=verification_subset(tables,group)
    columns=['event_number','review_key','valid_time_utc','init_time_utc','polygon','lead_hours','lead_days',
             'field','field_title','pressure_hpa','units','vortex_label','vortex_scan_status',
             'vortex_classification','reference_product','member_count','era5_value','crps',
             'ensemble_mean','ensemble_median','ensemble_min','ensemble_max','in_common_sample']
    if ranks.empty:return pd.DataFrame(columns=columns)
    keys=['review_key','polygon','lead_hours']
    if ranks.duplicated(keys).any():raise ValueError('Duplicate cases would overweight CRPS.')
    members=tables['member_values']
    groups={key:frame for key,frame in members.groupby(keys,sort=False)}
    rows=[]
    for row in ranks.loc[ranks.in_common_sample].to_dict('records'):
        frame=groups.get(tuple(row[key] for key in keys))
        if frame is None or len(frame)!=ENSEMBLE_SIZE or frame.member.astype(str).nunique()!=ENSEMBLE_SIZE:
            raise ValueError('CRPS requires all 64 distinct native ensemble members for every scored case.')
        for name in ('field','pressure_hpa','units'):
            if name in frame and name in row and not frame[name].eq(row[name]).all():
                raise ValueError(f'CRPS member/reference {name} metadata differs.')
        values=frame['value' if 'value' in frame else DIVERGENCE_COLUMN].to_numpy(dtype=float)
        observed=float(row['era5_value'] if 'era5_value' in row else row['era5_divergence_1e5_s-1'])
        info={name:row.get(name,'') for name in columns[:15]}
        info.update(field=row.get('field','divergence'),field_title=row.get('field_title','925-hPa divergence'),
                    pressure_hpa=row.get('pressure_hpa',925),units=row.get('units','10⁻⁵ s⁻¹'),
                    reference_product=row.get('reference_product','ERA5'))
        rows.append(dict(**info,member_count=ENSEMBLE_SIZE,era5_value=observed,
            crps=empirical_ensemble_crps(values,observed),ensemble_mean=float(values.mean()),
            ensemble_median=float(np.median(values)),ensemble_min=float(values.min()),
            ensemble_max=float(values.max()),in_common_sample=True))
    return pd.DataFrame(rows,columns=columns)


def continuous_score_summary(scores):
    columns=['field','pressure_hpa','units','polygon','lead_days','cases','mean_crps','median_crps',
             'q25_crps','q75_crps','min_crps','max_crps','era5t_cases','valid_time_start','valid_time_end']
    if scores.empty:return pd.DataFrame(columns=columns)
    rows=[]
    for key,frame in scores.groupby(columns[:5],sort=False):
        times=pd.to_datetime(frame.valid_time_utc,utc=True)
        rows.append(dict(zip(columns[:5],key),cases=len(frame),mean_crps=float(frame.crps.mean()),
            median_crps=float(frame.crps.median()),q25_crps=float(frame.crps.quantile(.25)),
            q75_crps=float(frame.crps.quantile(.75)),min_crps=float(frame.crps.min()),max_crps=float(frame.crps.max()),
            era5t_cases=int(frame.reference_product.eq('ERA5T').sum()),valid_time_start=iso_utc(times.min()),
            valid_time_end=iso_utc(times.max())))
    return pd.DataFrame(rows,columns=columns)


def verification_event_accounting(tables):
    """One row per selected event: membership, usable leads and full retrieval errors."""
    status=tables.get('batch_status',pd.DataFrame())
    ranks=tables['case_ranks']
    base=status if not status.empty else ranks
    columns=['event_number','review_key','valid_time_utc','field','pressure_hpa','vortex_classification',
             'used_in_common_sample','wn2_complete_leads','expected_leads','era5_status','wn2_errors','era5_errors']
    if base.empty:return pd.DataFrame(columns=columns)
    complete=set(ranks.loc[ranks.in_common_sample,'review_key']) if not ranks.empty else set()
    rows=[]
    for key,frame in base.groupby('review_key',sort=False):
        first=frame.iloc[0]
        def joined(name):return ' | '.join(dict.fromkeys(str(x) for x in frame.get(name,[]) if pd.notna(x) and str(x)))
        good=frame.loc[frame.wn2_status.isin(['cached','computed','complete']),'lead_days'].nunique() if 'wn2_status' in frame else frame.lead_days.nunique()
        rows.append({**{name:first.get(name,'') for name in columns[:6]},'review_key':key,
            'used_in_common_sample':key in complete,'wn2_complete_leads':good,'expected_leads':len(LEAD_HOURS),
            'era5_status':joined('era5_status'),'wn2_errors':joined('wn2_error'),'era5_errors':joined('era5_error')})
    return pd.DataFrame(rows,columns=columns)


def plot_crps_summary(scores, *, title_suffix='All'):
    """Lead-time comparison and statistics from stored scalar values, without field reads."""
    import matplotlib.pyplot as plt
    if scores.empty:raise ValueError('No complete cases for CRPS.')
    if scores[['field','pressure_hpa','units']].drop_duplicates().shape[0]!=1:
        raise ValueError('Plot each CRPS variable and level separately.')
    summary=continuous_score_summary(scores);first=scores.iloc[0]
    fig=plt.figure(figsize=(13,8.5))
    grid=fig.add_gridspec(2,2,height_ratios=[3.2,1.7],left=.08,right=.97,top=.79,bottom=.10,hspace=.27,wspace=.28)
    for column,region in enumerate(POLYGONS):
        ax=fig.add_subplot(grid[0,column]);table_ax=fig.add_subplot(grid[1,column])
        region_summary=summary.loc[summary.polygon.eq(region)].set_index('lead_days').reindex([1,5,10])
        samples=[scores.loc[scores.polygon.eq(region)&scores.lead_days.eq(lead),'crps'].to_numpy() for lead in (1,5,10)]
        ax.boxplot(samples,positions=[1,5,10],widths=.9,patch_artist=True,
            boxprops={'facecolor':'#dcebea','edgecolor':'#277f84'},medianprops={'color':'#b06c19','linewidth':1.6},
            flierprops={'marker':'.','markerfacecolor':'#6e8087','markeredgecolor':'#6e8087','markersize':4})
        ax.plot([1,5,10],region_summary.mean_crps,'o-',color='#087d78',linewidth=2,label='Mean CRPS')
        ax.set(xticks=[1,5,10],xticklabels=['1 day','5 days','10 days'],xlim=(0,11),ylim=(0,None),
               xlabel='Forecast lead',ylabel=f'CRPS ({first.units})',title=region)
        ax.grid(axis='y',alpha=.22);ax.set_axisbelow(True);ax.legend(frameon=False,loc='upper left')
        table_ax.axis('off')
        rows=[['Usable events',*[str(int(x)) for x in region_summary.cases]],
              ['Mean CRPS',*[f'{x:.3f}' for x in region_summary.mean_crps]],
              ['Median CRPS',*[f'{x:.3f}' for x in region_summary.median_crps]],
              ['25th–75th percentile',*[f'{a:.3f}–{b:.3f}' for a,b in zip(region_summary.q25_crps,region_summary.q75_crps)]],
              ['Preliminary ERA5T cases',*[str(int(x)) for x in region_summary.era5t_cases]]]
        table=table_ax.table(cellText=rows,colLabels=['Statistic','1 day','5 days','10 days'],
            colWidths=[.43,.19,.19,.19],cellLoc='left',bbox=[0,0,1,1]);table.auto_set_font_size(False);table.set_fontsize(10)
        for (r,c),cell in table.get_celld().items():
            cell.set_edgecolor('#d7e0e7');cell.set_linewidth(.6)
            if r==0:cell.set_facecolor('#e6eef3');cell.set_text_props(weight='bold')
            elif r%2==0:cell.set_facecolor('#f5f8fa')
    fig.suptitle(f'Continuous ranked probability score · {title_suffix}\n{first.field_title}\n'
                 'All 64 members · same complete events at every lead · lower is better',fontsize=14,y=.975,linespacing=1.5)
    times=pd.to_datetime(scores.valid_time_utc,utc=True)
    fig.text(.08,.025,f'Verifying dates: {times.min():%Y-%m-%d} to {times.max():%Y-%m-%d}. '
             'Boxes: middle 50% of case scores; orange line: median; green line: mean.\n'
             'CRPS scores the distribution of polygon means. CRPSS awaits an independent climatological reference.',fontsize=9)
    return fig


def export_continuous_scores(tables, destination, *, group='All'):
    import matplotlib.pyplot as plt
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    scores=continuous_ranked_scores(tables,group=group);summary=continuous_score_summary(scores)
    scores.to_csv(destination/'crps_cases.csv',index=False);summary.to_csv(destination/'crps_summary.csv',index=False)
    if scores.empty:
        for suffix in ('png','pdf'):(destination/f'crps_summary.{suffix}').unlink(missing_ok=True)
    else:
        fig=plot_crps_summary(scores,title_suffix=group)
        for suffix in ('png','pdf'):fig.savefig(destination/f'crps_summary.{suffix}',dpi=180)
        plt.close(fig)
    atomic_json_file(destination/'crps_settings.json',dict(group=group,ensemble_members=64,
        estimator='empirical equally weighted ensemble CDF',units='same units as each selected field',
        formula='mean(abs(X-y)) - sum_ij(abs(X_i-X_j))/(2*M*M)',
        sample='common all-lead complete events; instantaneous polygon means',
        crpss_status='not calculated; independent reference not supplied',
        source='https://scoringrules.readthedocs.io/en/latest/crps_estimators.html'))
    return scores,summary


def verification_subset(tables, group='All'):
    """Use completed vortex screens for strata; retain unchecked cases as Unknown."""
    ranks = tables['case_ranks'].copy()
    if ranks.empty:
        return ranks
    classification = ranks.get('vortex_classification', pd.Series('', index=ranks.index))
    checked = ranks.vortex_scan_status.eq('complete')
    groups = np.select([checked & classification.eq('Vortex'), checked & classification.eq('Non-vortex')],
                       ['Vortex', 'Non-vortex'], default='Unknown')
    ranks['verification_group'] = groups
    if group not in ('All', 'Vortex', 'Non-vortex', 'Unknown'):
        raise ValueError('Choose All, Vortex, Non-vortex, or Unknown.')
    return ranks if group == 'All' else ranks.loc[ranks.verification_group.eq(group)].copy()


def scheduled_verification_catalog(start, end, *, interval_hours=24, months=(11, 12, 1, 2)):
    """Outcome-independent, half-open UTC schedule including crossings and non-crossings."""
    start, end = utc_naive(start), utc_naive(end)
    if start >= end or interval_hours not in (6, 12, 24):
        raise ValueError('Use start < exclusive end and a 6, 12, or 24 hour schedule.')
    if start != start.floor('6h') or end != end.floor('6h'):
        raise ValueError('Schedule endpoints must be exact six-hour UTC times.')
    times = pd.date_range(start, end, freq=f'{interval_hours}h', inclusive='left')
    times = times[times.month.isin(months)]
    if times.empty:
        raise ValueError('The schedule contains no winter verification targets.')
    frame = pd.DataFrame({'event_start': times, 'event_end': times, 'event_peak': times,
                          'event_number': np.arange(1, len(times) + 1), 'target_valid_time': times,
                          'onset_offset_hours': 0., 'event_peak_D_1e5_s-1': np.nan,
                          'vortex_plot_label': '', 'vortex_scan_status': 'not_applicable'})
    frame['review_key'] = [f'scheduled|{iso_utc(t)}' for t in times]
    return frame


def reliability_probabilities(members, ranks, thresholds, *, sample='event_conditioned', direction='less'):
    """A selected-field threshold probability from all 64 members and binary ERA5 outcome.

    This is instantaneous polygon convergence, not the hourly trailing-D12 JPCZ detector.
    Only the common all-lead sample enters, exactly as in the rank histograms.
    """
    if sample not in ('event_conditioned', 'scheduled_winter'):
        raise ValueError('Unknown reliability sampling design.')
    if set(thresholds) != set(POLYGONS) or not np.isfinite(list(thresholds.values())).all():
        raise ValueError('Supply one finite snapshot threshold in the selected field units per polygon.')
    if direction not in ('less','greater'):raise ValueError('Choose less or greater for the probability threshold.')
    keys = ['review_key', 'polygon', 'lead_hours']
    columns = [*keys, 'lead_days', 'valid_time_utc', 'event_number', 'vortex_label', 'vortex_scan_status',
               'sample', 'threshold_1e5_s-1', 'member_count', 'forecast_probability', 'observed', 'brier_score',
               'threshold','direction','field','field_title','units','pressure_hpa']
    if ranks.empty:
        return pd.DataFrame(columns=columns)
    if ranks.duplicated(keys).any():
        raise ValueError('Duplicate verification cases would overweight reliability.')
    groups = {key: frame for key, frame in members.groupby(keys, sort=False)}
    rows = []
    for row in ranks.loc[ranks.in_common_sample].to_dict('records'):
        key = tuple(row[name] for name in keys)
        frame = groups.get(key)
        if frame is None or len(frame) != ENSEMBLE_SIZE or frame.member.astype(str).nunique() != ENSEMBLE_SIZE:
            raise ValueError(f'Reliability requires 64 distinct members for {key}.')
        values = frame['value' if 'value' in frame else DIVERGENCE_COLUMN].to_numpy(dtype=float)
        observed_value = float(row['era5_value'] if 'era5_value' in row else row['era5_divergence_1e5_s-1'])
        if not np.isfinite(values).all() or not np.isfinite(observed_value):
            raise ValueError('Reliability inputs must be finite.')
        threshold = float(thresholds[row['polygon']])
        probability = float(np.mean(values < threshold if direction=='less' else values > threshold))
        observed = int(observed_value < threshold if direction=='less' else observed_value > threshold)
        rows.append({**{name: row.get(name, '') for name in columns[:8]},
                     'sample': sample, 'threshold_1e5_s-1': threshold if row.get('field','divergence')=='divergence' else np.nan,
                     'member_count': ENSEMBLE_SIZE,
                     'forecast_probability': probability, 'observed': observed,
                     'brier_score': (probability - observed)**2, 'threshold':threshold,'direction':direction,
                     'field':row.get('field','divergence'),'field_title':row.get('field_title','925-hPa divergence'),
                     'units':row.get('units','10⁻⁵ s⁻¹'),'pressure_hpa':row.get('pressure_hpa',925)})
    return pd.DataFrame(rows, columns=columns)


def reliability_bin_counts(probabilities, *, n_bins=10):
    """Fixed [left,right) bins, final bin closed at 1, with empty bins retained."""
    if not isinstance(n_bins, int) or n_bins < 2:
        raise ValueError('Use at least two probability bins.')
    rows = []
    for region in POLYGONS:
        for lead in LEAD_HOURS:
            subset = probabilities.loc[probabilities.polygon.eq(region) & probabilities.lead_hours.eq(lead)]
            if subset.empty:
                continue
            p = subset.forecast_probability.to_numpy(dtype=float)
            y = subset.observed.to_numpy(dtype=float)
            if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any() or not np.isin(y, (0, 1)).all():
                raise ValueError('Need probabilities in [0,1] and binary observed outcomes.')
            bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
            for index in range(n_bins):
                mask = bins == index
                n, hits = int(mask.sum()), int(y[mask].sum())
                frequency = hits / n if n else np.nan
                # Pointwise Wilson 95% binomial intervals; serial dependence is not modeled.
                z = 1.959963984540054
                denominator = 1 + z*z/n if n else np.nan
                center = (frequency + z*z/(2*n)) / denominator if n else np.nan
                half = z * np.sqrt(frequency*(1-frequency)/n + z*z/(4*n*n)) / denominator if n else np.nan
                rows.append({'polygon': region, 'lead_days': lead // 24, 'sample': subset['sample'].iloc[0],
                             'threshold_1e5_s-1': subset['threshold_1e5_s-1'].iloc[0],
                             'threshold':subset['threshold'].iloc[0] if 'threshold' in subset else subset['threshold_1e5_s-1'].iloc[0],
                             'direction':subset['direction'].iloc[0] if 'direction' in subset else 'less',
                             'bin_number': index + 1, 'probability_from': index/n_bins,
                             'probability_through': (index+1)/n_bins, 'case_count': n, 'observed_count': hits,
                             'mean_forecast_probability': p[mask].mean() if n else np.nan,
                             'observed_frequency': frequency, 'wilson_lower': max(0., center-half) if n else np.nan,
                             'wilson_upper': min(1., center+half) if n else np.nan,
                             'panel_cases': len(subset), 'base_rate': float(y.mean()),
                             'brier_score': float(subset.brier_score.mean())})
    return pd.DataFrame(rows, columns=[
        'polygon', 'lead_days', 'sample', 'threshold_1e5_s-1', 'bin_number',
        'probability_from', 'probability_through', 'case_count', 'observed_count',
        'mean_forecast_probability', 'observed_frequency', 'wilson_lower', 'wilson_upper',
        'panel_cases', 'base_rate', 'brier_score','threshold','direction'])


def reliability_panel_statistics(probabilities, *, n_bins=10):
    """Numbers printed below the diagrams, with the same case/bin accounting as the curve."""
    columns=['polygon','lead_days','cases','observed_crossings','observed_non_crossings',
             'base_rate','mean_forecast_probability','brier_score','populated_bins',
             'probability_bins','valid_time_start','valid_time_end','field','pressure_hpa',
             'units','threshold','direction','ensemble_members']
    if probabilities.empty:return pd.DataFrame(columns=columns)
    counts=reliability_bin_counts(probabilities,n_bins=n_bins)
    rows=[]
    for (region,lead),frame in probabilities.groupby(['polygon','lead_days'],sort=False):
        bins=counts.loc[counts.polygon.eq(region)&counts.lead_days.eq(lead)]
        times=pd.to_datetime(frame.valid_time_utc,utc=True)
        hits=int(frame.observed.sum())
        rows.append(dict(polygon=region,lead_days=int(lead),cases=len(frame),observed_crossings=hits,
            observed_non_crossings=len(frame)-hits,base_rate=float(frame.observed.mean()),
            mean_forecast_probability=float(frame.forecast_probability.mean()),
            brier_score=float(frame.brier_score.mean()),populated_bins=int(bins.case_count.gt(0).sum()),
            probability_bins=n_bins,valid_time_start=iso_utc(times.min()),valid_time_end=iso_utc(times.max()),
            field=frame.field.iloc[0],pressure_hpa=int(frame.pressure_hpa.iloc[0]),units=frame.units.iloc[0],
            threshold=float(frame.threshold.iloc[0]),direction=frame.direction.iloc[0],
            ensemble_members=int(frame.member_count.iloc[0])))
    return pd.DataFrame(rows,columns=columns)


def _report_statistics_table(ax, rows, *, widths=(.66,.34)):
    ax.axis('off')
    table=ax.table(cellText=rows,colLabels=['Statistic','Value'],colWidths=list(widths),
                   cellLoc='left',bbox=[0,0,1,1])
    table.auto_set_font_size(False);table.set_fontsize(10)
    for (r,c),cell in table.get_celld().items():
        cell.set_edgecolor('#d7e0e7');cell.set_linewidth(.6);cell.PAD=.08
        if r==0:cell.set_facecolor('#e6eef3');cell.set_text_props(weight='bold',color='#17324d')
        elif r%2==0:cell.set_facecolor('#f5f8fa')
    return table


def plot_reliability_diagrams(probabilities, *, n_bins=10, title_suffix='All', region=None,
                              show_intervals=False, lead_hours=None):
    """Readable percent axes and a separate statistics table beneath every panel."""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.ticker import PercentFormatter
    if probabilities.empty:raise ValueError('No complete cases for reliability.')
    regions=list(POLYGONS) if region is None else [region]
    if any(item not in POLYGONS for item in regions):raise ValueError('Unknown verification polygon.')
    leads=list(LEAD_HOURS) if lead_hours is None else [lead_hours]
    if any(lead not in LEAD_HOURS for lead in leads):raise ValueError('Choose an exact configured forecast lead.')
    single=len(leads)==1
    counts=reliability_bin_counts(probabilities,n_bins=n_bins)
    statistics=reliability_panel_statistics(probabilities,n_bins=n_bins)
    fig=plt.figure(figsize=(8 if single else 16,(10.5 if single else 7.7)*len(regions)))
    grid=fig.add_gridspec(2*len(regions),len(leads),height_ratios=[3.5,1.65]*len(regions),
                          left=.07,right=.975,bottom=.09 if len(regions)==1 else .055,
                          top=(.72 if single else .80) if len(regions)==1 else .89,wspace=.29,hspace=.32)
    for r,polygon in enumerate(regions):
        for c,lead in enumerate(leads):
            ax=fig.add_subplot(grid[2*r,c]);info_ax=fig.add_subplot(grid[2*r+1,c])
            bins=counts.loc[counts.polygon.eq(polygon)&counts.lead_days.eq(lead//24)]
            ax.plot([0,100],[0,100],'--',color='#46515a',linewidth=1.4,zorder=1)
            ax.set(xlim=(-2,102),ylim=(-2,102),xlabel='Forecast probability',ylabel='Observed frequency (ERA5)',
                   title=f'{polygon} · {lead//24}-day lead')
            ax.set_aspect('equal',adjustable='box');ax.set_xticks(range(0,101,20));ax.set_yticks(range(0,101,20))
            ax.xaxis.set_major_formatter(PercentFormatter(100,decimals=0))
            ax.yaxis.set_major_formatter(PercentFormatter(100,decimals=0))
            ax.tick_params(labelsize=10);ax.title.set_fontsize(12);ax.grid(color='#dfe5e9',linewidth=.6)
            if bins.empty:
                ax.text(.5,.5,'No complete cases',ha='center',transform=ax.transAxes)
                _report_statistics_table(info_ax,[['Usable cases','0']]);continue
            # NaNs leave gaps at empty probability bins; no smoothing or invented frequencies.
            ax.plot(100*bins.mean_forecast_probability,100*bins.observed_frequency,
                    marker='o',markersize=6,color='#087d78',linewidth=2,zorder=3)
            ax.axhline(100*bins.base_rate.iloc[0],color='#b06c19',linestyle=':',linewidth=1.4,zorder=2)
            if show_intervals:
                filled=bins.loc[bins.case_count.gt(0)]
                y=100*filled.observed_frequency
                ax.errorbar(100*filled.mean_forecast_probability,y,
                    yerr=np.maximum(0,np.array([y-100*filled.wilson_lower,100*filled.wilson_upper-y])),
                    fmt='none',ecolor='#087d78',alpha=.4,capsize=3,zorder=2)
            stat=statistics.loc[statistics.polygon.eq(polygon)&statistics.lead_days.eq(lead//24)].iloc[0]
            _report_statistics_table(info_ax,[['Usable events',str(int(stat.cases))],
                ['ERA5 threshold crossings',f'{int(stat.observed_crossings)} / {int(stat.cases)}'],
                ['Observed frequency',f'{stat.base_rate:.1%}'],
                ['Mean forecast probability',f'{stat.mean_forecast_probability:.1%}'],
                ['Brier score (lower is better)',f'{stat.brier_score:.3f}'],
                ['Populated probability bins',f'{int(stat.populated_bins)} / {n_bins}']])
    first=probabilities.iloc[0];operator='<' if first.direction=='less' else '>'
    design='ERA5-event-conditioned sample' if first['sample']=='event_conditioned' else 'Scheduled winter sample'
    threshold_label=(f'{first.threshold:g} × {first.units}' if first.field=='divergence'
                     else f'{first.threshold:g} {first.units}')
    criterion=f'Criterion: polygon mean {operator} {threshold_label}'
    header=(f'{criterion}\n64 members · {design}' if single else f'{criterion} · 64 members · {design}')
    fig.suptitle(f'Reliability · {title_suffix}\n{first.field_title}\n{header}',
                 fontsize=14,y=.985,linespacing=1.5)
    handles=[Line2D([],[],color='#087d78',marker='o',linewidth=2,label='Observed reliability'),
             Line2D([],[],color='#46515a',linestyle='--',label='Perfect reliability'),
             Line2D([],[],color='#b06c19',linestyle=':',label='Observed frequency in this sample')]
    fig.legend(handles=handles,loc='center',bbox_to_anchor=(.5,(.795 if single else .865) if len(regions)==1 else .935),
               ncol=1 if single else 3,frameon=False,fontsize=10)
    times=pd.to_datetime(probabilities.valid_time_utc,utc=True)
    caption=(f'Verifying dates: {times.min():%Y-%m-%d} to {times.max():%Y-%m-%d}. '
             + ('\n' if single else '')
             + f'Each event contributes once per panel. Bins: {100/n_bins:g} percentage points.\n'
             + 'Above the diagonal: probabilities too low; below: probabilities too high. '
             + ('\n' if single else '')
             + 'Counts and pointwise Wilson 95% intervals are saved separately.')
    fig.text(.07,.02,caption,fontsize=9,color='#344858')
    return fig


def plot_forecast_probability_counts(probabilities, *, n_bins=10, title_suffix='All'):
    """Sharpness/counts companion figure, with no inset inside reliability axes."""
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter,MaxNLocator
    if probabilities.empty:raise ValueError('No complete cases for forecast counts.')
    counts=reliability_bin_counts(probabilities,n_bins=n_bins)
    fig,axes=plt.subplots(2,3,figsize=(15,8.5),layout='constrained')
    for r,region in enumerate(POLYGONS):
        for c,lead in enumerate(LEAD_HOURS):
            ax=axes[r,c];bins=counts.loc[counts.polygon.eq(region)&counts.lead_days.eq(lead//24)]
            ax.bar(50*(bins.probability_from+bins.probability_through),bins.case_count,
                   width=90/n_bins,color='#277f84',edgecolor='white')
            ax.set(xlim=(0,100),ylim=(0,None),xlabel='Forecast probability bin',ylabel='Number of events',
                   title=f'{region} · {lead//24}-day lead · N={int(bins.case_count.sum())}')
            ax.xaxis.set_major_formatter(PercentFormatter(100,decimals=0));ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
    fig.suptitle(f'Forecast probability counts · {title_suffix}\n'
                 'One forecast probability per event, polygon and lead. Empty bins contain no events.',fontsize=14)
    return fig


def export_reliability(tables, destination, thresholds, *, sample='event_conditioned', group='All', n_bins=10,
                       direction='less', show_intervals=False):
    """Refresh probability tables and plots from the current sample, never append old aggregates."""
    import matplotlib.pyplot as plt

    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    ranks = verification_subset(tables, group)
    probabilities = reliability_probabilities(tables['member_values'], ranks, thresholds, sample=sample,direction=direction)
    bins = reliability_bin_counts(probabilities, n_bins=n_bins)
    probabilities.to_csv(destination / 'reliability_probabilities.csv', index=False)
    bins.to_csv(destination / 'reliability_bins.csv', index=False)
    if not probabilities.empty:
        summary = reliability_panel_statistics(probabilities,n_bins=n_bins)
        summary.to_csv(destination / 'reliability_summary.csv', index=False)
        figures=[('reliability_diagrams',None,None),('reliability_jpcz_polygon','JPCZ polygon',None),
                 ('reliability_coastal_wedge','Coastal wedge',None)]
        figures.extend((f'reliability_{tag}_lead{lead//24:02d}d',polygon,lead)
                       for tag,polygon in (('jpcz_polygon','JPCZ polygon'),('coastal_wedge','Coastal wedge'))
                       for lead in LEAD_HOURS)
        for stem,region,lead in figures:
            fig=plot_reliability_diagrams(probabilities,n_bins=n_bins,title_suffix=group,region=region,
                                          show_intervals=show_intervals,lead_hours=lead)
            for extension in ('png','pdf'):fig.savefig(destination/f'{stem}.{extension}',dpi=180)
            plt.close(fig)
        fig=plot_forecast_probability_counts(probabilities,n_bins=n_bins,title_suffix=group)
        for extension in ('png','pdf'):fig.savefig(destination/f'forecast_probability_counts.{extension}',dpi=180)
        plt.close(fig)
    else:
        # Remove plots from a previous successful run if no current cases remain.
        for name in ('reliability_summary.csv',*[f'{stem}.{ext}' for stem in
            ('reliability_diagrams','reliability_jpcz_polygon','reliability_coastal_wedge','forecast_probability_counts',
             *[f'reliability_{tag}_lead{lead//24:02d}d' for tag in ('jpcz_polygon','coastal_wedge') for lead in LEAD_HOURS])
            for ext in ('png','pdf')]):
            (destination / name).unlink(missing_ok=True)
    atomic_json_file(destination / 'reliability_settings.json', {
        'sample': sample, 'group': group, 'thresholds':thresholds,'direction':direction, 'n_bins': n_bins,
        'criterion': f'instantaneous polygon mean strictly {direction} than threshold',
        'common_sample': 'ERA5 + 64 WN2 members at every lead and both polygons',
        'not_catalog_D12_event_probability': True, 'plot_intervals':show_intervals,
        'intervals':'pointwise Wilson 95%; serial dependence not modeled',
    })
    return probabilities, bins


def display_saved_verification_image(path, *, max_width=960):
    """Fit a saved PNG to the actual notebook width instead of creating horizontal overflow."""
    import base64
    from IPython.display import HTML,display
    encoded=base64.b64encode(Path(path).read_bytes()).decode('ascii')
    display(HTML(f'<img src="data:image/png;base64,{encoded}" alt="Saved verification figure" '
                 f'style="width:100%;max-width:{int(max_width)}px;height:auto;display:block;margin:0 auto;"/>'))


"""CORP/PAV calibration diagnostics from saved binary probabilities, without archive access.

Dimitriadis, Gneiting & Jordan (2021), doi:10.1073/pnas.2016191118.
The fit is descriptive and in-sample. It does not replace the original forecasts.
"""
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd



CORP_REFERENCE = 'https://www.pnas.org/doi/10.1073/pnas.2016191118'
CORP_POINT_COLUMNS = ['forecast_probability','case_count','observed_count','empirical_frequency',
    'corp_probability','pool_id','pool_count','pool_observed_count','pool_min_probability','pool_max_probability']


def pav_binary_probabilities(probabilities, outcomes):
    """Weighted PAV on tied forecast values; return unique-x points and fitted case values.

    Tied probabilities are aggregated before pooling, so ordering within ties cannot
    change the result. Integer cross-products determine pool merges without tolerances.
    Adjacent equal levels are merged into maximal constant pools for reproducible counts.
    """
    p=np.asarray(probabilities,dtype=float);y=np.asarray(outcomes,dtype=float)
    if p.ndim!=1 or y.shape!=p.shape or not len(p):raise ValueError('Supply paired, nonempty probability/outcome vectors.')
    if not np.isfinite(p).all() or ((p<0)|(p>1)).any() or not np.isin(y,[0,1]).all():
        raise ValueError('PAV needs finite probabilities in [0,1] and binary outcomes.')
    unique,inverse,counts=np.unique(p,return_inverse=True,return_counts=True)
    hits=np.bincount(inverse,weights=y,minlength=len(unique)).astype(int)
    pools=[]
    for index,(count,hit) in enumerate(zip(counts,hits)):
        pools.append([index,index+1,int(count),int(hit)])
        while len(pools)>1 and pools[-2][3]*pools[-1][2]>=pools[-1][3]*pools[-2][2]:
            right=pools.pop();left=pools.pop()
            pools.append([left[0],right[1],left[2]+right[2],left[3]+right[3]])
    fitted=np.empty(len(unique));rows=[]
    for number,(start,stop,count,hit) in enumerate(pools,1):
        fitted[start:stop]=hit/count
        for index in range(start,stop):
            rows.append(dict(forecast_probability=float(unique[index]),case_count=int(counts[index]),
                observed_count=int(hits[index]),empirical_frequency=float(hits[index]/counts[index]),
                corp_probability=hit/count,pool_id=number,pool_count=count,pool_observed_count=hit,
                pool_min_probability=float(unique[start]),pool_max_probability=float(unique[stop-1])))
    return pd.DataFrame(rows,columns=CORP_POINT_COLUMNS),fitted[inverse]


def corp_calibration_tables(probabilities, *, consistency_bands=False, assume_exchangeable=False,
                            n_resamples=2000, seed=37):
    """Fit each polygon/lead separately and compute the exact CORP Brier decomposition.

    Optional bands simulate Y*~Bernoulli(original p) at fixed forecast values, refit
    PAV, and take 5/95 percentiles. These are pointwise 90% CONSISTENCY bands under
    calibration and exchangeable/independent cases, not confidence or simultaneous bands.
    No uncertainty band is made by default for the small clustered winter event sample.
    """
    metadata=['polygon','lead_days','field','field_title','pressure_hpa','units','threshold','direction','sample']
    summary_columns=metadata+['cases','distinct_probabilities','pav_pools','observed_crossings','base_rate',
        'brier_score','pav_brier_score','miscalibration_mcb','discrimination_dsc','uncertainty_unc',
        'decomposition_residual','valid_time_start','valid_time_end','ensemble_members','plot_type',
        'band_type','n_resamples','seed']
    if consistency_bands and not assume_exchangeable:
        raise ValueError('Consistency bands require an explicit independent/exchangeable-case assumption.')
    if consistency_bands and (not isinstance(n_resamples,int) or n_resamples<100):
        raise ValueError('Use at least 100 consistency resamples.')
    if probabilities.empty:
        return (pd.DataFrame(columns=metadata+CORP_POINT_COLUMNS+['consistency_lower','consistency_upper']),
                probabilities.assign(corp_probability=pd.Series(dtype=float)),pd.DataFrame(columns=summary_columns))
    keys=['review_key','polygon','lead_hours']
    if probabilities.duplicated(keys).any():raise ValueError('Duplicate cases would overweight CORP.')
    rows=[];cases=[];summaries=[]
    for (region,lead),frame in probabilities.groupby(['polygon','lead_days'],sort=False):
        frame=frame.copy()
        for name in ('field','pressure_hpa','units','threshold','direction','sample','member_count'):
            if name in frame and frame[name].nunique(dropna=False)!=1:
                raise ValueError(f'Plot separate CORP diagnostics for different {name} settings.')
        first=frame.iloc[0]
        info={name:first[name] for name in metadata}
        p=frame.forecast_probability.to_numpy(dtype=float);y=frame.observed.to_numpy(dtype=float)
        points,q=pav_binary_probabilities(p,y)
        raw=float(np.mean((p-y)**2));cal=float(np.mean((q-y)**2));base=float(y.mean())
        unc=base*(1-base);mcb=raw-cal;dsc=unc-cal
        if min(mcb,dsc)<-1e-12:raise ArithmeticError('CORP score decomposition is not nonnegative.')
        mcb=max(0.,mcb);dsc=max(0.,dsc)
        points['consistency_lower']=np.nan;points['consistency_upper']=np.nan
        if consistency_bands:
            identity=json.dumps(info,sort_keys=True,default=lambda value:value.item() if isinstance(value,np.generic) else str(value))
            panel_seed=(int(seed)+int(hashlib.sha256(identity.encode()).hexdigest()[:8],16))%(2**32)
            rng=np.random.default_rng(panel_seed)
            fits=np.empty((n_resamples,len(points)))
            for index in range(n_resamples):
                simulated,_=pav_binary_probabilities(p,rng.binomial(1,p))
                fits[index]=simulated.corp_probability
            points['consistency_lower'],points['consistency_upper']=np.quantile(fits,[.05,.95],axis=0)
        rows.append(points.assign(**info))
        frame['corp_probability']=q
        frame['original_brier_score']=(p-y)**2;frame['pav_in_sample_brier_score']=(q-y)**2
        cases.append(frame)
        times=pd.to_datetime(frame.valid_time_utc,utc=True)
        differences=np.diff(points.forecast_probability)
        plot_type='discrete' if len(differences)==0 or float(differences.min())>=.01-1e-12 else 'continuous'
        summaries.append(dict(**info,cases=len(frame),distinct_probabilities=len(points),pav_pools=points.pool_id.nunique(),
            observed_crossings=int(y.sum()),base_rate=base,brier_score=raw,pav_brier_score=cal,
            miscalibration_mcb=mcb,discrimination_dsc=dsc,uncertainty_unc=unc,
            decomposition_residual=raw-(mcb-dsc+unc),valid_time_start=str(times.min()),valid_time_end=str(times.max()),
            ensemble_members=int(first.member_count),plot_type=plot_type,
            band_type='90% pointwise consistency; independent/exchangeable cases assumed' if consistency_bands else 'none',
            n_resamples=n_resamples if consistency_bands else 0,seed=int(seed)))
    return pd.concat(rows,ignore_index=True),pd.concat(cases,ignore_index=True),pd.DataFrame(summaries,columns=summary_columns)


def plot_corp_reliability(points, summary, *, region, lead_hours, title_suffix='All'):
    """A publication-style discrete CORP graph, with a separate statistics panel underneath."""
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter
    selected=points.loc[points.polygon.eq(region)&points.lead_days.eq(lead_hours//24)].sort_values('forecast_probability')
    stats=summary.loc[summary.polygon.eq(region)&summary.lead_days.eq(lead_hours//24)]
    if selected.empty or len(stats)!=1:raise ValueError('No unique CORP panel for the selected polygon/lead.')
    stat=stats.iloc[0]
    fig=plt.figure(figsize=(7.2,9.3),facecolor='white')
    ax=fig.add_axes([.14,.385,.78,.47]);table_ax=fig.add_axes([.09,.075,.85,.235])
    ax.plot([0,100],[0,100],'--',color='#323a42',linewidth=1.1,label='Perfect reliability',zorder=1)
    if selected.consistency_lower.notna().any():
        ax.fill_between(100*selected.forecast_probability,100*selected.consistency_lower,100*selected.consistency_upper,
            color='#bec9d1',alpha=.5,linewidth=0,label='90% consistency band',zorder=2)
    ax.plot(100*selected.forecast_probability,100*selected.corp_probability,'o-',color='#087f8c',
            linewidth=2,markersize=4.5,label='CORP / PAV estimate',zorder=3)
    ax.set(xlim=(-2,102),ylim=(-2,102),xlabel='Original WN2 forecast probability',
           ylabel='Estimated ERA5 crossing frequency',xticks=np.arange(0,101,20),yticks=np.arange(0,101,20))
    ax.xaxis.set_major_formatter(PercentFormatter(100,decimals=0));ax.yaxis.set_major_formatter(PercentFormatter(100,decimals=0))
    ax.set_aspect('equal',adjustable='box');ax.grid(color='#e0e6eb',linewidth=.6);ax.set_axisbelow(True)
    ax.spines[['top','right']].set_visible(False);ax.tick_params(labelsize=11)
    handles,labels=ax.get_legend_handles_labels()
    fig.legend(handles,labels,loc='center',bbox_to_anchor=(.53,.885),ncol=2,
               fontsize=10,frameon=False)
    operator='<' if stat.direction=='less' else '>'
    criterion=f'{stat.threshold:g} × {stat.units}' if stat.field=='divergence' else f'{stat.threshold:g} {stat.units}'
    fig.suptitle(f'CORP reliability · {title_suffix}',fontsize=16,y=.985)
    fig.text(.5,.947,f'{region} · {lead_hours//24}-day lead',ha='center',fontsize=14)
    fig.text(.5,.917,f'{stat.field_title}: polygon mean {operator} {criterion}',ha='center',fontsize=10.5)
    rows=[['Events / members',f'{int(stat.cases)} / {int(stat.ensemble_members)}'],
          ['Distinct forecasts / PAV pools',f'{int(stat.distinct_probabilities)} / {int(stat.pav_pools)}'],
          ['ERA5 crossings / observed frequency',f'{int(stat.observed_crossings)} / {stat.base_rate:.1%}'],
          ['Original Brier score',f'{stat.brier_score:.4f}'],
          ['Miscalibration (MCB)',f'{stat.miscalibration_mcb:.4f}'],
          ['Discrimination (DSC)',f'{stat.discrimination_dsc:.4f}'],
          ['Uncertainty (UNC)',f'{stat.uncertainty_unc:.4f}']]
    table_ax.axis('off');table=table_ax.table(cellText=rows,colLabels=['Statistic','Value'],
        colWidths=[.7,.3],cellLoc='left',bbox=[0,0,1,1]);table.auto_set_font_size(False);table.set_fontsize(10)
    for (r,c),cell in table.get_celld().items():
        cell.set_edgecolor('#d7e0e7');cell.set_linewidth(.55);cell.PAD=.07
        if r==0:cell.set_facecolor('#e8eef3');cell.set_text_props(weight='bold')
        elif r==5:cell.set_facecolor('#e2f0ef');cell.set_text_props(weight='bold',color='#075e65')
        elif r%2==0:cell.set_facecolor('#f6f8fa')
    start=pd.Timestamp(stat.valid_time_start);end=pd.Timestamp(stat.valid_time_end)
    band_note='90% pointwise consistency band; exchangeability assumed.' if stat.band_type!='none' else 'Uncertainty bands not shown; winter cases may be dependent.'
    fig.text(.09,.018,f'{start:%Y-%m-%d} to {end:%Y-%m-%d} · ERA5-selected events · BS = MCB − DSC + UNC\n'
             f'{band_note}\nCORP/PAV: Dimitriadis et al. (2021). Diagnostic fit; original forecasts retained.',fontsize=8.5,color='#435567')
    return fig


def plot_corp_forecast_counts(points, *, title_suffix='All'):
    """Exact native probability-value counts, saved separately from the CORP graph."""
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter,MaxNLocator
    fig,axes=plt.subplots(2,3,figsize=(14,7.5),layout='constrained')
    for r,region in enumerate(POLYGONS):
        for c,lead in enumerate(LEAD_HOURS):
            frame=points.loc[points.polygon.eq(region)&points.lead_days.eq(lead//24)]
            ax=axes[r,c]
            ax.bar(100*frame.forecast_probability,frame.case_count,width=1.2,color='#087f8c')
            ax.set(xlim=(-2,102),ylim=(0,None),xlabel='Original forecast probability',ylabel='Number of events',
                   title=f'{region} · {lead//24} days · N={int(frame.case_count.sum())}')
            ax.xaxis.set_major_formatter(PercentFormatter(100,decimals=0));ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            ax.spines[['top','right']].set_visible(False);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
    fig.suptitle(f'Forecast counts at native probability values · {title_suffix}\n'
                 '64-member probabilities occur in increments of 1/64; one case per event and panel.',fontsize=13)
    return fig


def export_corp_reliability(probabilities, destination, *, title_suffix='All', consistency_bands=False,
                             assume_exchangeable=False, n_resamples=2000, seed=37):
    """Export only saved probability diagnostics; this API accepts no archive or weather arrays."""
    import matplotlib.pyplot as plt
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    points,cases,summary=corp_calibration_tables(probabilities,consistency_bands=consistency_bands,
        assume_exchangeable=assume_exchangeable,n_resamples=n_resamples,seed=seed)
    for name,frame in (('corp_points',points),('corp_cases',cases),('corp_summary',summary)):
        temporary=destination/f'.{name}.tmp.csv';frame.to_csv(temporary,index=False);temporary.replace(destination/f'{name}.csv')
    pool_rows=[]
    for _,frame in points.groupby(['polygon','lead_days','pool_id'],sort=False):
        first=frame.iloc[0]
        pool_rows.append(dict(polygon=first.polygon,lead_days=first.lead_days,pool_id=first.pool_id,
            cases=int(frame.case_count.sum()),observed_crossings=int(frame.observed_count.sum()),
            distinct_probabilities=len(frame),forecast_min=first.pool_min_probability,forecast_max=first.pool_max_probability,
            mean_forecast_probability=float(np.average(frame.forecast_probability,weights=frame.case_count)),
            corp_probability=first.corp_probability))
    pd.DataFrame(pool_rows,columns=['polygon','lead_days','pool_id','cases','observed_crossings',
        'distinct_probabilities','forecast_min','forecast_max','mean_forecast_probability','corp_probability']).to_csv(
            destination/'corp_pools.csv',index=False)
    for polygon,tag in (('JPCZ polygon','jpcz_polygon'),('Coastal wedge','coastal_wedge')):
        for lead in LEAD_HOURS:
            stem=f'corp_{tag}_lead{lead//24:02d}d'
            present=not summary.loc[summary.polygon.eq(polygon)&summary.lead_days.eq(lead//24)].empty
            if present:
                fig=plot_corp_reliability(points,summary,region=polygon,lead_hours=lead,title_suffix=title_suffix)
                with plt.rc_context({'pdf.fonttype':42,'ps.fonttype':42}):
                    for extension in ('png','pdf'):fig.savefig(destination/f'{stem}.{extension}',dpi=300)
                plt.close(fig)
            else:
                for extension in ('png','pdf'):(destination/f'{stem}.{extension}').unlink(missing_ok=True)
    if not points.empty:
        fig=plot_corp_forecast_counts(points,title_suffix=title_suffix)
        with plt.rc_context({'pdf.fonttype':42,'ps.fonttype':42}):
            for extension in ('png','pdf'):fig.savefig(destination/f'corp_forecast_counts.{extension}',dpi=300)
        plt.close(fig)
    else:
        for extension in ('png','pdf'):(destination/f'corp_forecast_counts.{extension}').unlink(missing_ok=True)
    atomic_json_file(destination/'corp_settings.json',dict(method='CORP; weighted isotonic regression / PAV',
        reference=CORP_REFERENCE,original_forecasts_changed=False,in_sample_diagnostic=True,
        panels=json.loads(summary[['polygon','lead_days','field','pressure_hpa','units','threshold','direction','cases']].to_json(orient='records')),
        png_dpi=300,pdf_fonts='embedded TrueType',
        use='threshold crossing calibration within the ERA5-selected sample',
        monotonicity='conditional crossing frequency constrained nondecreasing in forecast probability',
        band_type='90% pointwise consistency' if consistency_bands else 'none',
        exchangeability_assumed=assume_exchangeable,n_resamples=n_resamples if consistency_bands else 0,seed=int(seed),
        brier_identity='original BS = MCB - DSC + UNC',
        reference_for_unc='within-sample base rate; descriptive decomposition, not independent CRPSS climatology'))
    return points,cases,summary


"""Per-event CRPS figures and matched CRPSS, using saved scalar values only."""
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd



SCORE_CASE_KEYS = ['review_key', 'polygon', 'lead_hours']


def validate_saved_case_scores(scores):
    required = SCORE_CASE_KEYS + ['event_number', 'valid_time_utc', 'init_time_utc', 'lead_days',
        'field', 'field_title', 'pressure_hpa', 'units', 'era5_value', 'crps']
    missing = set(required) - set(scores)
    if missing: raise ValueError(f'Missing saved CRPS columns: {sorted(missing)}. Run section 5 first.')
    if scores.empty: raise ValueError('No complete saved CRPS cases. Run section 5 first.')
    if scores.duplicated(SCORE_CASE_KEYS).any(): raise ValueError('Duplicate saved CRPS case identities.')
    if not np.isfinite(scores[['crps', 'era5_value']].to_numpy(dtype=float)).all() or scores.crps.lt(0).any():
        raise ValueError('Saved scores and ERA5 values must be finite; CRPS must be nonnegative.')
    for name in ('field', 'pressure_hpa', 'units'):
        if scores[name].nunique(dropna=False) != 1: raise ValueError(f'Keep each {name} in a separate report.')
    expected = {(region, lead) for region in POLYGONS for lead in LEAD_HOURS}
    for _, frame in scores.groupby('review_key', sort=False):
        if set(zip(frame.polygon, frame.lead_hours)) != expected:
            raise ValueError('Each event must have both polygons and all three leads in the common sample.')
        if frame.event_number.nunique() != 1 or pd.to_datetime(frame.valid_time_utc, utc=True).nunique() != 1:
            raise ValueError('An event identity has inconsistent labels or valid times.')
    if not scores.lead_days.eq(scores.lead_hours / 24).all(): raise ValueError('Inconsistent lead units.')
    valid = pd.to_datetime(scores.valid_time_utc, utc=True)
    initialized = pd.to_datetime(scores.init_time_utc, utc=True)
    if valid.isna().any() or initialized.isna().any() or not (initialized+pd.to_timedelta(scores.lead_hours, unit='h')).eq(valid).all():
        raise ValueError('Saved initialization, lead and valid time do not agree.')
    if 'in_common_sample' in scores and not scores.in_common_sample.astype(str).str.lower().eq('true').all():
        raise ValueError('Use only the complete common sample for these comparisons.')
    return scores.sort_values(['valid_time_utc', 'event_number', 'polygon', 'lead_hours']).copy()


def matched_climatology_skill(scores, reference, settings, calculation_settings):
    """Score a frozen empirical climatology on exactly the saved model cases.

    CRPSS is a ratio of mean scores. Individual cases retain model/reference CRPS
    and their difference, not an averageable individual-case skill percentage.
    """
    scores = validate_saved_case_scores(scores)
    if not settings.get('complete'): raise ValueError('Reference incomplete: finish its saved schedule before CRPSS.')
    if settings.get('weighting') != 'one target, one equal weight': raise ValueError('Unsupported reference weighting.')
    for name in ('era5_store', 'domain', 'polygons', 'pressure_hpa', 'diagnostic'):
        actual = json.loads(json.dumps(calculation_settings.get(name), sort_keys=True))
        if settings.get('calculation', {}).get(name) != actual:
            raise ValueError(f'Reference and forecast {name} definitions differ.')
    required = ['reference_key', 'valid_time_utc', 'polygon', 'value', 'field', 'pressure_hpa', 'units', 'reference_product']
    if set(required) - set(reference): raise ValueError('Incomplete reference-value schema.')
    if reference.duplicated(['reference_key', 'polygon']).any(): raise ValueError('Duplicate climatology targets.')
    if not reference.reference_product.eq('ERA5').all(): raise ValueError('Use final historical ERA5 for climatology.')
    if not np.isfinite(reference.value.to_numpy(dtype=float)).all(): raise ValueError('Nonfinite reference values.')
    for name in ('field', 'pressure_hpa', 'units'):
        if not reference[name].eq(scores[name].iloc[0]).all(): raise ValueError(f'Reference {name} mismatch.')
    times = pd.to_datetime(reference.valid_time_utc, utc=True)
    start = pd.Timestamp(settings['start_utc']); end = pd.Timestamp(settings['end_utc_exclusive'])
    if not ((times >= start) & (times < end)).all(): raise ValueError('Reference dates exceed its frozen period.')
    earliest_init = pd.to_datetime(scores.init_time_utc, utc=True).min()
    if end > earliest_init or (times >= earliest_init).any():
        raise ValueError('Climatology must end before every evaluated forecast initialization.')
    expected = int(settings['expected_targets'])
    if expected < 2: raise ValueError('Climatology requires at least two historical targets.')
    if set(reference.polygon) != set(POLYGONS): raise ValueError('Climatology must cover both polygons.')
    target_sets = []
    distributions = {}
    for region, frame in reference.groupby('polygon', sort=False):
        if len(frame) != expected: raise ValueError('Reference inventory differs from the complete frozen schedule.')
        target_sets.append(set(zip(frame.reference_key, frame.valid_time_utc)))
        distributions[region] = frame.value.to_numpy(dtype=float)
    if target_sets[0] != target_sets[1]: raise ValueError('Climatology polygons have different targets.')
    frozen_targets = {(row['reference_key'], row['valid_time_utc']) for row in settings['schedule']}
    if target_sets[0] != frozen_targets: raise ValueError('Reference values differ from the frozen target list.')
    cases = scores.copy()
    cases['reference_crps'] = [empirical_ensemble_crps(distributions[row.polygon], row.era5_value)
                               for row in cases.itertuples()]
    cases['crps_improvement'] = cases.reference_crps - cases.crps
    cases['reference_targets'] = expected
    cases['reference_kind'] = settings['kind']
    rows = []
    for (region, lead), frame in cases.groupby(['polygon', 'lead_days'], sort=False):
        model = float(frame.crps.mean()); baseline = float(frame.reference_crps.mean())
        rows.append(dict(polygon=region, lead_days=int(lead), cases=len(frame),
            mean_model_crps=model, mean_reference_crps=baseline, mean_improvement=baseline-model,
            crpss_percent=100*(1-model/baseline) if baseline > 0 else np.nan,
            status='complete' if baseline > 0 else 'undefined_zero_reference_crps',
            reference_targets=expected, reference_kind=settings['kind'],
            field=frame.field.iloc[0], pressure_hpa=int(frame.pressure_hpa.iloc[0]), units=frame.units.iloc[0]))
    return cases, pd.DataFrame(rows)


def plot_event_crps(scores, review_key):
    """One case, six saved scores. No boxplots: there is one score per lead/polygon."""
    import matplotlib.pyplot as plt
    frame = scores.loc[scores.review_key.eq(review_key)]
    if frame.empty: raise ValueError('Event not present in saved scores.')
    first = frame.iloc[0]; time = pd.Timestamp(first.valid_time_utc)
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.8))
    fig.subplots_adjust(left=.08, right=.98, top=.75, bottom=.34, wspace=.29)
    fig.suptitle(f'Event {int(first.event_number)} · CRPS at 1, 5 and 10 days', y=.97, fontsize=16)
    fig.text(.5, .89, f'{first.field_title} · verifying {time:%Y-%m-%d %H:%M} UTC', ha='center', fontsize=11)
    fig.text(.5, .84, 'One score per polygon and lead · all 64 WN2 members · lower is better', ha='center', fontsize=10)
    for ax, region in zip(axes, POLYGONS):
        panel = frame.loc[frame.polygon.eq(region)].set_index('lead_days').reindex([1, 5, 10])
        ax.plot([1, 5, 10], panel.crps, 'o-', color='#087f8c', label='WN2 CRPS')
        rows = [['WN2 CRPS', *[f'{v:.3f}' for v in panel.crps]]]
        if 'reference_crps' in panel:
            ax.plot([1, 5, 10], panel.reference_crps, 's--', color='#b56a19', label='Climatology CRPS')
            rows.extend([['Reference CRPS', *[f'{v:.3f}' for v in panel.reference_crps]],
                         ['Improvement', *[f'{v:+.3f}' for v in panel.crps_improvement]]])
        ax.set(title=region, xlabel='Forecast lead (days)', ylabel=f'CRPS ({first.units})',
               xticks=[1, 5, 10], ylim=(0, None))
        ax.grid(alpha=.2); ax.spines[['top', 'right']].set_visible(False); ax.legend(fontsize=9)
        table = ax.table(cellText=rows, colLabels=['Saved score', '1 day', '5 days', '10 days'],
            colWidths=[.43, .19, .19, .19], cellLoc='center', bbox=[0, -.49, 1, .27])
        table.auto_set_font_size(False); table.set_fontsize(9)
        for (r, _), cell in table.get_celld().items():
            cell.set_edgecolor('#d8e1e8')
            if r == 0: cell.set_facecolor('#e8eef3'); cell.set_text_props(weight='bold')
    fig.text(.08, .035, 'Scores describe polygon means, not spatial placement. '
        'CRPSS is computed from mean paired scores across cases.', fontsize=9, color='#435567')
    return fig


def plot_all_event_crps(scores):
    """Every case is labelled; color shows each case's score rather than a pooled boxplot."""
    import matplotlib.pyplot as plt
    events = scores[['review_key', 'event_number', 'valid_time_utc']].drop_duplicates('review_key')
    n = len(events); fig, axes = plt.subplots(1, 2, figsize=(11, max(5, .25*n+2.4)), layout='constrained')
    maximum = float(scores.crps.max()) or 1.
    labels = [f'#{int(r.event_number)} · {pd.Timestamp(r.valid_time_utc):%Y-%m-%d}' for r in events.itertuples()]
    for ax, region in zip(axes, POLYGONS):
        values = scores.loc[scores.polygon.eq(region)].pivot(index='review_key', columns='lead_days', values='crps')
        values = values.reindex(index=events.review_key, columns=[1, 5, 10])
        artist = ax.imshow(values, aspect='auto', vmin=0, vmax=maximum, cmap='YlGnBu', interpolation='nearest')
        ax.set(xticks=[0, 1, 2], xticklabels=['1 day', '5 days', '10 days'], yticks=np.arange(n),
               yticklabels=labels, title=region, xlabel='Forecast lead')
        ax.tick_params(axis='y', labelsize=8)
        for row in range(n):
            for col in range(3):
                value = values.iloc[row, col]
                ax.text(col, row, f'{value:.2f}', ha='center', va='center', fontsize=8,
                        color='white' if value > .55*maximum else '#182d3b')
    fig.colorbar(artist, ax=list(axes), shrink=.6, label=f'CRPS ({scores.units.iloc[0]}); lower is better')
    fig.suptitle(f'Every event · {scores.field_title.iloc[0]}\nN = {n}; each row is one case, not one member', fontsize=14)
    return fig


def export_event_crps_reports(scores, destination):
    """Write each case and one combined PDF; keep only one event figure in RAM at once."""
    import matplotlib.pyplot as plt
    from pypdf import PdfReader, PdfWriter
    scores = validate_saved_case_scores(scores)
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    scores.to_csv(destination/'all_event_scores.csv', index=False)
    events = scores[['review_key', 'event_number', 'valid_time_utc']].drop_duplicates('review_key')
    wide = scores.pivot(index=['review_key', 'event_number', 'valid_time_utc'],
                        columns=['polygon', 'lead_days'], values='crps')
    wide.columns = [f'{region} | {int(lead)} day CRPS' for region, lead in wide.columns]
    wide = wide.reset_index().sort_values(['valid_time_utc', 'event_number'])
    for value, label in (('reference_crps', 'reference CRPS'), ('crps_improvement', 'CRPS improvement')):
        if value in scores:
            extra = scores.pivot(index=['review_key', 'event_number', 'valid_time_utc'],
                                 columns=['polygon', 'lead_days'], values=value)
            extra.columns = [f'{region} | {int(lead)} day {label}' for region, lead in extra.columns]
            wide = wide.merge(extra.reset_index(), on=['review_key', 'event_number', 'valid_time_utc'], validate='one_to_one')
    wide.to_csv(destination/'one_row_per_event.csv', index=False)
    with pd.ExcelWriter(destination/'all_event_scores.xlsx', engine='xlsxwriter',
            engine_kwargs={'options': {'strings_to_formulas': False, 'strings_to_urls': False}}) as writer:
        wide.to_excel(writer, sheet_name='All events', index=False)
        sheet = writer.sheets['All events']; sheet.freeze_panes(1, 3); sheet.set_column(0, len(wide.columns)-1, 24)
        sheet.autofilter(0, 0, len(wide), len(wide.columns)-1)
    index = []
    with plt.rc_context({'pdf.fonttype': 42, 'ps.fonttype': 42}):
        temporary_pdf = destination/'.all_events_crps.tmp.pdf'
        with PdfWriter() as combined:
            for row in events.itertuples():
                stamp = pd.Timestamp(row.valid_time_utc).strftime('%Y%m%dT%H%MZ')
                stem = f'event_{int(row.event_number):04d}_{stamp}_crps'
                case = scores.loc[scores.review_key.eq(row.review_key)]
                signature = hashlib.sha256(('event_crps_v1|'+case.to_json(orient='split', double_precision=15)).encode()).hexdigest()
                marker = destination/f'{stem}.json'
                cached = False
                if marker.is_file() and all((destination/f'{stem}.{ext}').is_file() for ext in ('png', 'pdf', 'csv')):
                    try: cached = json.loads(marker.read_text()).get('signature') == signature
                    except (ValueError, OSError): pass
                if not cached:
                    fig = plot_event_crps(scores, row.review_key)
                    try:
                        fig.savefig(destination/f'{stem}.png', dpi=220)
                        fig.savefig(destination/f'{stem}.pdf')
                    finally: plt.close(fig)
                    case.to_csv(destination/f'{stem}.csv', index=False)
                    atomic_json_file(marker, dict(signature=signature))
                combined.append(PdfReader(destination/f'{stem}.pdf'))
                index.append(dict(event_number=int(row.event_number), review_key=row.review_key,
                    event_png=str(destination/f'{stem}.png'), event_pdf=str(destination/f'{stem}.pdf'),
                    status='cached' if cached else 'exported'))
            combined.write(temporary_pdf)
        temporary_pdf.replace(destination/'all_events_crps.pdf')
        fig = plot_all_event_crps(scores)
        try:
            fig.savefig(destination/'all_events_crps_overview.png', dpi=220)
            fig.savefig(destination/'all_events_crps_overview.pdf')
        finally: plt.close(fig)
    index = pd.DataFrame(index); index.to_csv(destination/'event_figure_index.csv', index=False)
    atomic_json_file(destination/'case_score_settings.json', dict(events=len(events), scores=len(scores),
        source='saved crps_cases.csv', weather_fields_recalculated=False, score_unit=scores.units.iloc[0],
        combined_pdf='all_events_crps.pdf', sample='same complete cases at all leads'))
    return index, wide


def export_crpss_report(scores, reference_directory, destination, calculation_settings):
    """No archive access. Missing/partial climatology yields an explicit pending status."""
    import matplotlib.pyplot as plt
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    reference_directory = Path(reference_directory)
    paths = [reference_directory/'reference_values.csv', reference_directory/'reference_settings.json']
    if not all(path.is_file() for path in paths):
        status = dict(status='reference_not_prepared', message='Build the independent ERA5 climatology in section 5c first.')
        cases = summary = None
    else:
        settings = json.loads(paths[1].read_text())
        if not settings.get('complete'):
            status = dict(status='reference_incomplete', completed_targets=settings.get('completed_targets', 0),
                          expected_targets=settings.get('expected_targets', 0), message='Resume section 5c; no partial-reference skill is reported.')
            cases = summary = None
        else:
            reference = pd.read_csv(paths[0], float_precision='round_trip')
            cases, summary = matched_climatology_skill(scores, reference, settings, calculation_settings)
            status = dict(status='complete', reference_directory=str(reference_directory),
                definition='100 * (1 - mean(model CRPS) / mean(reference CRPS))',
                sample='identical cases and equal event weights', reference_kind=settings['kind'])
    atomic_json_file(destination/'crpss_status.json', status)
    if cases is None:
        for name in ('crpss_cases.csv', 'crpss_summary.csv', 'crpss_summary.png', 'crpss_summary.pdf'):
            (destination/name).unlink(missing_ok=True)
        return status, cases, summary
    cases.to_csv(destination/'crpss_cases.csv', index=False); summary.to_csv(destination/'crpss_summary.csv', index=False)
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 7))
    fig.subplots_adjust(left=.08, right=.98, top=.77, bottom=.39, wspace=.3)
    for ax, region in zip(axes, POLYGONS):
        panel = summary.loc[summary.polygon.eq(region)].sort_values('lead_days')
        ax.axhline(0, color='#45535e', linestyle='--', linewidth=1)
        ax.plot(panel.lead_days, panel.crpss_percent, 'o-', color='#087f8c')
        ax.set(title=region, xticks=[1, 5, 10], xlabel='Forecast lead (days)', ylabel='CRPSS (%)')
        ax.spines[['top', 'right']].set_visible(False); ax.grid(alpha=.2)
        panel = panel.set_index('lead_days').reindex([1, 5, 10])
        rows = [['Paired events', *[str(int(x)) for x in panel.cases]],
                ['Reference targets', *[str(int(x)) for x in panel.reference_targets]],
                ['Mean WN2 CRPS', *[f'{x:.3f}' for x in panel.mean_model_crps]],
                ['Mean reference CRPS', *[f'{x:.3f}' for x in panel.mean_reference_crps]],
                ['CRPSS (%)', *[f'{x:+.1f}' if np.isfinite(x) else 'undefined' for x in panel.crpss_percent]]]
        table = ax.table(cellText=rows, colLabels=['Statistic', '1 day', '5 days', '10 days'],
            colWidths=[.46, .18, .18, .18], cellLoc='center', bbox=[0, -.76, 1, .51])
        table.auto_set_font_size(False); table.set_fontsize(9)
        for (r, _), cell in table.get_celld().items():
            cell.set_edgecolor('#d8e1e8')
            if r == 0: cell.set_facecolor('#e8eef3'); cell.set_text_props(weight='bold')
    first = scores.iloc[0]
    fig.suptitle(f'CRPSS · {first.field_title}\nSkill relative to a frozen historical ERA5 distribution', y=.97, fontsize=14)
    fig.text(.5, .84, f"{settings['kind']} · {settings['start_utc'][:10]} to {settings['end_utc_exclusive'][:10]} (end exclusive)",
             ha='center', fontsize=10)
    fig.text(.08, .035, f'Positive: better than reference. CRPS units: {first.units}. '
        'CRPSS is a ratio of mean paired scores.\nReference dates precede evaluated forecasts; '
        'this does not verify the WN2 checkpoint training cutoff.', fontsize=9, color='#435567')
    with plt.rc_context({'pdf.fonttype': 42}):
        fig.savefig(destination/'crpss_summary.png', dpi=220); fig.savefig(destination/'crpss_summary.pdf')
    plt.close(fig)
    return status, cases, summary


"""Frozen, resumable historical ERA5 reference; no WN2 forecasts are requested."""
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd





def climatology_schedule(catalog_path, *, kind, start='2000-01-01', end='2022-01-01'):
    """A fixed historical JPCZ-event sample or every NDJF synoptic hour; end is exclusive."""
    start = pd.Timestamp(start, tz='UTC'); end = pd.Timestamp(end, tz='UTC')
    if end <= start: raise ValueError('Reference end must follow its start.')
    if kind == 'Historical JPCZ cases':
        frame = normalise_catalog(pd.read_csv(catalog_path))
        onset = pd.to_datetime(frame.event_start, utc=True)
        frame = frame.loc[onset.ge(start) & onset.lt(end) & onset.dt.month.isin([11, 12, 1, 2])].copy()
        frame['valid_time_utc'] = frame.target_valid_time.map(iso_utc)
        frame['reference_key'] = frame.review_key
        frame = frame[['reference_key', 'event_number', 'valid_time_utc']]
    elif kind == 'All winter conditions':
        times = pd.date_range(start, end, freq='6h', inclusive='left')
        times = times[times.month.isin([11, 12, 1, 2])]
        frame = pd.DataFrame({'reference_key': [f'winter|{t.isoformat()}' for t in times],
                              'event_number': pd.NA, 'valid_time_utc': [iso_utc(t) for t in times]})
    else: raise ValueError('Choose Historical JPCZ cases or All winter conditions.')
    if frame.empty: raise ValueError('No reference targets in the selected period.')
    if frame.reference_key.duplicated().any() or frame.valid_time_utc.duplicated().any():
        raise ValueError('Reference targets must be unique; duplicate targets would overweight a weather state.')
    times = pd.to_datetime(frame.valid_time_utc, utc=True)
    if not (times.ge(start) & times.lt(end)).all(): raise ValueError('Reference target lies outside its frozen period.')
    return frame.sort_values('valid_time_utc').reset_index(drop=True)


def climatology_definition(schedule, calculation, *, kind, start, end):
    """Archive the sampling and spatial/derivative conventions before any skill evaluation."""
    definition = dict(kind=kind, start_utc=iso_utc(start), end_utc_exclusive=iso_utc(end),
        weighting='one target, one equal weight', expected_targets=len(schedule),
        season='NDJF', matching='pooled historical distribution; no test-outcome tuning',
        target_rule='nearest 00/06/12/18 UTC on onset date, ties earlier' if kind=='Historical JPCZ cases' else 'every 6-hourly NDJF target',
        calculation={key: calculation.get(key) for key in ('era5_store', 'domain', 'polygons', 'pressure_hpa', 'diagnostic')},
        field=calculation.get('field', 'divergence'),
        schedule=schedule[['reference_key', 'valid_time_utc']].to_dict('records'))
    definition = json.loads(json.dumps(definition, sort_keys=True))
    signature = hashlib.sha256(json.dumps(definition, sort_keys=True).encode()).hexdigest()[:16]
    return definition, signature


def build_saved_climatology(schedule, destination, definition, spec, *, job_runner=None,
                            download_missing=False, limit=25, shared_reference_cache=None, progress=print):
    """Save each target immediately; decode weather only in the existing bounded worker.

    Preview/reopen mode reads JSON/scalar caches only. A complete frozen schedule is
    required by CRPSS; failures and unattempted dates remain explicit rather than zero.
    """
    if download_missing and job_runner is None: raise ValueError('Missing-reference retrieval requires the bounded ERA5 worker.')
    if limit is not None and (not isinstance(limit, int) or isinstance(limit, bool) or limit < 1):
        raise ValueError('Reference cases per run must be a positive integer or None.')
    if len(schedule) != definition['expected_targets'] or schedule[['reference_key', 'valid_time_utc']].to_dict('records') != definition['schedule']:
        raise ValueError('The reference schedule changed after its definition was frozen.')
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    settings_path = destination/'reference_settings.json'
    if settings_path.is_file():
        old = json.loads(settings_path.read_text())
        if any(old.get(key) != value for key, value in definition.items()):
            raise ValueError('Use a new reference folder for changed sampling or calculation definitions.')
    schedule.to_csv(destination/'reference_schedule.csv', index=False)
    statuses = []; values = []; attempted = 0
    field_identity = dict(field=spec.key, pressure_hpa=spec.level, units=spec.units,
        calculation=definition['calculation'])

    def checked(data, target):
        if data.get('valid_time') != target or data.get('reference_product', 'ERA5') != 'ERA5':
            raise ValueError('Reference needs exact-time final ERA5.')
        if set(data.get('values', {})) != set(POLYGONS) or not np.isfinite(list(data['values'].values())).all():
            raise ValueError('Reference target lacks finite means for both polygons.')
        if data.get('source_store') != definition['calculation']['era5_store']:
            raise ValueError('Reference came from a different ERA5 archive.')
        return data

    def save_progress():
        done = len({row['reference_key'] for row in values})
        pd.DataFrame(values, columns=['reference_key', 'event_number', 'valid_time_utc', 'polygon',
            'field', 'field_title', 'pressure_hpa', 'units', 'value', 'reference_product']).to_csv(
                destination/'.reference_values.tmp.csv', index=False)
        (destination/'.reference_values.tmp.csv').replace(destination/'reference_values.csv')
        pd.DataFrame(statuses).to_csv(destination/'reference_status.csv', index=False)
        atomic_json_file(settings_path, dict(**definition, completed_targets=done,
            complete=done==len(schedule), pending_targets=len(schedule)-done,
            independent_reference='all reference dates precede evaluated forecast initializations; checked again when scoring'))

    # Collect valid cached targets first. A small retrieval limit never hides completed
    # targets later in the schedule after a runtime restart.
    pending = []
    for row in schedule.itertuples():
        target = row.valid_time_utc; stamp = pd.Timestamp(target).strftime('%Y%m%dT%H%MZ')
        path = destination/'targets'/f'era5_{stamp}.json'
        identity = dict(**field_identity, valid_time=target)
        data = None
        if path.is_file():
            saved = json.loads(path.read_text())
            if saved.get('request') == identity:
                try: data = checked(saved['data'], target)
                except (KeyError, ValueError): pass
        if data is None and shared_reference_cache is not None:
            shared = Path(shared_reference_cache)/f'era5_{stamp}.json'
            if shared.is_file():
                saved = json.loads(shared.read_text())
                if saved.get('request') == {'valid_time': target}:
                    try:
                        data = checked(saved['data'], target)
                        atomic_json_file(path, dict(request=identity, data=data))
                    except (KeyError, ValueError): pass
        statuses.append(dict(reference_key=row.reference_key, valid_time_utc=target, status='cached' if data else 'pending', error=''))
        if data is None:
            pending.append((row, path, identity, len(statuses)-1))
        else:
            for region in POLYGONS:
                values.append(dict(reference_key=row.reference_key, event_number=row.event_number,
                    valid_time_utc=target, polygon=region, field=spec.key, field_title=spec.title,
                    pressure_hpa=spec.level, units=spec.units, value=data['values'][region], reference_product='ERA5'))
    save_progress()
    for row, path, identity, status_index in pending:
        if not download_missing or (limit is not None and attempted >= limit): break
        attempted += 1
        progress(f'Climatology ERA5 {attempted}: {row.valid_time_utc}; one target in a bounded worker')
        try:
            data = checked(job_runner('reference', dict(valid_time=row.valid_time_utc, allow_era5t=False, spec=spec.__dict__)), row.valid_time_utc)
            atomic_json_file(path, dict(request=identity, data=data))
            for region in POLYGONS:
                values.append(dict(reference_key=row.reference_key, event_number=row.event_number,
                    valid_time_utc=row.valid_time_utc, polygon=region, field=spec.key, field_title=spec.title,
                    pressure_hpa=spec.level, units=spec.units, value=data['values'][region], reference_product='ERA5'))
            statuses[status_index]['status'] = 'computed'
        except Exception as exc:
            statuses[status_index].update(status='failed', error=f'{type(exc).__name__}: {exc}')
            progress(statuses[status_index]['error'])
        save_progress()
    result = json.loads(settings_path.read_text())
    progress(f"Reference: {result['completed_targets']}/{result['expected_targets']} saved targets; complete={result['complete']}")
    return result


"""Configurable verification runs and Drive-only report collection."""
from dataclasses import replace
import json
from pathlib import Path
import pandas as pd





SUPPORTED_VERIFICATION_FIELDS=('divergence','u','v','winds','height','geopotential')


def verification_field_configs(configs):
    if not configs:raise ValueError('Select at least one verification field.')
    records=[]
    for config in configs:
        key=config['field']
        if key not in SUPPORTED_VERIFICATION_FIELDS:
            raise ValueError(f'Choose a supported verification field: {SUPPORTED_VERIFICATION_FIELDS}.')
        spec=replace(FIELD_PRESETS[key],level=int(config.get('level',FIELD_PRESETS[key].level)))
        if key=='winds':spec=replace(spec,label='wind speed (derived from u/v)')
        if spec.level not in (50,100,150,200,250,300,400,500,600,700,850,925,1000):
            raise ValueError('Select an available pressure level.')
        identifier=f'{spec.key}_{spec.level}hpa'
        if any(item['id']==identifier for item in records):raise ValueError('Duplicate verification field and level.')
        threshold=config.get('threshold')
        if threshold is not None:
            import math
            if not math.isfinite(float(threshold)):raise ValueError('A reliability threshold must be finite.')
        direction=config.get('direction','less')
        if direction not in ('less','greater'):raise ValueError('Threshold direction must be less or greater.')
        records.append(dict(id=identifier,spec=spec,config={**config,'level':spec.level,'threshold':threshold,'direction':direction}))
    return records


def load_saved_verification(destination, *, catalog=None, spec=None):
    """Read exported CSVs only. This function has no archive/reader callback."""
    destination=Path(destination)
    names=('case_ranks','member_values','era5_reference','batch_status')
    paths=[destination/f'{name}.csv' for name in names]
    if not all(path.is_file() for path in paths):
        raise FileNotFoundError(f'Saved verification tables are incomplete in {destination}. Retrieve missing inputs first.')
    settings=json.loads((destination/'calculation_settings.json').read_text())
    if spec is not None:
        if settings.get('field','divergence')!=spec.key or settings.get('pressure_hpa',925)!=spec.level:
            raise ValueError('Saved tables use a different verification field/level.')
    tables={}
    for name,path in zip(names,paths):
        try:frame=pd.read_csv(path,dtype={'member':'string','review_key':'string'},float_precision='round_trip')
        except pd.errors.EmptyDataError:frame=pd.DataFrame()
        if catalog is not None and 'review_key' in frame:
            frame=frame.loc[frame.review_key.isin(catalog.review_key)].copy()
            if 'valid_time_utc' in frame and 'target_valid_time' in catalog:
                targets=pd.to_datetime(catalog.set_index('review_key').target_valid_time,utc=True)
                frame=frame.loc[pd.to_datetime(frame.valid_time_utc,utc=True,errors='coerce').eq(frame.review_key.map(targets))].copy()
            labels=[c for c in ('vortex_label','vortex_scan_status','vortex_classification') if c in frame]
            for key in labels:
                column='vortex_plot_label' if key=='vortex_label' else key
                if column in catalog:frame[key]=frame.review_key.map(catalog.set_index('review_key')[column])
        tables[name]=frame
    if catalog is not None:
        present=set(zip(tables['batch_status'].get('review_key',[]),tables['batch_status'].get('lead_hours',[])))
        missing=[]
        for _,row in catalog.iterrows():
            for lead in LEAD_HOURS:
                if (row.review_key,lead) not in present:
                    missing.append(dict(event_number=int(row.event_number),review_key=row.review_key,
                        lead_hours=lead,lead_days=lead//24,era5_status='not_downloaded',wn2_status='not_downloaded',
                        era5_error='Saved-only mode: target/lead not yet retrieved.',wn2_error='Saved-only mode: target/lead not yet retrieved.'))
        if missing:tables['batch_status']=pd.concat([tables['batch_status'],pd.DataFrame(missing)],ignore_index=True)
    return tables,settings


def run_verification_fields(catalog, archive, output_dir, result_directory, configs, *,
                            saved_only=False,job_runner=None,allow_era5t=False,progress=print):
    """Keep each variable/level's physical units, checkpoints, and plots independent."""
    results=[]
    for item in verification_field_configs(configs):
        spec=item['spec']
        destination=Path(result_directory) if item['id']=='divergence_925hpa' else Path(result_directory)/'fields'/item['id']
        progress(f"Field {spec.title} | {'saved tables only' if saved_only else 'resume missing inputs'}")
        if saved_only:
            tables,settings=load_saved_verification(destination,catalog=catalog,spec=spec)
            expected=json.loads(json.dumps(checkpoint_identity(archive,spec),sort_keys=True))
            if any(settings.get(key)!=value for key,value in expected.items()):
                raise ValueError('Saved verification settings differ from this source/field configuration. '
                                 'Use the original settings or retrieve the new configuration explicitly.')
        else:
            tables,settings=run_ensemble_verification(catalog,archive,output_dir,spec=spec,
                job_runner=job_runner,allow_era5t=allow_era5t,progress=progress)
            destination.mkdir(parents=True,exist_ok=True)
            for name,frame in tables.items():
                temporary=destination/f'.{name}.tmp.csv';frame.to_csv(temporary,index=False);temporary.replace(destination/f'{name}.csv')
            atomic_json_file(destination/'calculation_settings.json',settings)
        results.append({**item,'tables':tables,'settings':settings,'directory':destination})
    return results


def consolidate_saved_verification(sources, destination):
    """Combine existing ordered-value rows into ONE Excel sheet, without any field calculation."""
    frames=[]
    seen=set()
    for source in sources:
        path=Path(source)/'ordered_values.csv'
        if str(path.resolve()) in seen:continue
        seen.add(str(path.resolve()))
        if not path.is_file():raise FileNotFoundError(f'Run the report export first: {path}')
        try:frame=pd.read_csv(path,dtype={'member':'string','review_key':'string'},float_precision='round_trip')
        except pd.errors.EmptyDataError:frame=pd.DataFrame()
        if 'value' not in frame and 'divergence_1e5_s-1' in frame:frame['value']=frame['divergence_1e5_s-1']
        if 'field' not in frame:frame['field']='divergence'
        if 'units' not in frame:frame['units']='10⁻⁵ s⁻¹'
        frame['saved_table']=str(path)
        # Keep native values even when a missing reference/lead prevents a rank from being assigned.
        supplements=[]
        for table,source_name in (('member_values','WN2'),('era5_reference','ERA5')):
            raw_path=Path(source)/f'{table}.csv'
            if not raw_path.is_file():continue
            try:raw=pd.read_csv(raw_path,dtype={'member':'string','review_key':'string'},float_precision='round_trip')
            except pd.errors.EmptyDataError:continue
            if raw.empty:continue
            raw['source']=source_name
            raw['saved_table']=str(raw_path)
            if 'value' not in raw:raw['value']=raw['divergence_1e5_s-1']
            if 'field' not in raw:raw['field']='divergence'
            if 'units' not in raw:raw['units']='10⁻⁵ s⁻¹'
            if source_name=='ERA5':
                raw['member']=''
                copies=[]
                for lead in LEAD_HOURS:
                    copy=raw.copy();copy['lead_hours']=lead;copy['lead_days']=lead//24
                    copy['init_time_utc']=(pd.to_datetime(copy.valid_time_utc,utc=True)-pd.Timedelta(hours=lead)).astype(str)
                    copies.append(copy)
                raw=pd.concat(copies,ignore_index=True)
            supplements.append(raw)
        if supplements:
            frame=pd.concat([frame,*supplements],ignore_index=True,sort=False)
            frame['member']=frame['member'].fillna('').astype(str)
            keys=['review_key','field','pressure_hpa','polygon','lead_hours','source','member']
            for col in ('pressure_hpa','lead_hours'):
                frame[col]=pd.to_numeric(frame[col],errors='raise').astype('Int64')
            frame=frame.drop_duplicates(keys,keep='first')
        scores_path=Path(source)/'crps_cases.csv'
        if scores_path.is_file() and not frame.empty:
            try:scores=pd.read_csv(scores_path,dtype={'review_key':'string'},float_precision='round_trip')
            except pd.errors.EmptyDataError:scores=pd.DataFrame()
            if not scores.empty:
                keys=['review_key','field','pressure_hpa','polygon','lead_hours']
                scores=scores[keys+['crps']].rename(columns={'crps':'case_crps'})
                if scores.duplicated(keys).any():raise ValueError('Duplicate saved CRPS case identities.')
                frame=frame.drop(columns=['case_crps'],errors='ignore').merge(scores,on=keys,how='left',validate='many_to_one')
        frames.append(frame)
    if not frames:raise ValueError('Select saved result folders to combine.')
    values=pd.concat(frames,ignore_index=True,sort=False)
    preferred=['event_number','review_key','field','pressure_hpa','units','vortex_classification','polygon','lead_days',
               'valid_time_utc','init_time_utc','source','member','value','position','bin_number','in_common_sample','case_crps','saved_table']
    values=values[[c for c in preferred if c in values]+[c for c in values if c not in preferred]]
    values=values.sort_values([c for c in ['event_number','field','pressure_hpa','polygon','lead_days','position'] if c in values])
    if len(values)>1048575:raise ValueError('The combined values exceed one Excel sheet; select fewer saved folders.')
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    csv=destination/'all_verification_values.csv';tmp=csv.with_suffix('.tmp.csv')
    values.to_csv(tmp,index=False);tmp.replace(csv)
    workbook=destination/'all_verification_values.xlsx';tmp=workbook.with_suffix('.tmp.xlsx')
    with pd.ExcelWriter(tmp,engine='xlsxwriter',engine_kwargs={'options':{'strings_to_formulas':False,'strings_to_urls':False}}) as writer:
        values.to_excel(writer,sheet_name='All values',index=False)
        sheet=writer.sheets['All values'];sheet.freeze_panes(1,0);sheet.hide_gridlines(2)
        sheet.autofilter(0,0,len(values),max(0,len(values.columns)-1));sheet.set_column(0,len(values.columns)-1,18)
        if 'source' in values:
            from xlsxwriter.utility import xl_col_to_name
            col=xl_col_to_name(values.columns.get_loc('source'))
            sheet.conditional_format(1,0,len(values),len(values.columns)-1,{'type':'formula',
                'criteria':f'=${col}2="ERA5"','format':writer.book.add_format({'bg_color':'#DDF0DE','bold':True})})
    tmp.replace(workbook)
    return values,workbook


def saved_verification_sources(result_directory):
    """Find saved field tables within one catalog/pilot run; do not mix other study folders."""
    root=Path(result_directory)
    sources=([root] if (root/'ordered_values.csv').is_file() else [])
    sources.extend(path.parent for path in sorted((root/'fields').glob('*/ordered_values.csv')))
    return sources


def export_field_reports(runs, selection_report=None):
    """Build report artifacts from already retrieved tables; never fetch weather fields."""
    import matplotlib.pyplot as plt
    for run in runs:
        tables,settings,destination=run['tables'],run['settings'],Path(run['directory'])
        if selection_report is not None:tables['event_selection']=selection_report
        export_verification(tables,settings,destination)
        export_continuous_scores(tables,destination)
        for group in ('All','Vortex','Non-vortex','Unknown'):
            export_continuous_scores(tables,destination/'event_conditioned'/group.lower().replace('-','_'),group=group)
        ranks=tables['case_ranks']
        for grouped,stem in ((False,'rank_histograms_65ranks'),(True,'rank_histograms')):
            if not ranks.empty and ranks.in_common_sample.any():
                fig=plot_rank_histograms(ranks,grouped=grouped)
                for extension in ('png','pdf'):fig.savefig(destination/f'{stem}.{extension}',dpi=180)
                plt.close(fig)
            else:
                for extension in ('png','pdf'):(destination/f'{stem}.{extension}').unlink(missing_ok=True)


def _fixed_reliability_matches(folder, probabilities, n_bins):
    """Reuse the fixed-bin companion when its complete saved case inventory still matches."""
    try:
        settings=json.loads((folder/'reliability_settings.json').read_text())
        if settings.get('n_bins')!=n_bins:return False
        saved=pd.read_csv(folder/'reliability_probabilities.csv',float_precision='round_trip')
        if saved.empty and probabilities.empty:return True
        columns=['review_key','polygon','lead_hours','forecast_probability','observed','threshold','direction','field','pressure_hpa','units','sample']
        keys=['review_key','polygon','lead_hours']
        pd.testing.assert_frame_equal(saved[columns].sort_values(keys).reset_index(drop=True),
            probabilities[columns].sort_values(keys).reset_index(drop=True),check_dtype=False,check_exact=True)
        stems=['reliability_diagrams','forecast_probability_counts',
               *[f'reliability_{tag}_lead{lead//24:02d}d' for tag in ('jpcz_polygon','coastal_wedge') for lead in LEAD_HOURS]]
        return all((folder/f'{stem}.{ext}').is_file() for stem in stems for ext in ('png','pdf'))
    except (OSError,ValueError,KeyError,AssertionError):return False


def export_field_reliability(runs, *, n_bins=10, method='Fixed bins', consistency_bands=False,
                              assume_exchangeable=False, n_resamples=2000, seed=37):
    """Rebuild calibration figures from saved scalar tables with an explicit per-field criterion."""
    import matplotlib.pyplot as plt
    if method not in ('Fixed bins','CORP'):raise ValueError('Choose Fixed bins or CORP.')
    folders=[]
    for run in runs:
        threshold=run['config'].get('threshold')
        if threshold is None:
            print(f"{run.get('id','Field')}: no reliability threshold selected; values/ranks remain saved.")
            continue
        for group in ('All','Vortex','Non-vortex','Unknown'):
            folder=Path(run['directory'])/'event_conditioned'/group.lower().replace('-','_')
            thresholds={region:float(threshold) for region in POLYGONS}
            direction=run['config'].get('direction','less')
            p=reliability_probabilities(run['tables']['member_values'],verification_subset(run['tables'],group),
                thresholds,direction=direction)
            if not _fixed_reliability_matches(folder,p,n_bins):
                p,_=export_reliability(run['tables'],folder,thresholds,group=group,n_bins=n_bins,direction=direction)
            if method=='CORP':
                export_corp_reliability(p,folder/'corp',title_suffix=group,consistency_bands=consistency_bands,
                    assume_exchangeable=assume_exchangeable,n_resamples=n_resamples,seed=seed)
            ranks=verification_subset(run['tables'],group)
            n=0 if ranks.empty else ranks.loc[ranks.in_common_sample,'review_key'].nunique()
            print(f"{run.get('id','Field')} / {group}: {n} complete events; {folder}")
            for grouped,stem in ((False,'rank_histograms_65ranks'),(True,'rank_histograms')):
                if n:
                    rank_histogram_counts(ranks,grouped=grouped).to_csv(folder/f'{stem}_counts.csv',index=False)
                    fig=plot_rank_histograms(ranks,grouped=grouped,title_suffix=group)
                    for extension in ('png','pdf'):fig.savefig(folder/f'{stem}.{extension}',dpi=180)
                    plt.close(fig)
                else:
                    for suffix in ('.png','.pdf','_counts.csv'):(folder/(stem+suffix)).unlink(missing_ok=True)
            folders.append(folder/'corp' if method=='CORP' else folder)
    return folders

print('Verification helpers loaded. Continue to section 3d.')


In [ ]:
# @title 3c2 · Load bounded forecast batch runtime
"""Fresh-process forecast jobs, with credentials passed only over an in-memory pipe."""
import hashlib
from io import BytesIO
import json
import os
from pathlib import Path
import select
import subprocess
import sys
import tempfile
import time
from urllib.request import urlopen
import zipfile

FORECAST_WORKER_SHA256 = ''
FORECAST_WORKER_URL = ''


def forecast_json(value):
    import numpy as np
    import pandas as pd
    if isinstance(value,dict):return {str(k):forecast_json(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [forecast_json(v) for v in value]
    if isinstance(value,np.generic):return forecast_json(value.item())
    if isinstance(value,(pd.Timestamp,Path)):return str(value)
    if value is None or value is pd.NA or (isinstance(value,float) and not np.isfinite(value)):return None
    return value


def prepare_forecast_worker(digest, url, archive_path=None):
    root=Path(tempfile.gettempdir())/'jpcz_forecast_workers'/digest
    root.mkdir(parents=True,exist_ok=True)
    archive=root/'runtime.zip'
    if archive_path is not None:payload=Path(archive_path).read_bytes()
    elif archive.exists():payload=archive.read_bytes()
    else:
        with urlopen(url,timeout=60) as response:payload=response.read()
    if hashlib.sha256(payload).hexdigest()!=digest:
        raise RuntimeError('Forecast worker checksum mismatch. Reload the updated notebook.')
    archive.write_bytes(payload)
    allowed={'worker.py','jpcz_catalog/__init__.py','jpcz_catalog/weathernext_atlas.py',
             'jpcz_catalog/ensemble_verification.py'}
    with zipfile.ZipFile(BytesIO(payload)) as bundle:
        if set(bundle.namelist())!=allowed:raise RuntimeError('Unexpected forecast worker files.')
        for name in sorted(allowed):
            path=root/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(bundle.read(name))
    return root/'worker.py'


def make_forecast_job_runner(archive, *, archive_path=None, digest=None, url=None, timeout=1200):
    """Jobs return compact metadata/scalars; weather arrays stay in their child process."""
    worker=None
    def run(kind,payload):
        nonlocal worker
        if kind=='forecast':
            # Recycle the interpreter after 16 members (four source groups), never all 64.
            parts=[run('forecast_part',{**payload,'member_start':start,'member_stop':start+16})
                   for start in range(0,64,16)]
            first=dict(parts[0])
            for part in parts[1:]:
                if any(part[key]!=first[key] for key in ('init_time','valid_time','lead_hours','source_store','polygon_cells')):
                    raise ValueError('Member workers returned inconsistent forecast sources or timing.')
            first['members']=[member for part in parts for member in part['members']]
            if len(first['members'])!=64 or len(set(first['members']))!=64:
                raise ValueError('Forecast workers did not return 64 distinct native members.')
            first['values']={region:[value for part in parts for value in part['values'][region]] for region in first['values']}
            return first
        if worker is None:
            worker=prepare_forecast_worker(digest or FORECAST_WORKER_SHA256,url or FORECAST_WORKER_URL,archive_path)
        credentials=archive.storage_options['token']
        expiry=getattr(credentials,'expiry',None)
        expiring=False
        if expiry is not None:
            import pandas as pd
            expiry=pd.Timestamp(expiry)
            expiry=expiry.tz_localize('UTC') if expiry.tzinfo is None else expiry.tz_convert('UTC')
            expiring=(expiry-pd.Timestamp.now(tz='UTC')).total_seconds()<timeout+120
        if not credentials.valid or expiring:
            from google.auth.transport.requests import Request
            credentials.refresh(Request())
        if not credentials.token:raise RuntimeError('Rerun Google authentication before retrieving forecasts.')
        request=forecast_json(dict(kind=kind,payload=payload,token=credentials.token,
            project=archive.storage_options.get('project',''),root=archive.root,overrides=archive.overrides,
            input_cache_dir=getattr(archive,'input_cache_dir',None)))
        env=os.environ.copy()
        env.update(MPLBACKEND='Agg',OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',NUMEXPR_NUM_THREADS='1')
        with tempfile.TemporaryDirectory(prefix='jpcz_forecast_reply_') as folder:
            reply=Path(folder)/'reply.json';request['reply_path']=str(reply)
            process=subprocess.Popen([sys.executable,'-u',str(worker)],stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
            try:
                process.stdin.write(json.dumps(request,allow_nan=False));process.stdin.close();del request
                deadline=time.monotonic()+timeout
                while process.poll() is None:
                    if time.monotonic()>deadline:raise RuntimeError('Forecast worker timed out; saved groups remain reusable.')
                    ready,_,_=select.select([process.stdout],[],[],1)
                    if ready:
                        line=process.stdout.readline()
                        if line:print(line,end='',flush=True)
                for line in process.stdout:print(line,end='',flush=True)
                if process.returncode or not reply.exists():
                    raise RuntimeError(f'Forecast worker exited {process.returncode}; completed checkpoints remain reusable.')
                result=json.loads(reply.read_text())
                if 'error' in result:raise RuntimeError(result['error'])
                return result['data']
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:process.wait(timeout=5)
                    except subprocess.TimeoutExpired:process.kill();process.wait()
                for stream in (process.stdin,process.stdout):
                    if stream and not stream.closed:stream.close()
    return run

FORECAST_WORKER_SHA256 = 'c039b197797130a2637566e85e5b456d21799630bb9ed5ec8cf195dda0dd8b94'
FORECAST_WORKER_URL = 'https://raw.githubusercontent.com/angelicasophyaramirez-blip/JPCZcatalogcolab/codex/notebook36-weathernext2/notebooks/support/forecast_worker_c039b197797130a2.zip'


In [ ]:
# @title 3d · Preview the selected catalog
from IPython.display import display

if not callable(globals().get('read_catalog')) or not callable(globals().get('run_ensemble_verification')):
    raise RuntimeError(
        'Run sections 3b and 3c in this Notebook 37 runtime, then rerun section 3d. '
        'Each helper cell prints a confirmation when it finishes.')
catalog, selection_report = read_catalog(
    CATALOG_PATH, NOTEBOOK35_DIR, STUDY_START_UTC, STUDY_END_UTC or None)
if catalog.empty:
    raise ValueError('The current catalog has no events in the configured study window.')
if RUN_MODE not in ('Pilot', 'Catalog'):
    raise ValueError('Choose Pilot or Catalog.')
if RUN_MODE == 'Pilot':
    selected = catalog.head(1) if PILOT_EVENT_NUMBER == 0 else catalog.loc[
        catalog.event_number.eq(PILOT_EVENT_NUMBER)]
    if selected.empty:
        raise ValueError('Pilot event not found in this study window. Pick a listed event number.')
    result_directory = OUTPUT_DIR / f'pilot_event_{int(selected.event_number.iloc[0]):04d}'
else:
    selected = catalog
    result_directory = OUTPUT_DIR / 'catalog'
base_result_directory = result_directory
archive = GoogleCloudArchive(credentials, PROJECT_ID or detected_project or '',
                             overrides=RUN_OVERRIDES, lazy_indexing=True,
                             input_cache_dir=DRIVE_ROOT / 'wn2_shared_inputs')
print(f'{len(catalog)} events in the study window. {RUN_MODE} will process {len(selected)}.')
print('Selected merged events by onset year:', catalog.groupby(catalog.event_start.dt.year).size().to_dict())
print('Year counts are disjoint; a count after 2024 is already included in the count after 2022.')
if RUN_MODE == 'Pilot':
    print('Pilot contributes ONE ERA5 rank to each histogram. Choose Catalog to aggregate all selected events.')
print('Result folder:', result_directory)
display(catalog[['event_number', 'event_start', 'event_peak', 'target_valid_time',
                 'onset_offset_hours', 'vortex_plot_label']])


## 4 · Run the verification

Both inputs use the same spherical divergence calculation with a two-grid-cell halo,
and the same cosine-latitude weighting and polygon boundaries on their native grids.
The fixed 925-hPa fields include any below-ground pressure-level values supplied by the
datasets. The coastal wedge results should be interpreted with that in mind. The notebook
reads WN2 members in source chunk groups and saves two means per member and lead.
Archive arrays use lazy indexing. The notebook selects the exact time, pressure level,
and region before reading one wind component at a time, with limited read concurrency.

Spatial averages use every finite grid cell inside each polygon with area weights.
Negative divergence represents convergence. Positive and negative values retain their
signs when averaged and ranked, so the resulting value measures the net convergence
or divergence of the whole polygon.

Each ERA5 rank is one plus the number of member values below ERA5, with seeded random
tie-breaking. Rank 1 means ERA5 was at the most convergent end and rank 65 at the least
convergent end. The exact out-of-range flags are recorded separately from the rank.

The batch status records missing or failed inputs. Rerunning retries those inputs and
reuses completed checkpoints. Cases with ERA5 and all three forecast leads contribute
to the six histograms so that the lead comparison uses the same events.

After a runtime restart, run sections 1–3 again before section 4. Wait for the
confirmations from both helper cells, 3b and 3c, and the catalog preview in 3d.
Completed checkpoints are stored in Drive.


In [ ]:
# @title 4 · Read the selected forecasts and calculate ranks
missing_setup = []
if not callable(globals().get('read_catalog')):
    missing_setup.append('3b · Load WeatherNext archive helpers')
if not callable(globals().get('run_ensemble_verification')):
    missing_setup.append('3c · Load rank and Excel helpers')
if any(name not in globals() for name in ('selected', 'archive', 'OUTPUT_DIR')):
    missing_setup.append('3d · Preview the selected catalog')
if missing_setup:
    raise RuntimeError(
        'Notebook 37 setup is incomplete in this runtime. Run these cells in order, '
        'wait for each to finish, then rerun section 4.\n' + '\n'.join(missing_setup)
        + '\nIf the runtime restarted, first rerun sections 1, 2, and 3a. '
        'Completed Drive checkpoints will be reused.')
# Clear stale archive metadata/listings when retrying a newly published date.
open_verification_era5.cache_clear()
archive.clear()
field_runs = run_verification_fields(selected, archive, OUTPUT_DIR, base_result_directory, VERIFY_FIELDS,
    saved_only=LOAD_SAVED_RESULTS_ONLY, job_runner=make_forecast_job_runner(archive), allow_era5t=ALLOW_ERA5T)
for _run in field_runs:
    _run['settings'] = {**_run['settings'], 'catalog':str(CATALOG_PATH),
        'study_start_utc':STUDY_START_UTC, 'study_end_utc_exclusive':STUDY_END_UTC or None,
        'run_mode':RUN_MODE, 'selected_events':len(selected), 'lead_hours':list(LEAD_HOURS),
        'tie_seed':37, 'allow_era5t':ALLOW_ERA5T,
        'training_cutoff_status':'operational archive training cutoff unverified'}
tables, calculation_settings = field_runs[0]['tables'], field_runs[0]['settings']
result_directory = field_runs[0]['directory']
display(tables['batch_status'][['event_number', 'lead_days', 'era5_status', 'wn2_status',
                                'era5_error', 'wn2_error']])
ranks = tables['case_ranks']
complete_events = 0 if ranks.empty else ranks.loc[ranks.in_common_sample, 'review_key'].nunique()
print(f'{len(selected)} selected events; {complete_events} complete events contribute to EACH histogram.')
print(f'{len(selected) - complete_events} events lack at least one required input. Details appear in Batch status.')
if not ranks.empty:
    columns = ['event_number', 'field', 'pressure_hpa', 'units', 'valid_time_utc',
               'polygon', 'lead_days', 'member_count', 'era5_value', 'era5_rank', 'in_common_sample']
    display(ranks[[column for column in columns if column in ranks]])


## 5 · Save and inspect results

The Excel workbook opens in Excel or Google Sheets. The same tables are available as
CSVs. The plots save as PNG and PDF. Pilots and the catalog have separate result folders.

**Case ranks** lists ERA5's value, exact rank, bin number, and ensemble minimum, mean,
and maximum for each event, polygon, and lead. **Member values** retains each forecast
value and native member ID. **ERA5 reference** retains the verifying values.
**Ordered values** is a flat table with all 65 entries per verifiable case, sorted from
lowest to highest selected-field value. **Ordered bins** presents the same
numbers in five columns of 13 entries, with ERA5 explicitly labeled and highlighted.
**Bin counts** gives each polygon/lead's event count in each numbered bin.
The default values are signed divergence in 10⁻⁵ s⁻¹; other fields retain their labeled
units. Exact rankings use full numerical precision.
Ties retain the same reproducibly chosen ERA5 position as the graphs.

**CRPS cases** and **CRPS summary** score the same complete all-lead cases, using all
64 signed polygon-mean values. Scores retain the field's units; lower is better.
The CRPS figure includes a table of sample size, mean, median, interquartile range
and preliminary ERA5T count beneath the lead comparison. **Event accounting** lists
every selected event, usable leads, inclusion status and complete retrieval errors.

Both plot versions are exported. **65 ranks** shows all possible positions of ERA5
within the ensemble. **5 groups** combines 13 consecutive ranks per bar for readability
with a small event sample. Both use the same calculated ranks and the same set of
complete events. The dashed line shows the expected counts for a uniform rank
distribution. The counts in every panel sum to its displayed N.

A single pilot event populates just one bar per panel. Catalog histograms describe
cases selected around ERA5-detected JPCZ events.


In [ ]:
# @title 5 · Save field reports and collect all saved values into one Excel sheet
from IPython.display import Image, display
RANK_DISPLAY = '65 ranks'  # @param ["65 ranks", "5 groups"]
export_field_reports(field_runs, selection_report)
combined_values, comparison_workbook = consolidate_saved_verification(
    saved_verification_sources(base_result_directory),
    OUTPUT_DIR / 'comparison' if RUN_MODE == 'Catalog' else base_result_directory / 'comparison')
print('One-sheet comparison:', comparison_workbook)
print('Rows collected from saved ordered/member/reference CSVs; no weather fields recalculated.')
preview = result_directory / ('rank_histograms_65ranks.png' if RANK_DISPLAY == '65 ranks' else 'rank_histograms.png')
if preview.is_file(): display_saved_verification_image(preview)
else: print('No complete common sample yet. Inspect batch_status.csv for missing inputs.')
for _run in field_runs:
    _crps = _run['directory'] / 'crps_summary.png'
    if _crps.is_file(): display_saved_verification_image(_crps)


## 5b · Every event's CRPS, from saved scores

CRPS is calculated **per event, polygon and lead**, using all 64 members. One event
contributes six scores: two polygons × three leads. Section 5's boxes summarize all
event scores; the orange line is their median, and the teal line is their mean.
Those boxes are neither ensemble spread nor confidence intervals on the mean.

This cell reads `crps_cases.csv`, labels every event in an overview, and saves a
separate PNG/PDF/CSV for each event plus **one combined PDF with one event per page**.
`one_row_per_event.csv` and `all_event_scores.xlsx` put all six CRPS values on one row.
A single-event figure uses six saved scores, without an across-event boxplot.
No weather fields or member-level CRPS are recalculated. Repeated runs reuse each
unchanged event figure. All exports stay in each field's `case_scores/` folder.


In [ ]:
# @title 5b · Display and export CRPS for every saved event
EVENT_CRPS_DISPLAY = 'All events'  # @param ["All events", "Overview only", "One event"]
CRPS_EVENT_NUMBER = 0  # @param {type:"integer"} # 0 = first saved event.
for _run in field_runs:
    _score_path = Path(_run['directory'])/'crps_cases.csv'
    if not _score_path.is_file(): raise FileNotFoundError('Run section 5 to save CRPS first: '+str(_score_path))
    _event_scores = pd.read_csv(_score_path, float_precision='round_trip')
    _case_dir = Path(_run['directory'])/'case_scores'
    _case_index, _case_wide = export_event_crps_reports(_event_scores, _case_dir)
    print(_run['id'], ':', len(_case_index), 'events;', len(_event_scores), 'saved CRPS values')
    print('Combined PDF:', _case_dir/'all_events_crps.pdf')
    print('One Excel row per event:', _case_dir/'all_event_scores.xlsx')
    display_saved_verification_image(_case_dir/'all_events_crps_overview.png', max_width=1100)
    display(_case_wide.drop(columns='review_key'))
    _show = _case_index if EVENT_CRPS_DISPLAY == 'All events' else _case_index.iloc[:0]
    if EVENT_CRPS_DISPLAY == 'One event':
        _show = _case_index.head(1) if CRPS_EVENT_NUMBER == 0 else _case_index.loc[_case_index.event_number.eq(CRPS_EVENT_NUMBER)]
        if _show.empty: print('That event is not in this saved common sample.')
    for _case in _show.itertuples(): display_saved_verification_image(Path(_case.event_png), max_width=1000)


## 5c · Independent ERA5 climatology for CRPSS

Freeze the reference definition before evaluating skill. **Historical JPCZ cases**
uses one exact onset-aligned ERA5 target per historical merged event in 2000–2021;
this compares performance within an event-selected sample. **All winter conditions**
uses every 6-hourly NDJF target and answers a broader climatological comparison.
Both use a pooled empirical distribution, equal target weights, and the identical
field, pressure level, polygons, units and divergence calculation as verification.
The reference period must finish before the earliest evaluated forecast initialization.
This does not establish independence of WN2's own training data; checkpoint provenance
remains a separate requirement.

Preview is the default. It reuses saved ERA5 targets and reports how many are missing.
For a new reference only, set `BUILD_MISSING_REFERENCE = True` and run in resumable
batches. This retrieves **ERA5 only**, one target in a bounded worker, and saves each
completed result immediately. WN2 inputs, event scores and forecast checkpoints stay
unchanged. Completed targets are skipped on restart; failures are listed explicitly.
`REFERENCE_CASES_PER_RUN = None` processes all remaining targets. All-winter sampling
has substantially more targets than the historical-event reference.


In [ ]:
# @title 5c · Preview or resume the independent ERA5 reference
CLIMATOLOGY_KIND = 'Historical JPCZ cases'  # @param ["Historical JPCZ cases", "All winter conditions"]
REFERENCE_START_UTC = '2000-01-01'  # @param {type:"date"}
REFERENCE_END_UTC = '2022-01-01'  # @param {type:"date"} # Exclusive: covers through 2021.
BUILD_MISSING_REFERENCE = False  # @param {type:"boolean"}
REFERENCE_CASES_PER_RUN = 25  # None = all remaining targets; failed targets retry next run.
_reference_schedule = climatology_schedule(CATALOG_PATH, kind=CLIMATOLOGY_KIND,
    start=REFERENCE_START_UTC, end=REFERENCE_END_UTC)
climatology_folders = {}
for _run in field_runs:
    _definition, _signature = climatology_definition(_reference_schedule, _run['settings'],
        kind=CLIMATOLOGY_KIND, start=REFERENCE_START_UTC, end=REFERENCE_END_UTC)
    _scores = pd.read_csv(Path(_run['directory'])/'crps_cases.csv', float_precision='round_trip')
    if pd.Timestamp(_definition['end_utc_exclusive']) > pd.to_datetime(_scores.init_time_utc, utc=True).min():
        raise ValueError('Reference period overlaps evaluated forecast initializations. Choose an earlier reference end.')
    _folder = OUTPUT_DIR/'climatology'/(_run['id']+'_'+_signature)
    climatology_folders[_run['id']] = _folder
    _shared_signature = hashlib.sha256(json.dumps(_run['settings'], sort_keys=True).encode()).hexdigest()[:16]
    print(_run['id'], ':', len(_reference_schedule), 'frozen reference targets;', _folder)
    _reference_status = build_saved_climatology(_reference_schedule, _folder, _definition, _run['spec'],
        job_runner=make_forecast_job_runner(archive) if BUILD_MISSING_REFERENCE else None,
        download_missing=BUILD_MISSING_REFERENCE, limit=REFERENCE_CASES_PER_RUN,
        shared_reference_cache=OUTPUT_DIR/'checkpoints'/_shared_signature)
    print('Status:', _reference_status['completed_targets'], '/', _reference_status['expected_targets'])


## 5d · CRPSS from paired saved model/reference scores

**CRPSS (%) = 100 × [1 − mean(model CRPS) / mean(reference CRPS)].** The same
complete events enter both means, separately for each polygon and lead. Positive
means improvement over this reference; zero means equal scores; negative means worse.
Do not average individual-event skill ratios. Per-event CSVs retain both CRPS values
and their difference (reference minus model; positive means improvement).

Until the complete frozen climatology is saved, status is **reference not prepared**
or **incomplete**, and no numerical CRPSS is reported. A zero mean reference CRPS
makes skill undefined, not zero. This cell does no weather retrieval.
[ECMWF's CRPSS definition](https://www.ecmwf.int/en/newsletter/153/meteorology/25-years-ensemble-forecasting-ecmwf)


In [ ]:
# @title 5d · Calculate CRPSS from the completed saved reference
if 'climatology_folders' not in globals(): raise RuntimeError('Run the section 5c preview to select the reference folder first.')
for _run in field_runs:
    _scores = pd.read_csv(Path(_run['directory'])/'crps_cases.csv', float_precision='round_trip')
    _skill_dir = Path(_run['directory'])/'crpss'
    _skill_status, _paired_scores, _skill_summary = export_crpss_report(_scores,
        climatology_folders[_run['id']], _skill_dir, _run['settings'])
    print(_run['id'], ':', _skill_status)
    if _skill_summary is not None:
        display(_skill_summary)
        display_saved_verification_image(_skill_dir/'crpss_summary.png', max_width=1100)
        _paired_index, _ = export_event_crps_reports(_paired_scores, _skill_dir/'case_scores')
        print('Per-event WN2/reference comparison PDF:', _skill_dir/'case_scores/all_events_crps.pdf')
    print('CRPSS outputs:', _skill_dir)


## 6 · Reliability diagrams and vortex strata

For each polygon and lead, forecast probability is the fraction of the **64 members**
whose instantaneous polygon-mean value meets the field's configured strict `less` or
`greater` criterion. The default is **925-hPa divergence below −2 × 10⁻⁵ s⁻¹**.
ERA5 supplies a binary crossing/non-crossing at the same valid time. Section 3a sets
each field's threshold in its displayed units; `None` skips its reliability plots.
This is an **editable exploratory snapshot criterion**.
Choose it before evaluating the forecast sample. It is **not** the catalog's hourly
trailing-D12 threshold, and these probabilities are not probabilities of the complete
JPCZ detector event or vortex occurrence. Testing a six-hourly approximation to D12
remains a separate scientific step.

These first diagrams use the **ERA5-event-conditioned sample**, the same complete
all-lead cases as the rank histograms. They describe convergence calibration among
selected events, not winter-wide occurrence calibration. Section 7 supplies the
independent winter schedule including non-events for broader assessment.

**Default: CORP reliability.** Following [Dimitriadis, Gneiting and Jordan (2021)](https://www.pnas.org/doi/10.1073/pnas.2016191118),
weighted isotonic regression/PAV pools adjacent forecast values automatically while
constraining estimated crossing frequency to be nondecreasing. The x-axis retains
original WN2 probabilities; the y-axis is the fitted conditional ERA5 frequency.
With 64 members, probabilities are discrete multiples of 1/64. Plot dots at the
observed probability values and connect them as in the paper's discrete setting.

CORP saves under each stratum's `corp/` subfolder. Fixed-bin diagrams remain alongside
it for comparison; choose **Fixed bins** below to display them. Original forecasts,
original Brier scores, CRPS and rank histograms are unchanged. CORP's fitted values
are an **in-sample diagnostic**, not validated replacement forecasts.

The table reports case count, distinct probabilities/PAV pools, observed frequency,
original Brier score, **MCB** (miscalibration), **DSC** (discrimination), and **UNC**
(uncertainty), with the identity **BS = MCB − DSC + UNC**. MCB smaller and DSC larger
are favorable; UNC describes the selected outcomes, not model quality.

Bands default off. Optional **90% pointwise consistency bands** use calibration-null
resampling and require explicitly assuming independent/exchangeable cases. They are
not confidence bands or simultaneous coverage. The paper cautions about dependent
cases and cites rank-histogram work under serial dependence (reference 41).

Fixed probability bins include 0 and 1; empty bins remain in the saved tables but
are not plotted as zero occurrence. Each diagram has percent axes and a statistics
table underneath: usable events, observed crossings/frequency, mean forecast probability,
Brier score and populated bins. Forecast counts save in a separate companion figure.
Wilson 95% intervals remain in the CSVs; the default curve hides them for clarity.
They are pointwise descriptive intervals and do not model serial dependence.

**Read the curve:** the diagonal is perfect reliability. Above it, estimated crossings
were more frequent than forecast; below it, less frequent. CORP uses a fitted frequency
at each distinct probability; the companion view uses fixed probability bins.
[ECMWF's reliability-diagram explanation](https://confluence.ecmwf.int/spaces/FUG/pages/673551875/Section%2B12.B%2BStatistical%2BConcepts%2B-%2BProbabilistic%2BData)

Separate All / Vortex / Non-vortex / Unknown outputs use Notebook 35's completed
screen status. Unchecked or failed screens never enter the Non-vortex stratum.
Changed review labels update the strata without redownloading the forecasts.


In [ ]:
# @title 6 · Refresh reliability and group ranks from saved scalar values
SNAPSHOT_THRESHOLD_1E5 = -2.0  # Legacy single-field fallback; VERIFY_FIELDS sets each current field's threshold.
PROBABILITY_BINS = 5  # Fixed 20-percentage-point bins; use 10 for a finer, sparser diagram.
RELIABILITY_METHOD = 'CORP'  # @param ["CORP", "Fixed bins"]
CORP_CONSISTENCY_BANDS = False  # Requires the independent/exchangeable-case assumption below.
ASSUME_EXCHANGEABLE_CASES = False  # Keep False for the current clustered winter sample.
SHOW_FORECAST_COUNTS = False  # Always saved separately; True also displays the count figure.
_report_runs = globals().get('field_runs', [dict(tables=tables, directory=result_directory,
    config=dict(threshold=SNAPSHOT_THRESHOLD_1E5, direction='less'), id='divergence_925hpa')])
reliability_folders = export_field_reliability(_report_runs, n_bins=PROBABILITY_BINS, method=RELIABILITY_METHOD,
    consistency_bands=CORP_CONSISTENCY_BANDS, assume_exchangeable=ASSUME_EXCHANGEABLE_CASES)
from IPython.display import Image, display
for _folder in reliability_folders:
    if _folder.name == 'all' or (_folder.name == 'corp' and _folder.parent.name == 'all'):
        for _polygon in ('jpcz_polygon','coastal_wedge'):
            for _lead in LEAD_HOURS:
                _prefix = 'corp' if _folder.name == 'corp' else 'reliability'
                _image = _folder / f'{_prefix}_{_polygon}_lead{_lead//24:02d}d.png'
                if _image.is_file(): display_saved_verification_image(_image, max_width=1000)
        _summary = _folder / ('corp_summary.csv' if _folder.name == 'corp' else 'reliability_summary.csv')
        if _summary.is_file():
            _statistics = pd.read_csv(_summary)
            _columns = ['polygon','lead_days','cases','brier_score','miscalibration_mcb','discrimination_dsc','uncertainty_unc']
            display(_statistics[[c for c in _columns if c in _statistics]].rename(columns={
                'polygon':'Region','lead_days':'Lead (days)','cases':'Events','brier_score':'Brier score',
                'miscalibration_mcb':'MCB','discrimination_dsc':'DSC','uncertainty_unc':'UNC'}))
        _counts = _folder / ('corp_forecast_counts.png' if _folder.name == 'corp' else 'forecast_probability_counts.png')
        if SHOW_FORECAST_COUNTS and _counts.is_file(): display_saved_verification_image(_counts)


## 7 · Scheduled winter reliability, including non-events

This optional retrieval uses a regular UTC schedule fixed independently of observed
events. Both threshold crossings and non-crossings are retained; missing reference or
forecast inputs are marked failed/pending rather than coded as zero. Set the **exclusive**
end to extend the winter sample. By default there is one target per day at 00 UTC in
November–February; choose 6 or 12 hours for a denser schedule. Do not select only dates
with catalog events or select dates according to forecast probabilities.

This can require substantial WN2 reads: every target uses all members at three leads.
The preview is free of field downloads; set RUN_SCHEDULED_RELIABILITY to True to retrieve.
It shares compact wind-derived checkpoints with the catalog run, but saves its own
probabilities, counts, settings, and status table. Its sample contains no event-derived
vortex strata. Training overlap and missing-case selection still limit interpretation.

To extend, rerun this cell with the same start and later end. Failed dates retry;
completed WN2 targets/leads reuse their checkpoints. ERA5T targets refresh until final.
A November 19 target requests initializations on November 18, 14, and 9 at the same UTC
hour for the 1-, 5-, and 10-day leads. Exact WN2 availability is checked on each request;
the workflow never substitutes a later run or another model.


In [ ]:
# @title 7 · Preview or run the outcome-independent winter schedule
RUN_SCHEDULED_RELIABILITY = False  # @param {type:"boolean"}
SCHEDULE_START_UTC = '2026-11-01'  # @param {type:"date"}
SCHEDULE_END_UTC = '2027-03-01'  # @param {type:"date"} # Exclusive; future times stay pending.
SCHEDULE_INTERVAL_HOURS = 24  # @param [6, 12, 24] {type:"raw"}
# The independent schedule uses the first configured field and its own threshold.
_schedule_field = verification_field_configs(VERIFY_FIELDS)[0]
_schedule_threshold = _schedule_field['config'].get('threshold')
scheduled = scheduled_verification_catalog(
    SCHEDULE_START_UTC, SCHEDULE_END_UTC, interval_hours=SCHEDULE_INTERVAL_HOURS)
print(f'{len(scheduled)} winter targets; {len(scheduled)*len(LEAD_HOURS)} forecast requests.')
display(scheduled[['event_number', 'target_valid_time']])
if RUN_SCHEDULED_RELIABILITY:
    if _schedule_threshold is None:
        raise ValueError('Choose a reliability threshold for the scheduled field before retrieving data.')
    # Future schedule entries remain visible in the preview, but are not downloaded.
    due = scheduled.loc[scheduled.target_valid_time.le(pd.Timestamp.now(tz='UTC').tz_localize(None))]
    if due.empty:
        raise ValueError('No schedule targets are due yet.')
    open_verification_era5.cache_clear()
    archive.clear()
    scheduled_tables, scheduled_settings = run_ensemble_verification(
        due, archive, OUTPUT_DIR, allow_era5t=ALLOW_ERA5T, spec=_schedule_field['spec'],
        job_runner=make_forecast_job_runner(archive))
    schedule_dir = OUTPUT_DIR / 'scheduled_winter' / _schedule_field['id']
    schedule_dir.mkdir(parents=True, exist_ok=True)
    for name, frame in scheduled_tables.items():
        frame.to_csv(schedule_dir / f'{name}.csv', index=False)
    schedule_settings = {**scheduled_settings, 'start': SCHEDULE_START_UTC,
        'end_exclusive': SCHEDULE_END_UTC, 'interval_hours': SCHEDULE_INTERVAL_HOURS,
        'due_targets': len(due), 'future_targets': len(scheduled)-len(due), 'allow_era5t': ALLOW_ERA5T}
    atomic_json_file(schedule_dir / 'schedule_settings.json', schedule_settings)
    scheduled_probabilities, scheduled_bins = export_reliability(
        scheduled_tables, schedule_dir, {region:float(_schedule_threshold) for region in POLYGONS},
        sample='scheduled_winter', n_bins=PROBABILITY_BINS,
        direction=_schedule_field['config'].get('direction','less'))
    display(scheduled_tables['batch_status'])
    verified = scheduled_probabilities.review_key.nunique() if not scheduled_probabilities.empty else 0
    print(f'{verified}/{len(due)} due targets verified at all leads; {len(scheduled)-len(due)} future targets.')
    if not scheduled_probabilities.empty:
        figure = plot_reliability_diagrams(scheduled_probabilities, n_bins=PROBABILITY_BINS)
        display(figure)
        plt.close(figure)
